<a href="https://colab.research.google.com/github/Beki1737/Beki1737/blob/main/CoalitionRAG-Debias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CoalitionRAG-Debias
### Diagnosing Behavioral Mode Coverage Failure in Retrieval-Augmented LLM Political Simulation

**Authors:** Becky Firomssa Gudeta · Beakcheol Jang (ANDlab, Yonsei University)  
**Venue:** EMNLP 2026 (submitted May 25, 2026)  
**Extends:** Choi et al. AAAI 2026 — *"As Eastern Powers, I Will Veto"* ([arXiv:2511.10695](https://arxiv.org/abs/2511.10695))

---

## What This Project Does

This notebook implements **CoalitionRAG-Debias**, a retrieval-augmented debiasing system for LLM political simulation. The system diagnoses and addresses a fundamental failure in RAG-based behavioral simulation: when the retrieval pool does not contain the relevant **behavioral mode**, no reranker can recover it regardless of its design.

We study this failure through P5 nation-level vote simulation in the United Nations Security Council (UNSC), where behavioral modes are structured (Y/N/A coalition vectors), publicly documented, and historically verifiable — making coverage failure directly measurable.

The project delivers three contributions:

| Contribution | Description |
|---|---|
| **UNSC-CKG** | First public coalition-structure knowledge graph for the UNSC — 3,087 resolutions from 1946 to 2026 with P5 coalition vectors, geopolitical theme labels, and historical voting profiles |
| **CSR** (Coalition-Similarity Retrieval) | Five-variant two-stage retrieval family that reranks text-recall candidates by coalition structure signal. Canonical method: CSR-v3 (entropy reranking, parameter-free) |
| **BCS** (Behavioral Consistency Score) | Annotation-free calibration metric: `BCS = 1 − JSD(P_LLM ∥ P_historical)` — measures distributional alignment between simulated and historical voting |

**Central finding:** For Western P5 nations (France, UK, USA), 71–80% of contested queries have no coalition-matching precedent in the top-10 retrieved candidates regardless of retrieval method. This pool ceiling — not reranker design — is the binding constraint on debiasing quality.

---

## Notebook Structure

### Section 1 — Environment Setup (Cells 1–2)
Mount Google Drive, install dependencies (`rank_bm25`, `sentence-transformers`, `langchain`, `langgraph`), clone the AAAI 2026 baseline repository ([Nation-Level_Bias](https://github.com/concistency/Nation-Level_Bias)).

### Section 2 — Data Loading and Inspection (Cells 3–4)
Load and audit the AAAI 2026 test set (66 non-adopted / vetoed UNSC resolutions, 2013–2024). Normalize field types — `keyword` is stored as a comma-separated string, not a list. Produce the 66 canonical test-resolution dicts used throughout all experiments.

### Section 3 — UNSC-CKG Construction (Cells 5–8)
Build the coalition knowledge graph from two public sources:
- **UN Dag Hammarskjöld Library** voting archive (41,349 votes, 2,815 adopted resolutions)
- **DPPA Security Council Vetoes Dataset** (272 vetoed drafts, 1946–2026)

Steps: load voting CSV → extract P5 coalition vectors → rule-based theme classifier (30 themes, 87.2% coverage) → country mention extractor (112-pattern dictionary) → hierarchical coalition profile (pre-2013, leakage-safe). Saves `canonical_state_v1.pkl` (4.7 MB).

### Section 4 — CSR Variant Development and Gate Tests (Cells 9–22)
Systematically develops and evaluates five CSR reranking strategies, each tested against the AAAI text baseline using coalition match rate at k=1 as the gate metric.

| Cell | Variant | Result |
|---|---|---|
| 9–9.5 | Initial gate + recency baseline | Day-1 diagnostic |
| 10 | Oracle headroom analysis | +54.6pp mean headroom in top-50 |
| 11 | CSR-v2 (predicted coalition) | −0.3pp — collapses to all-Y |
| 12 | Consensus-bias diagnostic | 79% of targets rank unanimous-Y higher |
| 13 | **CSR-v3 (entropy reranking)** | **+2.7pp — canonical method** |
| 14–17 | CSR-v4 (cross-encoder, full data) | −2.4pp — learns wrong prior |
| 18 | Consensus-bias diagnostic v4 | Confirms corpus-induced failure |
| 19–22 | CSR-v5 (cross-encoder, contested-only) | −0.3pp — era mismatch |

### Section 5 — AAAI Pipeline Integration (Cells 22.5–23)
Integrates CSR-v3 into the AAAI LangGraph pipeline via monkey-patching `RAGRflxGraph.retrieve_document`. Resolves the AAAI adopted-dataset format issues (no `vote` field; `res_no` format changes at 2020). Achieves 100% coalition recovery via CKG cross-reference. Verifies end-to-end pipeline with mock LLM (5 calls, 13s, no errors). Saves `canonical_state_v2.pkl` (23.6 MB).

### Section 6 — Standard Retrieval Baselines (Cells 24–25)
Implements two standard baselines for comparison:
- **BM25** (`rank_bm25.BM25Okapi`) over `summary + action_item + keyword` fields
- **Dense** (`BAAI/bge-small-en-v1.5`, BGE-small, 33M params) with normalized cosine similarity

Both use the same text fields and pool-separated architecture as CSR for fair comparison. Dense embeddings cached to `dense_embeddings_v1.pkl` (~10 MB).

### Section 7 — Four-Way Retrieval Comparison (Cells 26–26.7)
Runs all four retrievers (AAAI text, BM25, Dense, CSR-v3) on all 66 test resolutions × 5 P5 nations. Produces the paper's primary retrieval quality tables including combined-pool coalition match rate, per-pool decomposition, case breakdown (AA/AN/NA/NN), oracle headroom, and per-region analysis.

**Key finding:** Dense wins overall (60.9% mean); CSR-v3 wins on adopted-pool Western nations (FRA/GBR/USA: 22.7% vs Dense's 7.6–15.2%). Non-adopted pool drives most useful LLM signal (Dense NA=33.3%).

### Section 8 — Speech Augmentation and RRF Experiments (Cells 27–29)
Tests two additional retrieval approaches:
- **Speech-augmented retrieval** (Cell 27): embeds per-nation speech text, matches target resolution against past nation speeches. Result: −4.8pp vs Dense — speeches too noisy for regime disambiguation.
- **Reciprocal Rank Fusion** (Cell 29): fuses rankings from all four methods via RRF (k=60). Result: −4.5pp vs Dense — orthogonal signals do not compound. Retrieval design phase complete.

Saves `canonical_state_v3.pkl` with all retrieval results and paper decisions.

### Section 9 — LLM Experiment Runner (Cells 30–32)
Runs the full downstream LLM evaluation across four conditions and three models:

| Condition | Description |
|---|---|
| **B0** | Zero-shot — no retrieval, no Reflexion (lower bound) |
| **B1** | AAAI text retrieval + Reflexion (exact replication of Choi et al. 2026) |
| **B2** | Dense retrieval + Reflexion |
| **B3** | CSR-v3 + Reflexion (proposed method) |

**Models:** GPT-4o-mini (OpenAI API), Llama-3.3-70B (Together.ai), Mistral-Small-24B (Together.ai).  
**Scale:** 4 conditions × 3 models × 5 nations × 66 resolutions × 3 runs = 11,880 LLM calls. ~$12 total.  
**Cell 31** computes Weighted F1 (per Choi et al. 2026 Table 3 format) and BCS with 95% bootstrap CI.  
**Cell 32** outputs paper-ready tables for §7.2 and §7.3 and flags BCS/F1 divergence cases.

---

## Key Results

| Metric | B0 → B3 improvement |
|---|---|
| Weighted F1 (mean, 3 models) | 0.42 → 0.57 (+0.15) |
| BCS — USA | 0.49 → 0.63 (+0.14) |
| BCS — FRA / GBR | 0.58 → 0.68 (+0.10 each) |
| Adopted-pool FRA match rate | 4.5% → 22.7% (+18pp, CSR-v3 vs AAAI) |

BCS and Weighted F1 **diverge** for France and UK: CSR-v3 improves BCS (+0.10) while Dense achieves higher per-resolution F1 — demonstrating that accuracy metrics alone cannot detect distributional debiasing.

---

## File Map (Google Drive: `CoalitionRAG/`)

| File | Contents |
|---|---|
| `canonical_state_v1.pkl` | CKG pool, test resolutions, profile, P5 maps |
| `canonical_state_v2.pkl` | + AAAI candidates, CSR retriever |
| `canonical_state_v3.pkl` | + all retrieval results, paper decisions |
| `dense_embeddings_v1.pkl` | BGE-small embeddings (adopted + non-adopted) |
| `speech_embeddings_v1.pkl` | Per-nation speech embeddings |
| `llm_results/` | Per-(model, condition) JSON result files |
| `paper_results/` | Retrieval quality JSON files (v1/v2/v3) |
| `Nation-Level_Bias/` | Cloned AAAI 2026 baseline repo |

---

## Requirements

In [ ]:
# ============================================================
# CELL 1 — Environment setup
# ============================================================
# Mounts Google Drive (so files persist across kernel restarts),
# installs minimal dependencies, sets up paths.

from google.colab import drive
drive.mount('/content/drive')

import os
import sys

# Working directory: persistent under Drive so we don't lose state
WORKDIR = '/content/drive/MyDrive/CoalitionRAG'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f"Working directory: {os.getcwd()}")

# Install dependencies
!pip install -q rank_bm25 sentence-transformers 2>&1 | tail -3

# Verify core libraries
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
from collections import defaultdict, Counter
from scipy.spatial.distance import jensenshannon

print(f"\npandas {pd.__version__}, numpy {np.__version__}")
print(f"Python {sys.version.split()[0]}")
print("✓ Cell 1 complete")

In [ ]:
# ============================================================
# CELL 2-fix — Clean up failed clone and retry
# ============================================================
# The previous clone died mid-stream. We delete the partial repo,
# then retry. If git clone fails again, we fall back to downloading
# the repo as a zip — slower but more reliable on flaky connections.

import os
import shutil

REPO_DIR = os.path.join(WORKDIR, 'Nation-Level_Bias')

# Step 1: nuke the broken partial clone
if os.path.exists(REPO_DIR):
    print(f"Removing broken clone at {REPO_DIR}...")
    shutil.rmtree(REPO_DIR)
    print("  removed")

# Step 2: try git clone with shallow depth (smaller transfer)
os.chdir(WORKDIR)
print("\nAttempt 1: shallow git clone...")
clone_result = !git clone --depth 1 https://github.com/concistency/Nation-Level_Bias.git 2>&1
print('\n'.join(clone_result[-5:]))  # last 5 lines

# Verify clone succeeded
test_file = 'Nation-Level_Bias/Dataset/not_adopted_resolutions_dataset.json'
if os.path.exists(os.path.join(WORKDIR, test_file)):
    print("\n✓ Shallow clone succeeded")
else:
    # Step 3: fall back to zip download
    print("\n✗ Git clone failed again. Falling back to zip download...")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)

    !wget -q https://github.com/concistency/Nation-Level_Bias/archive/refs/heads/main.zip -O nlb.zip
    !unzip -q nlb.zip
    !mv Nation-Level_Bias-main Nation-Level_Bias
    !rm nlb.zip

    if os.path.exists(os.path.join(WORKDIR, test_file)):
        print("✓ Zip download succeeded")
    else:
        print("✗ Both methods failed — check internet connection or GitHub status")

# Final verification
print("\n" + "=" * 50)
print("FILE CHECK")
print("=" * 50)
expected_files = [
    'Nation-Level_Bias/Dataset/not_adopted_resolutions_dataset.json',
    'Nation-Level_Bias/Dataset/adopted_resolutions_dataset.json',
    'Nation-Level_Bias/Dataset/keyword_pool.json',
    'Nation-Level_Bias/Framework/RAG_RFLX_GRAPH.py',
    'Nation-Level_Bias/[submission]Debiasing_Framework.py',
]
all_ok = True
for f in expected_files:
    full_path = os.path.join(WORKDIR, f)
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    flag = '✓' if exists else '✗'
    print(f"  {flag} {f:<70} ({size:,} bytes)")
    if not exists:
        all_ok = False

if not all_ok:
    print("\n✗ Stop here. Send me the output before proceeding.")
else:
    # Now safe to load the test set
    TEST_SET_PATH = os.path.join(WORKDIR,
        'Nation-Level_Bias/Dataset/not_adopted_resolutions_dataset.json')
    with open(TEST_SET_PATH) as f:
        aaai_test = json.load(f)

    print(f"\nAAAI test set: {len(aaai_test)} entries")
    print(f"Fields in entry [0]: {sorted(aaai_test[0].keys())}")
    print(f"\nSample entry [0] essentials:")
    e = aaai_test[0]
    for field in ['res_no', 'date', 'geopolitical_region', 'target_nations_list']:
        print(f"  {field:<22}: {e.get(field)}")
    print(f"  keyword (first 5)     : {e.get('keyword', [])[:5]}")
    print(f"  vote.favour count     : {len(e.get('vote', {}).get('favour', []))}")
    print(f"  vote.against count    : {len(e.get('vote', {}).get('against', []))}")
    print(f"  vote.abstention count : {len(e.get('vote', {}).get('abstention', []))}")
    speech_keys = list(e.get('speech', {}).keys())
    print(f"  speech nations ({len(speech_keys)}): {speech_keys[:5]}")

    print("\n✓ Cell 2-fix complete — ready for Cells 3–4")

In [ ]:
# ============================================================
# CELL 3 — Load AAAI test set, audit field types
# ============================================================
# Already loaded `aaai_test` at end of Cell 2-fix, but we re-bind here
# so this cell is self-contained if the kernel restarts.
# Then we audit every field across all 66 entries to catch type
# inconsistencies (string vs list, missing keys, etc.) that would
# silently break downstream code.

import json
import os
from collections import Counter

TEST_SET_PATH = os.path.join(WORKDIR,
    'Nation-Level_Bias/Dataset/not_adopted_resolutions_dataset.json')
with open(TEST_SET_PATH) as f:
    aaai_test = json.load(f)

print(f"AAAI test set: {len(aaai_test)} entries\n")

# Audit field types across all entries
print("FIELD TYPE AUDIT (across all 66 entries)")
print("=" * 60)
field_types = defaultdict(Counter)
for entry in aaai_test:
    for field, value in entry.items():
        field_types[field][type(value).__name__] += 1

for field in sorted(field_types):
    types = field_types[field]
    type_str = ', '.join(f"{t}={c}" for t, c in types.items())
    flag = '⚠' if len(types) > 1 else ' '
    print(f"  {flag} {field:<22} {type_str}")

# Look closer at `keyword` — Cell 2 output suggests it might be a string
print("\nKEYWORD FIELD INSPECTION (first 5 entries)")
print("=" * 60)
for i, entry in enumerate(aaai_test[:5]):
    kw = entry.get('keyword')
    print(f"  [{i}] type={type(kw).__name__}, "
          f"value={repr(kw)[:80]}{'...' if len(repr(kw)) > 80 else ''}")

# Look at target_nations_list — should be a list
print("\nTARGET_NATIONS_LIST INSPECTION (first 5 entries)")
print("=" * 60)
for i, entry in enumerate(aaai_test[:5]):
    tn = entry.get('target_nations_list')
    print(f"  [{i}] type={type(tn).__name__}, value={tn}")

# Region distribution across the test set
print("\nGEOPOLITICAL_REGION DISTRIBUTION (66 entries)")
print("=" * 60)
region_counts = Counter(e.get('geopolitical_region') for e in aaai_test)
for region, count in region_counts.most_common():
    print(f"  {region:<28} {count:>3}")

# Date range and year distribution
dates = [e['date'] for e in aaai_test]
print(f"\nDATE RANGE: {min(dates)} to {max(dates)}")
year_counts = Counter(d[:4] for d in dates)
print("Year distribution:")
for year in sorted(year_counts):
    print(f"  {year}: {year_counts[year]}")

# P5 voting outcomes on test set — sanity check, this is our ground truth
print("\nP5 VOTE DISTRIBUTION ON TEST SET (ground truth)")
print("=" * 60)
P5_AAAI = {'CHN': 'China', 'FRA': 'France', 'GBR': 'United Kingdom',
           'RUS': 'Russian Federation', 'USA': 'United States'}
for code, aaai_name in P5_AAAI.items():
    counts = Counter()
    for e in aaai_test:
        v = e['vote']
        if aaai_name in v.get('favour', []):
            counts['Y'] += 1
        elif aaai_name in v.get('against', []):
            counts['N'] += 1
        elif aaai_name in v.get('abstention', []):
            counts['A'] += 1
        else:
            counts['Missing'] += 1
    total = sum(counts.values())
    print(f"  {code} ({aaai_name:<22}): "
          f"FOR={counts['Y']:>2} ({counts['Y']/total*100:>4.0f}%) "
          f"AGAINST={counts['N']:>2} ({counts['N']/total*100:>4.0f}%) "
          f"ABSTAIN={counts['A']:>2} ({counts['A']/total*100:>4.0f}%) "
          f"Missing={counts['Missing']}")

print("\n✓ Cell 3 complete")

In [ ]:
# ============================================================
# CELL 4 — Define field normalizer for AAAI entries
# ============================================================
# Cell 3 will reveal whether `keyword` is a string or list, and whether
# any entry has missing fields. This cell defines a normalizer that
# converts every entry to a clean canonical form regardless of the
# input format. We use it everywhere downstream so we never have to
# special-case strings vs lists again.

from datetime import datetime

P5 = ['CHN', 'FRA', 'GBR', 'RUS', 'USA']
NATION_AAAI_TO_CODE = {
    'China': 'CHN',
    'France': 'FRA',
    'United Kingdom': 'GBR',
    'United Kingdom of Great Britain and Northern Ireland': 'GBR',
    'Russian Federation': 'RUS',
    'United States': 'USA',
    'United States of America': 'USA',
}
NATION_CODE_TO_AAAI = {
    'CHN': 'China', 'FRA': 'France', 'GBR': 'United Kingdom',
    'RUS': 'Russian Federation', 'USA': 'United States',
}


def to_list(x):
    """Coerce string-or-list to a list, handling None and comma-separated strings."""
    if x is None:
        return []
    if isinstance(x, list):
        return [str(item).strip() for item in x if item]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # If it looks comma-separated, split; otherwise treat as single-item list
        if ',' in s:
            return [item.strip() for item in s.split(',') if item.strip()]
        return [s]
    return [str(x)]


def parse_aaai_vote(vote_dict, nation_aaai_name):
    """Look up a single nation's vote in AAAI's vote dict format."""
    if nation_aaai_name in vote_dict.get('favour', []):
        return 'Y'
    if nation_aaai_name in vote_dict.get('against', []):
        return 'N'
    if nation_aaai_name in vote_dict.get('abstention', []):
        return 'A'
    return None


def extract_p5_coalition(vote_dict):
    """Extract the P5 voting pattern as a {code: 'Y'/'N'/'A'/None} dict."""
    return {code: parse_aaai_vote(vote_dict, NATION_CODE_TO_AAAI[code])
            for code in P5}


def parse_test_entry(entry):
    """
    Convert one AAAI test JSON entry into our canonical dict form.
    Use this everywhere downstream so field type quirks are handled once.
    """
    return {
        'resolution_id': entry['res_no'],
        'date': datetime.fromisoformat(entry['date']),
        'context': entry.get('context', ''),
        'summary': entry.get('summary', ''),
        'action_item': entry.get('action_item', ''),
        'title': entry.get('title', ''),
        'geopolitical_region': entry.get('geopolitical_region', 'Other'),
        'target_nations': to_list(entry.get('target_nations_list')),
        'keywords': to_list(entry.get('keyword')),
        'coalition': extract_p5_coalition(entry.get('vote', {})),
        'speech': entry.get('speech', {}),
        'is_adopted': False,  # this is the non-adopted set
        # 'theme' will be filled by our classifier in Cell 6
    }


# Apply normalizer to all 66 entries
test_resolutions = [parse_test_entry(e) for e in aaai_test]

# Verify: spot-check first 3 normalized entries
print("NORMALIZED TEST ENTRIES (first 3)")
print("=" * 60)
for r in test_resolutions[:3]:
    print(f"\n{r['resolution_id']} ({r['date'].date()})")
    print(f"  region:   {r['geopolitical_region']}")
    print(f"  targets:  {r['target_nations']}")
    print(f"  keywords ({len(r['keywords'])}): {r['keywords'][:5]}"
          f"{'...' if len(r['keywords']) > 5 else ''}")
    print(f"  coalition: {r['coalition']}")
    print(f"  has speech for P5? "
          f"{[n for n in NATION_CODE_TO_AAAI.values() if n in r['speech']]}")

# Summary
print("\n" + "=" * 60)
print("NORMALIZATION SUMMARY")
print("=" * 60)
print(f"Total entries: {len(test_resolutions)}")
print(f"Avg keywords per entry: "
      f"{np.mean([len(r['keywords']) for r in test_resolutions]):.1f}")
print(f"Avg target_nations per entry: "
      f"{np.mean([len(r['target_nations']) for r in test_resolutions]):.1f}")
print(f"Entries with empty keywords: "
      f"{sum(1 for r in test_resolutions if not r['keywords'])}")
print(f"Entries with empty target_nations: "
      f"{sum(1 for r in test_resolutions if not r['target_nations'])}")
print(f"Entries with all P5 votes recorded: "
      f"{sum(1 for r in test_resolutions if all(v is not None for v in r['coalition'].values()))}/66")

print("\n✓ Cell 4 complete — `test_resolutions` ready")


In [ ]:
# ============================================================
# CELL 5 — Load UN voting data, build clean P5 voting matrix
# ============================================================
# We need the raw UN voting CSV from your previous notebook. If it's
# already in Drive, we load it. Otherwise, we prompt for upload.
# Then we filter to P5, standardize names/votes, and produce a clean
# resolution-level voting matrix.

import pandas as pd
import numpy as np
import os
from datetime import datetime

CSV_PATH = os.path.join(WORKDIR, '2026_02_06_sc_voting.csv')

# Load — try Drive first, then prompt upload if missing
if os.path.exists(CSV_PATH):
    print(f"Loading from Drive: {CSV_PATH}")
    df_sc = pd.read_csv(CSV_PATH, low_memory=False)
else:
    print(f"File not found at {CSV_PATH}")
    print("Upload 2026_02_06_sc_voting.csv now:")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df_sc = pd.read_csv(fname, low_memory=False)
    # Save to Drive so we don't have to re-upload after kernel restart
    df_sc.to_csv(CSV_PATH, index=False)
    print(f"Saved to {CSV_PATH} for future runs")

print(f"\nLoaded: {len(df_sc):,} rows × {df_sc.shape[1]} columns")
print(f"Columns: {df_sc.columns.tolist()}")

# Filter to P5 only and clean
NAME_TO_CODE = {
    'CHINA': 'CHN',
    'FRANCE': 'FRA',
    'UNITED KINGDOM': 'GBR',
    'UNITED KINGDOM OF GREAT BRITAIN AND NORTHERN IRELAND': 'GBR',
    'RUSSIAN FEDERATION': 'RUS',
    'USSR': 'RUS',
    'UNION OF SOVIET SOCIALIST REPUBLICS': 'RUS',
    'UNITED STATES': 'USA',
    'UNITED STATES OF AMERICA': 'USA',
}
VOTE_TO_NUM = {'Y': 1, 'N': -1, 'A': 0, 'X': 0}  # X = non-voting → treat as abstain

df_p5 = df_sc[df_sc['permanent_member'] == True].copy()
df_p5['nation'] = df_p5['ms_name'].str.upper().map(NAME_TO_CODE)
df_p5 = df_p5[df_p5['nation'].notna()].copy()  # drop the Somalia data error
df_p5['vote_num'] = df_p5['ms_vote'].map(VOTE_TO_NUM)
df_p5['date'] = pd.to_datetime(df_p5['date'])
df_p5['year'] = df_p5['date'].dt.year

print(f"\nAfter P5 filter: {len(df_p5):,} rows")
print(f"Date range: {df_p5['date'].min().date()} to {df_p5['date'].max().date()}")
print(f"Unique resolutions: {df_p5['resolution'].nunique():,}")
print(f"\nVote-by-nation counts:")
print(df_p5.groupby(['nation', 'ms_vote']).size().unstack(fill_value=0))

# Build resolution-level voting matrix
# Use string votes ('Y', 'N', 'A') instead of numbers — preserves missing-data
# semantics and matches our coalition format.
df_p5['vote_str'] = df_p5['ms_vote'].map({'Y': 'Y', 'N': 'N', 'A': 'A', 'X': 'A'})

voting_matrix = df_p5.pivot_table(
    index='resolution',
    columns='nation',
    values='vote_str',
    aggfunc='first'
)
P5 = ['CHN', 'FRA', 'GBR', 'RUS', 'USA']
voting_matrix = voting_matrix[[c for c in P5 if c in voting_matrix.columns]]

# IMPORTANT: NaN in voting_matrix means "nation not on Council" (USSR pre-1992,
# RUS pre-USSR, etc.) — keep as None, don't fill with 'A'. This was a bug
# in the previous notebook that biased the historical profile.

# Resolution metadata
resolution_meta = df_p5.groupby('resolution').agg(
    date=('date', 'first'),
    year=('year', 'first'),
    description=('description', 'first'),
    subjects=('subjects', 'first'),
).reset_index()
resolution_meta['date'] = pd.to_datetime(resolution_meta['date'])

print(f"\n✓ Voting matrix shape: {voting_matrix.shape}")
print(f"✓ Resolution metadata: {len(resolution_meta):,} rows")
print(f"\nFirst 5 rows of voting matrix:")
print(voting_matrix.head())
print(f"\nMissing-data check (number of NaN per nation):")
print(voting_matrix.isna().sum())

print("\n✓ Cell 5 complete")

In [ ]:
# ============================================================
# CELL 6 — Theme classifier + build CKG candidate pool
# ============================================================
# Constructs the historical pool that CSR retrieves from. This is the
# single most important cell for retrieval quality — every entry must
# have correct `geopolitical_region`, `theme`, `coalition`, `keywords`,
# and `target_nations`. We also add the AAAI veto data.

import re
import json
from collections import Counter, defaultdict

# Load veto file (DPPA-SCVETOES)
VETO_PATH = os.path.join(WORKDIR, 'DPPA-SCVETOES.csv')
if os.path.exists(VETO_PATH):
    df_veto = pd.read_csv(VETO_PATH)
    print(f"Loaded veto file: {len(df_veto)} events")
else:
    print(f"DPPA-SCVETOES.csv not found at {VETO_PATH}")
    print("Upload it now:")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    df_veto = pd.read_csv(fname)
    df_veto.to_csv(VETO_PATH, index=False)
df_veto = df_veto.rename(columns={
    'china': 'CHN_v', 'france': 'FRA_v', 'united_kingdom': 'GBR_v',
    'russian_federation_ussr': 'RUS_v', 'united_states': 'USA_v',
    'draft_res#': 'draft_id',
})
df_veto['date'] = pd.to_datetime(df_veto['date'])

# Theme classifier — order matters (specific themes before general ones).
# This is rule-based for reproducibility; we'll note in the paper that an
# LLM-based classifier could be substituted but rules give deterministic
# outputs that won't drift between runs.
THEME_RULES = [
    # Most specific first
    ('Ukraine & Eastern Europe',
     ['ukraine', 'crimea', 'donbas', 'sevastopol']),
    ('North Korea / DPRK',
     ['democratic people.*republic of korea', 'dprk', 'north korea',
      'korean peninsula']),
    ('Middle East — Iran',
     [r'\biran\b', 'iranian', 'persian gulf']),
    ('Middle East — Israel/Palestine',
     ['palestin', 'gaza', 'west bank', 'israel', 'jerusalem',
      'lebanese', 'lebanon', 'golan', 'hezbollah', 'hamas',
      'occupied palestinian']),
    ('Middle East — Syria',
     ['syria', 'syrian']),
    ('Middle East — Iraq',
     ['iraq']),
    ('Middle East — Yemen',
     ['yemen']),
    ('Africa — Sudan & South Sudan',
     ['sudan', 'darfur', 'south sudan']),
    ('Africa — Somalia',
     ['somalia', 'somali', 'al-shabaab']),
    ('Africa — Congo (DRC)',
     ['democratic republic.*congo', r'\bdrc\b', 'kinshasa',
      'monusco', 'monuc']),
    ('Africa — Libya',
     ['libya', 'libyan', 'tripoli', 'benghazi']),
    ('Africa — Rwanda & Great Lakes',
     ['rwanda', 'burundi', 'great lakes']),
    ('Africa — West Africa',
     ['sierra leone', 'liberia', 'mali', 'burkina',
      'côte d.ivoire', 'ivory coast', 'ecowas',
      r'guinea[- ]bissau', r'\bguinea\b']),
    ('Africa — Other',
     ['ethiopia', 'eritrea', 'kenya', 'mozambique', 'zimbabwe',
      'angola', 'namibia', 'central african', 'tigray']),
    ('Nuclear & Non-Proliferation',
     ['nuclear', 'non-proliferation', 'proliferation', 'atomic',
      'wmd', 'iaea', 'chemical weapon', 'biological weapon']),
    ('Terrorism & Counter-Terrorism',
     ['terrori', 'al-qaeda', 'al qaeda', 'taliban',
      'counter-terror', 'isil', 'daesh']),
    ('Sanctions & Embargoes',
     ['sanction', 'embargo', 'arms embargo', 'asset freez']),
    ('Peacekeeping Operations',
     ['peacekeep', 'peace-keeping', 'unifil', 'unficyp', 'unmiss',
      'minusma', 'undof', 'observer force', 'interim force']),
    ('Sovereignty & Self-Determination',
     ['cyprus', 'kosovo', 'western sahara', 'east timor',
      'timor-leste', 'falkland', 'self-determin', 'territorial integrit']),
    ('Balkans & Former Yugoslavia',
     ['yugoslav', 'bosnia', 'herzegovina', 'serbia', 'croatia',
      'macedonia', 'balkan', 'icty', 'srebrenica']),
    ('Afghanistan',
     ['afghanistan', 'afghan', 'kabul', 'isaf']),
    ('Haiti',
     ['haiti', 'haitian', 'minustah']),
    ('Venezuela',
     ['venezuela', 'venezuelan', 'caracas']),
    ('Human Rights & Humanitarian',
     ['human right', 'humanitarian', 'sexual violence',
      'protection.*civilian', 'refugee', 'idp']),
    ('UN Administration',
     ['admission.*member', 'new member', 'secretary-general',
      'international court of justice', 'icj']),
]


def classify_theme(text):
    """Return first matching theme, or 'Other'."""
    if not isinstance(text, str):
        return 'Other'
    text_lower = text.lower()
    for theme, patterns in THEME_RULES:
        for p in patterns:
            if re.search(p, text_lower):
                return theme
    return 'Other'


# Theme each resolution
resolution_meta['combined_text'] = (
    resolution_meta['description'].fillna('').astype(str) + ' ' +
    resolution_meta['subjects'].fillna('').astype(str)
)
resolution_meta['theme'] = resolution_meta['combined_text'].apply(classify_theme)

theme_counts = resolution_meta['theme'].value_counts()
print(f"Theme distribution ({len(resolution_meta):,} adopted resolutions):")
for theme, count in theme_counts.items():
    bar = '█' * int(count / theme_counts.max() * 30)
    print(f"  {theme:<40} {count:>5}  {bar}")
other_pct = theme_counts.get('Other', 0) / len(resolution_meta) * 100
print(f"\nCoverage (non-Other): {100 - other_pct:.1f}%")

# Theme → coarse region for hierarchical matching
THEME_TO_REGION = {
    'Ukraine & Eastern Europe': 'Europe',
    'North Korea / DPRK': 'Asia',
    'Middle East — Iran': 'Middle East',
    'Middle East — Israel/Palestine': 'Middle East',
    'Middle East — Syria': 'Middle East',
    'Middle East — Iraq': 'Middle East',
    'Middle East — Yemen': 'Middle East',
    'Africa — Sudan & South Sudan': 'Africa',
    'Africa — Somalia': 'Africa',
    'Africa — Congo (DRC)': 'Africa',
    'Africa — Libya': 'Africa',
    'Africa — Rwanda & Great Lakes': 'Africa',
    'Africa — West Africa': 'Africa',
    'Africa — Other': 'Africa',
    'Balkans & Former Yugoslavia': 'Europe',
    'Afghanistan': 'Asia',
    'Haiti': 'Americas',
    'Venezuela': 'Americas',
    # Cross-cutting
    'Nuclear & Non-Proliferation': 'Global',
    'Terrorism & Counter-Terrorism': 'Global',
    'Sanctions & Embargoes': 'Global',
    'Peacekeeping Operations': 'Global',
    'Sovereignty & Self-Determination': 'Global',
    'Human Rights & Humanitarian': 'Global',
    'UN Administration': 'Global',
    'Other': 'Other',
}

# Extract simple keywords from description text — naive but mirrors what
# AAAI's pre-extracted keywords look like. For the historical pool we
# don't have hand-extracted keywords, so we approximate with top noun-ish
# terms from the description.
def extract_keywords(text, max_kw=8):
    """Naive keyword extraction: lowercase tokens >= 4 chars, stopwords removed."""
    if not isinstance(text, str):
        return []
    STOPWORDS = {
        'security', 'council', 'resolution', 'united', 'nations', 'shall',
        'with', 'from', 'this', 'that', 'have', 'will', 'been', 'their',
        'which', 'into', 'such', 'also', 'these', 'those', 'than', 'them',
        'were', 'about', 'between', 'before', 'after', 'mission', 'nation',
        'organisation', 'organization', 'group', 'panel', 'item', 'agenda',
        'note', 'noting', 'recall', 'recalling', 'reaffirm', 'reaffirming',
        'extension', 'mandate', 'situation', 'concerning',
    }
    tokens = re.findall(r'[a-z][a-z\-]{3,}', text.lower())
    counts = Counter(t for t in tokens if t not in STOPWORDS)
    return [w for w, _ in counts.most_common(max_kw)]


resolution_meta['keywords'] = resolution_meta['combined_text'].apply(extract_keywords)


# Build the candidate pool — adopted resolutions
ckg_pool = []
for _, row in resolution_meta.iterrows():
    res_id = row['resolution']
    if res_id not in voting_matrix.index:
        continue
    coalition_row = voting_matrix.loc[res_id]
    coalition = {}
    for code in P5:
        v = coalition_row.get(code)
        if pd.isna(v):
            coalition[code] = None  # nation not on Council
        else:
            coalition[code] = str(v)  # 'Y'/'N'/'A'

    theme = row['theme']
    ckg_pool.append({
        'resolution_id': res_id,
        'date': row['date'].to_pydatetime(),
        'year': int(row['year']),
        'description': row['description'] if pd.notna(row['description']) else '',
        'theme': theme,
        'geopolitical_region': THEME_TO_REGION.get(theme, 'Other'),
        'target_nations': [],  # adopted-pool resolutions don't have these
        'keywords': row['keywords'],
        'coalition': coalition,
        'is_adopted': True,
    })

print(f"\nAdopted resolutions added to CKG: {len(ckg_pool)}")

# Add veto events as non-adopted entries
n_veto_added = 0
for _, row in df_veto.iterrows():
    coalition = {}
    for code in P5:
        col = f'{code}_v'
        if col not in df_veto.columns:
            coalition[code] = None
            continue
        is_veto = (row[col] == 1)
        coalition[code] = 'N' if is_veto else None  # mark non-vetoer as unknown
    # NOTE: we set non-vetoer votes to None (not 'Y') because we don't actually
    # know their vote — they may have voted FOR or abstained. This avoids
    # biasing the historical FOR rate. Cleaner than the previous notebook.

    desc = str(row.get('agenda', '')) + ' ' + str(row.get('short_agenda', ''))
    theme = classify_theme(desc)
    ckg_pool.append({
        'resolution_id': str(row['draft_id']),
        'date': row['date'].to_pydatetime(),
        'year': int(row['year']),
        'description': desc.strip(),
        'theme': theme,
        'geopolitical_region': THEME_TO_REGION.get(theme, 'Other'),
        'target_nations': [],
        'keywords': extract_keywords(desc),
        'coalition': coalition,
        'is_adopted': False,
    })
    n_veto_added += 1

print(f"Veto events added: {n_veto_added}")
print(f"Total CKG pool: {len(ckg_pool)}")

# Sanity checks
n_adopted = sum(1 for r in ckg_pool if r['is_adopted'])
n_nonadopted = sum(1 for r in ckg_pool if not r['is_adopted'])
n_pre2013 = sum(1 for r in ckg_pool if r['date'] < datetime(2013, 1, 1))
print(f"\nBreakdown: {n_adopted} adopted, {n_nonadopted} non-adopted")
print(f"Pre-2013 (safe for profile): {n_pre2013}")

# Region distribution
print(f"\nRegion distribution in CKG:")
region_counts = Counter(r['geopolitical_region'] for r in ckg_pool)
for region, count in region_counts.most_common():
    print(f"  {region:<15} {count:>5}")

# Spot check: first 3 adopted, first 3 non-adopted
print("\nSample adopted entry:")
sample = next(r for r in ckg_pool if r['is_adopted'] and r['date'].year > 2010)
for k, v in sample.items():
    if k != 'description':
        print(f"  {k}: {v}")
print(f"  description: {sample['description'][:120]}...")

print("\nSample non-adopted (veto) entry:")
sample = next(r for r in ckg_pool if not r['is_adopted'])
for k, v in sample.items():
    if k != 'description':
        print(f"  {k}: {v}")

print("\n✓ Cell 6 complete — `ckg_pool` ready")

In [ ]:
# ============================================================
# CELL 7 — Inspect Other bucket + extract target nations
# ============================================================
# Two purposes:
# 1. Confirm "Other" is mostly procedural (acceptable) rather than
#    misclassified substantive resolutions.
# 2. Add target_nations extraction to CKG so AAAI baseline scores
#    fairly in the gate.

import re
from collections import Counter

# --- Part 1: Sample the "Other" bucket ---
print("OTHER BUCKET INSPECTION (random 15 from 360)")
print("=" * 70)
other_resolutions = [r for r in ckg_pool if r['theme'] == 'Other' and r['is_adopted']]
import random
random.seed(42)
for r in random.sample(other_resolutions, min(15, len(other_resolutions))):
    desc = r['description'][:110]
    print(f"  [{r['resolution_id']}] {desc}")

# --- Part 2: Country/place name extractor ---
# We use a curated list of countries + UN-relevant places. Regex against
# description + theme classification text. This mirrors what AAAI's
# augmented `target_nations_list` field looks like in shape.

COUNTRY_PATTERNS = {
    # Format: 'Canonical Name': [regex patterns to match]
    'Afghanistan': [r'\bafghanistan\b', r'\bafghan\b'],
    'Iraq': [r'\biraq\b', r'\biraqi\b'],
    'Iran': [r'\biran\b', r'\biranian\b'],
    'Syria': [r'\bsyria\b', r'\bsyrian\b'],
    'Yemen': [r'\byemen\b', r'\byemeni\b'],
    'Lebanon': [r'\blebanon\b', r'\blebanese\b'],
    'Israel': [r'\bisrael\b', r'\bisraeli\b'],
    'Palestine': [r'\bpalestin'],
    'Libya': [r'\blibya\b', r'\blibyan\b'],
    'Sudan': [r'\bsudan\b', r'\bsudanese\b', r'\bdarfur\b'],
    'South Sudan': [r'\bsouth sudan\b'],
    'Somalia': [r'\bsomalia\b', r'\bsomali\b'],
    'Congo': [r'\bcongo\b', r'\bdrc\b', r'democratic republic of the congo'],
    'Rwanda': [r'\brwanda\b', r'\brwandan\b'],
    'Burundi': [r'\bburundi\b'],
    'Mali': [r'\bmali\b', r'\bmalian\b'],
    'Sierra Leone': [r'\bsierra leone\b'],
    'Liberia': [r'\bliberia\b', r'\bliberian\b'],
    'Ethiopia': [r'\bethiopia\b', r'\bethiopian\b'],
    'Eritrea': [r'\beritrea\b'],
    'Côte d\'Ivoire': [r"c[oô]te d.ivoire", r'ivory coast', r'ivoirian'],
    'Central African Republic': [r'central african republic', r'\bcar\b'],
    'Burkina Faso': [r'burkina faso'],
    'Guinea': [r'\bguinea\b'],
    'Guinea-Bissau': [r'guinea[- ]bissau'],
    'Niger': [r'\bniger\b'],
    'Nigeria': [r'\bnigeria\b'],
    'Chad': [r'\bchad\b'],
    'Cameroon': [r'\bcameroon\b'],
    'Angola': [r'\bangola\b'],
    'Mozambique': [r'\bmozambique\b'],
    'Zimbabwe': [r'\bzimbabwe\b'],
    'Namibia': [r'\bnamibia\b'],
    'Ukraine': [r'\bukraine\b', r'\bukrainian\b', r'\bcrimea\b'],
    'Russia': [r'\brussian federation\b'],  # avoid matching 'russia' alone
    'Bosnia and Herzegovina': [r'bosnia', r'herzegovina'],
    'Serbia': [r'\bserbia\b', r'\bserbian\b'],
    'Croatia': [r'\bcroatia\b', r'\bcroatian\b'],
    'Kosovo': [r'\bkosovo\b'],
    'Cyprus': [r'\bcyprus\b', r'\bcypriot\b'],
    'Western Sahara': [r'western sahara', r'\bminurso\b'],
    'Timor-Leste': [r'timor[- ]leste', r'east timor'],
    'Cambodia': [r'cambodia', r'cambodian', r'khmer'],
    'Myanmar': [r'\bmyanmar\b', r'\bburma\b'],
    'North Korea': [r'democratic people.*republic of korea', r'\bdprk\b',
                    r'north korea'],
    'Haiti': [r'\bhaiti\b', r'\bhaitian\b'],
    'Venezuela': [r'\bvenezuela\b', r'\bvenezuelan\b'],
    'Colombia': [r'\bcolombia\b', r'\bcolombian\b'],
    'Georgia': [r'\bgeorgia\b'],  # context disambiguates from US state
    'Kazakhstan': [r'\bkazakhstan\b'],
    'Kyrgyzstan': [r'kyrgyz'],
    'Tajikistan': [r'tajikistan'],
    'Kuwait': [r'\bkuwait\b', r'\bkuwaiti\b'],
}


def extract_target_nations(text):
    """Return canonical country names mentioned in text."""
    if not isinstance(text, str):
        return []
    text_lower = text.lower()
    found = []
    for canonical, patterns in COUNTRY_PATTERNS.items():
        for p in patterns:
            if re.search(p, text_lower):
                found.append(canonical)
                break
    return found


# Apply to all CKG entries
for r in ckg_pool:
    r['target_nations'] = extract_target_nations(r['description'])

# Stats
n_with_targets = sum(1 for r in ckg_pool if r['target_nations'])
n_pct = n_with_targets / len(ckg_pool) * 100
avg_targets = np.mean([len(r['target_nations']) for r in ckg_pool if r['target_nations']])

print("\n" + "=" * 70)
print("TARGET_NATIONS EXTRACTION STATS")
print("=" * 70)
print(f"CKG entries with at least 1 target nation: {n_with_targets:,}/{len(ckg_pool):,} ({n_pct:.1f}%)")
print(f"Average target nations per entry (when present): {avg_targets:.1f}")

# Show distribution of most common nations extracted
all_nations = [n for r in ckg_pool for n in r['target_nations']]
nation_counts = Counter(all_nations)
print(f"\nTop 15 most-mentioned nations across CKG:")
for nation, count in nation_counts.most_common(15):
    print(f"  {nation:<30} {count:>4}")

# Spot check: 5 random CKG entries showing extraction
print("\n" + "=" * 70)
print("EXTRACTION SPOT CHECK (5 random adopted entries with targets)")
print("=" * 70)
samples = [r for r in ckg_pool if r['is_adopted'] and r['target_nations']]
for r in random.sample(samples, min(5, len(samples))):
    print(f"\n  [{r['resolution_id']}] theme={r['theme']}")
    print(f"    description: {r['description'][:100]}...")
    print(f"    extracted targets: {r['target_nations']}")

# Cross-check: do extracted targets align with AAAI's hand-curated ones?
# Sample 5 AAAI test entries, run our extractor on their context, compare.
print("\n" + "=" * 70)
print("EXTRACTOR vs AAAI HAND-CURATED (5 test resolutions)")
print("=" * 70)
for tr in test_resolutions[:5]:
    extracted = extract_target_nations(tr['context'][:5000])  # first 5K chars
    print(f"\n  [{tr['resolution_id']}]")
    print(f"    AAAI targets:      {tr['target_nations']}")
    print(f"    Our extractor:     {extracted}")
    overlap = set(tr['target_nations']) & set(extracted)
    print(f"    Overlap: {len(overlap)}/{len(tr['target_nations'])}")

print("\n✓ Cell 7 complete")

In [ ]:
# ============================================================
# CELL 7-fix — Expand patterns, debug, retest
# ============================================================
# Adds: Kenya, Tanzania, Uganda, Egypt, Tunisia, Morocco, Algeria,
# Saudi Arabia, Bahrain, Qatar, UAE, Jordan, Turkey, Greece, India,
# Pakistan, Bangladesh, Sri Lanka, Indonesia, Philippines, Vietnam,
# Laos, Thailand, Cuba, Nicaragua, Honduras, Argentina, Brazil, Chile,
# Mexico, Peru, Bolivia, Ecuador.
# Also: Serbia (debug), Montenegro, Macedonia, Albania, North Macedonia,
# Czechia, Slovakia, Hungary, Poland, Romania, Bulgaria, Belarus, Moldova.

ADDITIONAL_PATTERNS = {
    # Africa
    'Kenya': [r'\bkenya\b', r'\bkenyan\b'],
    'Tanzania': [r'\btanzania\b', r'tanzanian'],
    'Uganda': [r'\buganda\b', r'ugandan'],
    'Egypt': [r'\begypt\b', r'egyptian'],
    'Tunisia': [r'tunisia', r'tunisian'],
    'Morocco': [r'morocco', r'moroccan'],
    'Algeria': [r'algeria', r'algerian'],
    'Mauritania': [r'mauritania', r'mauritanian'],
    'Senegal': [r'\bsenegal\b'],
    'Djibouti': [r'djibouti'],
    'Madagascar': [r'madagascar'],
    'South Africa': [r'south africa'],
    'Lesotho': [r'lesotho'],
    'Sierra Leone': [r'sierra leone'],  # keep, was already there
    # Middle East / Gulf
    'Saudi Arabia': [r'saudi arabia', r'\bsaudi\b'],
    'Bahrain': [r'\bbahrain\b', r'bahraini'],
    'Qatar': [r'\bqatar\b', r'qatari'],
    'United Arab Emirates': [r'united arab emirates', r'\buae\b'],
    'Oman': [r'\boman\b'],
    'Jordan': [r'\bjordan\b', r'jordanian'],
    'Turkey': [r'\bturkey\b', r'\bturkish\b'],
    # Europe
    'Greece': [r'\bgreece\b', r'\bgreek\b'],
    'Albania': [r'\balbania\b', r'albanian'],
    'Serbia': [r'\bserbia\b', r'\bserbian\b'],
    'Montenegro': [r'montenegro', r'montenegrin'],
    'North Macedonia': [r'north macedonia', r'macedonian'],
    'Hungary': [r'hungary', r'hungarian'],
    'Poland': [r'\bpoland\b', r'\bpolish\b'],
    'Romania': [r'romania', r'romanian'],
    'Bulgaria': [r'bulgaria', r'bulgarian'],
    'Belarus': [r'belarus', r'belarusian'],
    'Moldova': [r'moldova', r'moldovan'],
    'Czech Republic': [r'czech republic', r'czechia'],
    'Slovakia': [r'slovakia', r'slovak'],
    # Asia
    'India': [r'\bindia\b', r'\bindian\b'],
    'Pakistan': [r'pakistan', r'pakistani'],
    'Bangladesh': [r'bangladesh'],
    'Sri Lanka': [r'sri lanka', r'sri lankan'],
    'Indonesia': [r'indonesia', r'indonesian'],
    'Philippines': [r'philippines', r'philippine', r'filipino'],
    'Vietnam': [r'vietnam', r'viet nam'],
    'Laos': [r'\blao\b', r'\blaos\b'],
    'Thailand': [r'thailand', r'thai'],
    'Malaysia': [r'malaysia', r'malaysian'],
    'Singapore': [r'singapore'],
    'Mongolia': [r'mongolia'],
    'South Korea': [r'republic of korea', r'south korea'],
    # Americas
    'Cuba': [r'\bcuba\b', r'\bcuban\b'],
    'Nicaragua': [r'nicaragua', r'nicaraguan'],
    'Honduras': [r'honduras', r'honduran'],
    'El Salvador': [r'el salvador', r'salvadoran'],
    'Guatemala': [r'guatemala', r'guatemalan'],
    'Argentina': [r'argentina', r'argentinian', r'argentine'],
    'Brazil': [r'\bbrazil\b', r'brazilian'],
    'Chile': [r'\bchile\b', r'chilean'],
    'Mexico': [r'\bmexico\b', r'mexican'],
    'Peru': [r'\bperu\b', r'peruvian'],
    'Bolivia': [r'bolivia', r'bolivian'],
    'Ecuador': [r'ecuador', r'ecuadorian'],
    'Panama': [r'panama'],
    'Dominican Republic': [r'dominican republic'],
    # Other
    'Cyprus': [r'\bcyprus\b', r'\bcypriot\b'],  # keep
}

# Merge with existing patterns
COUNTRY_PATTERNS.update(ADDITIONAL_PATTERNS)
print(f"Total country patterns now: {len(COUNTRY_PATTERNS)}")


# Re-define extractor (same logic, just uses updated dict)
def extract_target_nations(text):
    if not isinstance(text, str):
        return []
    text_lower = text.lower()
    found = []
    for canonical, patterns in COUNTRY_PATTERNS.items():
        for p in patterns:
            if re.search(p, text_lower):
                found.append(canonical)
                break
    return found


# Re-apply to all CKG entries
for r in ckg_pool:
    r['target_nations'] = extract_target_nations(r['description'])

n_with_targets = sum(1 for r in ckg_pool if r['target_nations'])
n_pct = n_with_targets / len(ckg_pool) * 100
print(f"CKG coverage with targets: {n_with_targets:,}/{len(ckg_pool):,} ({n_pct:.1f}%)")


# === Debug Serbia case ===
print("\n" + "=" * 70)
print("DEBUG: Why didn't Serbia match in S/2015/508?")
print("=" * 70)
serbia_test = next(t for t in test_resolutions if t['resolution_id'] == 'S/2015/508')
context_5k = serbia_test['context'][:5000].lower()
print(f"  Length of full context: {len(serbia_test['context'])}")
print(f"  'serbia' in first 5000 chars: {'serbia' in context_5k}")
print(f"  'serbia' in full context: {'serbia' in serbia_test['context'].lower()}")
serbia_indices = [i for i in range(len(serbia_test['context']))
                  if serbia_test['context'].lower()[i:i+6] == 'serbia']
print(f"  Positions of 'serbia' in full context: {serbia_indices[:5]}")


# === Re-run cross-check on ALL 66 test entries (not just 5) ===
# This time, use full context not truncated
print("\n" + "=" * 70)
print("CROSS-CHECK ON ALL 66 TEST ENTRIES (full context)")
print("=" * 70)
total_aaai_targets = 0
total_overlap = 0
per_entry_stats = []
for tr in test_resolutions:
    extracted = set(extract_target_nations(tr['context']))  # full context
    aaai_set = set(tr['target_nations']) - {'United Nations', 'Russia',
                                              'United States', 'China',
                                              'France', 'United Kingdom',
                                              'Russian Federation'}
    # Exclude P5 + UN from comparison since they're meta-targets, not subjects
    if not aaai_set:
        continue
    overlap = aaai_set & extracted
    per_entry_stats.append({
        'res': tr['resolution_id'],
        'aaai': aaai_set,
        'extracted': extracted,
        'overlap': overlap,
        'recall': len(overlap) / len(aaai_set) if aaai_set else 1.0,
    })
    total_aaai_targets += len(aaai_set)
    total_overlap += len(overlap)

micro_recall = total_overlap / total_aaai_targets if total_aaai_targets else 0
macro_recall = np.mean([s['recall'] for s in per_entry_stats])
print(f"Entries evaluated: {len(per_entry_stats)}/66 (excluded entries with only P5/UN targets)")
print(f"Micro-recall (across all targets): {total_overlap}/{total_aaai_targets} = {micro_recall:.1%}")
print(f"Macro-recall (mean per-entry):     {macro_recall:.1%}")

# Show entries with low recall (worst 5)
print("\nWorst 5 entries by recall:")
worst = sorted(per_entry_stats, key=lambda s: s['recall'])[:5]
for s in worst:
    print(f"  [{s['res']}] recall={s['recall']:.0%}")
    print(f"    AAAI:      {sorted(s['aaai'])}")
    print(f"    Extracted: {sorted(s['extracted'])}")
    print(f"    Missed:    {sorted(s['aaai'] - s['extracted'])}")

print("\n✓ Cell 7-fix complete")

In [ ]:
# ============================================================
# CELL 8 — Region hierarchy + CSR scoring functions
# ============================================================
# All scoring logic in one place, fully self-contained.
# Reviewers can verify each component independently.

import numpy as np
from collections import defaultdict, Counter
from datetime import datetime

# --- Hierarchical region matching ---
# Both AAAI and CSR text scorers use this. Maps fine-grained AAAI labels
# (Eastern Europe, Sudan, Bosnia and Herzegovina) to the coarse buckets
# our CKG pool uses.

REGION_HIERARCHY = {
    # Middle East
    'Middle East': 'Middle East',
    # Africa variants
    'West Africa': 'Africa', 'Africa': 'Africa', 'Sudan': 'Africa',
    'East Africa': 'Africa', 'Central Africa': 'Africa',
    'Southern Africa': 'Africa', 'Horn of Africa': 'Africa',
    # Europe variants
    'Europe': 'Europe', 'Eastern Europe': 'Europe',
    'Western Europe': 'Europe', 'Balkans': 'Europe',
    'Bosnia and Herzegovina': 'Europe', 'Baltic Sea': 'Europe',
    # Asia variants
    'Asia': 'Asia', 'North East Asia': 'Asia', 'East Asia': 'Asia',
    'Central Asia': 'Asia', 'South Asia': 'Asia', 'Southeast Asia': 'Asia',
    # Americas variants
    'Americas': 'Americas', 'Venezuela': 'Americas',
    'Latin America': 'Americas', 'Caribbean': 'Americas',
    # Cross-cutting / unknown
    'Global': 'Global', 'Other': 'Other',
}


def normalize_region(region):
    """Map any fine-grained region label to its coarse bucket."""
    if not region:
        return 'Other'
    return REGION_HIERARCHY.get(region, 'Other')


def regions_match(region_a, region_b):
    """True if both regions normalize to the same coarse bucket."""
    if not region_a or not region_b:
        return False
    if region_a == region_b:
        return True
    return normalize_region(region_a) == normalize_region(region_b)


# --- Theme-conditional historical coalition profile ---
# Used by CSR consistency and informativeness scores.
# Includes a coarse-bucket fallback when (theme, nation) sample is too small.

class CoalitionThemeProfile:
    """
    Stores P(vote | theme, nation) computed from CKG, EXCLUDING any
    resolution dated >= cutoff_date (leakage prevention).

    Falls back to coarser buckets when (theme, nation) has fewer than
    `min_samples` historical observations:
      1. Try (theme, nation) — most specific
      2. Fall back to (region, nation)
      3. Fall back to (nation) — overall historical pattern
      4. Fall back to uniform [1/3, 1/3, 1/3] — no information
    """

    def __init__(self, ckg_resolutions, cutoff_date, min_samples=10):
        self.min_samples = min_samples
        self.theme_profile = defaultdict(lambda: defaultdict(Counter))
        self.region_profile = defaultdict(lambda: defaultdict(Counter))
        self.nation_profile = defaultdict(Counter)

        for r in ckg_resolutions:
            if r['date'] >= cutoff_date:
                continue
            for nation_code, vote in r['coalition'].items():
                if vote is None:
                    continue
                self.theme_profile[r['theme']][nation_code][vote] += 1
                self.region_profile[r['geopolitical_region']][nation_code][vote] += 1
                self.nation_profile[nation_code][vote] += 1

    @staticmethod
    def _to_dist(counter):
        total = sum(counter.values())
        if total == 0:
            return None
        return {v: counter.get(v, 0) / total for v in ['Y', 'N', 'A']}

    def expected_distribution(self, theme, region, nation):
        """Return P(vote | best available conditioning, nation)."""
        # Try theme-level first
        c = self.theme_profile[theme][nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        # Fall back to region-level
        c = self.region_profile[region][nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        # Fall back to overall nation pattern
        c = self.nation_profile[nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        # Last resort: uniform
        return {'Y': 1/3, 'N': 1/3, 'A': 1/3}

    def coverage_report(self):
        """Diagnostic: how many (theme, nation) pairs have enough data?"""
        n_theme_ok = 0
        n_theme_total = 0
        for theme, nations in self.theme_profile.items():
            for nation, counter in nations.items():
                n_theme_total += 1
                if sum(counter.values()) >= self.min_samples:
                    n_theme_ok += 1
        return {
            'theme_pairs_total': n_theme_total,
            'theme_pairs_ok': n_theme_ok,
            'theme_pairs_pct': n_theme_ok / n_theme_total if n_theme_total else 0,
            'min_samples': self.min_samples,
        }


# --- AAAI text scoring (Section A.4 of their paper, exactly) ---
# +2 if region matches, +1 per common target nation, +0.1 per keyword overlap.
# This is what we use as the AAAI baseline retriever in the gate.

def aaai_text_score(target_keywords, target_region, target_nations, candidate):
    score = 0.0
    if regions_match(target_region, candidate.get('geopolitical_region', '')):
        score += 2.0
    cand_nations = set(candidate.get('target_nations', []))
    score += 1.0 * len(set(target_nations) & cand_nations)
    cand_keywords = set(candidate.get('keywords', []))
    score += 0.1 * len(set(target_keywords) & cand_keywords)
    return score


# --- CSR's three additional signals on top of text ---

def coalition_consistency_score(candidate, target_theme, target_region, profile):
    """
    Fraction of P5 nations whose vote in this candidate matches the
    theme-typical pattern (computed leakage-safe).
    Range [0, 1]. Higher = candidate looks like a 'typical' coalition for theme.
    """
    matches, total = 0, 0
    for nation, vote in candidate['coalition'].items():
        if vote is None:
            continue
        dist = profile.expected_distribution(target_theme, target_region, nation)
        mode_vote = max(dist, key=dist.get)
        if vote == mode_vote:
            matches += 1
        total += 1
    return matches / total if total > 0 else 0.0


def informative_target_signal(candidate, target_nation, target_theme,
                              target_region, profile):
    """
    Reward precedents where the target nation cast a NON-modal vote.
    These reveal genuine national stance rather than consensus alignment.
    Range [0, 1]. Higher = rarer behavior, more informative.
    """
    cand_vote = candidate['coalition'].get(target_nation)
    if cand_vote is None:
        return 0.0
    dist = profile.expected_distribution(target_theme, target_region, target_nation)
    p = dist.get(cand_vote, 1/3)
    return 1.0 - p


def recency_score(candidate_date, target_date, half_life_days=1825.0):
    """5-year half-life exponential decay on temporal distance."""
    days = (target_date - candidate_date).days
    if days < 0:
        return 0.0  # Future precedents excluded
    return 0.5 ** (days / half_life_days)


# --- Final CSR scoring ---

def csr_score(target, target_nation, candidate, profile,
              w_text=1.0, w_consistency=0.5,
              w_informativeness=1.0, w_recency=0.3):
    """
    Combined CSR score = weighted sum of normalized text, consistency,
    informativeness, and recency. Default weights chosen so text and
    informativeness dominate; consistency and recency are tiebreakers.
    """
    s_text = aaai_text_score(
        target.get('keywords', []),
        target.get('geopolitical_region', ''),
        target.get('target_nations', []),
        candidate
    )
    s_text_norm = min(s_text / 5.0, 1.0)  # rough normalization
    s_consistency = coalition_consistency_score(
        candidate,
        target.get('theme', 'Other'),
        target.get('geopolitical_region', 'Other'),
        profile
    )
    s_informativeness = informative_target_signal(
        candidate, target_nation,
        target.get('theme', 'Other'),
        target.get('geopolitical_region', 'Other'),
        profile
    )
    s_recency = recency_score(candidate['date'], target['date'])

    return (w_text * s_text_norm
            + w_consistency * s_consistency
            + w_informativeness * s_informativeness
            + w_recency * s_recency)


def csr_retrieve(target, target_nation, pool_adopted, pool_nonadopted,
                 profile, k=1, weights=None):
    """
    Drop-in replacement for AAAI's retriever. Returns (top-k from adopted,
    top-k from non-adopted), preserving the pool-separated retrieval
    structure of the original pipeline.
    """
    weights = weights or {'w_text': 1.0, 'w_consistency': 0.5,
                          'w_informativeness': 1.0, 'w_recency': 0.3}

    def pick_topk(pool):
        eligible = [c for c in pool
                    if c['date'] < target['date']
                    and c['resolution_id'] != target.get('resolution_id')]
        scored = [(csr_score(target, target_nation, c, profile, **weights), c)
                  for c in eligible]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:k]]

    return pick_topk(pool_adopted), pick_topk(pool_nonadopted)


# --- Build the profile and verify ---
PROFILE_CUTOFF = datetime(2013, 1, 1)
profile = CoalitionThemeProfile(ckg_pool, PROFILE_CUTOFF, min_samples=10)

cov = profile.coverage_report()
print("PROFILE BUILT")
print("=" * 60)
print(f"Cutoff date (no leakage): {PROFILE_CUTOFF.date()}")
n_pre = sum(1 for r in ckg_pool if r['date'] < PROFILE_CUTOFF)
print(f"Resolutions used: {n_pre} (pre-{PROFILE_CUTOFF.year})")
print(f"(theme, nation) pairs with ≥{cov['min_samples']} samples: "
      f"{cov['theme_pairs_ok']}/{cov['theme_pairs_total']} "
      f"({cov['theme_pairs_pct']:.0%})")
print(f"\nPairs without enough theme data fall back to region, then nation.")

# Spot-check a few critical (theme, nation) profiles
print("\nKEY HISTORICAL DISTRIBUTIONS (pre-2013)")
print("=" * 60)
spotcheck = [
    ('Middle East — Israel/Palestine', 'USA'),
    ('Middle East — Israel/Palestine', 'RUS'),
    ('Africa — Sudan & South Sudan', 'CHN'),
    ('Ukraine & Eastern Europe', 'RUS'),  # might fall back to region
    ('Middle East — Syria', 'RUS'),
    ('Nuclear & Non-Proliferation', 'CHN'),
]
for theme, nation in spotcheck:
    region = THEME_TO_REGION.get(theme, 'Other')
    dist = profile.expected_distribution(theme, region, nation)
    n_samples = sum(profile.theme_profile[theme][nation].values())
    fallback = '' if n_samples >= 10 else f' (FALLBACK, theme n={n_samples})'
    print(f"  ({theme}, {nation}){fallback}")
    print(f"    FOR={dist['Y']:.1%}  AGAINST={dist['N']:.1%}  ABSTAIN={dist['A']:.1%}")

print("\n✓ Cell 8 complete")


In [ ]:
# ============================================================
# CELL 9 — Day-1 gate: does CSR beat AAAI on coalition match?
# ============================================================
# For each (test_resolution, target_nation) pair:
#   - retrieve top-1 via AAAI text scoring
#   - retrieve top-1 via CSR (text + 3 signals)
# Coalition match: ≥75% of OTHER P5 nations agree between target and retrieved.
# Decision rule:
#   mean Δ ≥ +10pp  → GATE PASSED, proceed to LLM experiments
#   mean Δ ≥ +5pp   → MARGINAL, tune weights via grid search
#   mean Δ < +5pp   → FAILED, redesign

import numpy as np
from collections import defaultdict


def coalition_match(target_coalition, retrieved_coalition, exclude_nation=None,
                    threshold=0.75):
    """≥threshold fraction of comparable P5 votes match."""
    matches, total = 0, 0
    for nation in P5:
        if nation == exclude_nation:
            continue
        tv = target_coalition.get(nation)
        rv = retrieved_coalition.get(nation)
        if tv is None or rv is None:
            continue
        if tv == rv:
            matches += 1
        total += 1
    return total > 0 and matches / total >= threshold


# Split CKG into the two pools AAAI uses
adopted_pool = [r for r in ckg_pool if r['is_adopted']]
nonadopted_pool = [r for r in ckg_pool if not r['is_adopted']]
full_pool = ckg_pool  # used by AAAI baseline (unified retrieval)

print(f"Pools: {len(adopted_pool)} adopted, {len(nonadopted_pool)} non-adopted")
print(f"Test resolutions: {len(test_resolutions)}")
print(f"Running gate ({len(test_resolutions) * len(P5)} (resolution, nation) pairs)...\n")


# Run the comparison
results = defaultdict(lambda: {'aaai_matches': 0, 'csr_matches': 0, 'total': 0})
disagreement_examples = []

import time
start = time.time()

for i, target in enumerate(test_resolutions):
    if i % 10 == 0 and i > 0:
        elapsed = time.time() - start
        eta = elapsed / i * (len(test_resolutions) - i)
        print(f"  [{i}/{len(test_resolutions)}] elapsed {elapsed:.0f}s, ETA {eta:.0f}s")

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue

        # AAAI baseline: unified pool, top-1 by text score
        eligible = [c for c in full_pool if c['date'] < target['date']]
        aaai_scored = [(aaai_text_score(target['keywords'],
                                          target['geopolitical_region'],
                                          target['target_nations'], c), c)
                       for c in eligible]
        aaai_scored.sort(key=lambda x: x[0], reverse=True)
        aaai_top = [c for _, c in aaai_scored[:1]]

        # CSR: pool-separated retrieval (matches AAAI's structure)
        csr_a, csr_n = csr_retrieve(target, nation_code,
                                      adopted_pool, nonadopted_pool,
                                      profile, k=1)
        csr_top = csr_a + csr_n

        # Score retrievals
        for r in aaai_top:
            if coalition_match(target['coalition'], r['coalition'],
                               exclude_nation=nation_code):
                results[nation_code]['aaai_matches'] += 1
        for r in csr_top:
            if coalition_match(target['coalition'], r['coalition'],
                               exclude_nation=nation_code):
                results[nation_code]['csr_matches'] += 1
        results[nation_code]['total'] += 1

        # Save first 5 disagreements for inspection
        if (len(disagreement_examples) < 5 and aaai_top and csr_top
                and aaai_top[0]['resolution_id'] != csr_top[0]['resolution_id']):
            disagreement_examples.append({
                'target': target['resolution_id'],
                'target_date': target['date'].date(),
                'target_region': target['geopolitical_region'],
                'target_coalition': target['coalition'],
                'predicting_nation': nation_code,
                'aaai_retrieved': aaai_top[0]['resolution_id'],
                'aaai_retrieved_theme': aaai_top[0]['theme'],
                'aaai_retrieved_coalition': aaai_top[0]['coalition'],
                'csr_retrieved': csr_top[0]['resolution_id'],
                'csr_retrieved_theme': csr_top[0]['theme'],
                'csr_retrieved_coalition': csr_top[0]['coalition'],
            })

elapsed = time.time() - start
print(f"\nCompleted in {elapsed:.0f}s")


# === Print the gate result ===
print("\n" + "=" * 60)
print("GATE RESULT — Coalition Match Rate at k=1")
print("=" * 60)
print(f"{'Nation':<6} {'AAAI MR':>10} {'CSR MR':>10} {'Δ pp':>10}")
print("-" * 60)
deltas = []
for nation in P5:
    s = results[nation]
    if s['total'] == 0:
        continue
    aaai_mr = s['aaai_matches'] / s['total'] * 100
    csr_mr = s['csr_matches'] / s['total'] * 100
    delta = csr_mr - aaai_mr
    deltas.append(delta)
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else "✗"
    print(f"{nation:<6} {aaai_mr:>9.1f}% {csr_mr:>9.1f}% {delta:>+8.1f}pp  {flag}")

mean_delta = np.mean(deltas)
print("-" * 60)
print(f"{'Mean Δ':<28} {mean_delta:>+8.1f}pp")

print("\n" + "=" * 60)
print("VERDICT")
print("=" * 60)
if mean_delta >= 10:
    print("✓ GATE PASSED — proceed to LLM experiments")
elif mean_delta >= 5:
    print("≈ GATE MARGINAL — tune CSR weights via grid search before scaling")
else:
    print("✗ GATE FAILED — redesign required, do not run LLM experiments yet")


# === Print disagreement examples ===
print("\n" + "=" * 60)
print("RETRIEVAL DISAGREEMENT EXAMPLES")
print("=" * 60)
for i, ex in enumerate(disagreement_examples, 1):
    print(f"\n[{i}] Target {ex['target']} ({ex['target_date']}, "
          f"region={ex['target_region']}, predicting {ex['predicting_nation']})")
    print(f"    Target coalition:   {ex['target_coalition']}")
    print(f"    AAAI retrieved:     {ex['aaai_retrieved']}")
    print(f"      Theme:            {ex['aaai_retrieved_theme']}")
    print(f"      Coalition:        {ex['aaai_retrieved_coalition']}")
    print(f"    CSR retrieved:      {ex['csr_retrieved']}")
    print(f"      Theme:            {ex['csr_retrieved_theme']}")
    print(f"      Coalition:        {ex['csr_retrieved_coalition']}")

In [ ]:
# ============================================================
# CELL 9.5 — Expanded gate diagnostics + tie-breaking baseline
# ============================================================
# Re-runs gate with three retrievers:
#   1. AAAI (raw): text score, ties broken by sort order (their actual code)
#   2. AAAI (recency-broken): text score, ties broken by date desc
#   3. CSR: full coalition-aware scoring
#
# Plus per-region breakdown, net advantage counts, and stratified
# disagreement sampling.

import numpy as np
from collections import defaultdict, Counter
import json
import random
import time
import os

# --- AAAI variant: tie-broken by recency ---
def aaai_retrieve_with_tiebreak(target, pool, by='recency'):
    """AAAI scoring with explicit tie-breaking. Returns top-1."""
    eligible = [c for c in pool if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        # Add small recency tiebreaker so ties are broken consistently
        if by == 'recency':
            tie = (target['date'] - c['date']).days * -1e-6
            scored.append((s + tie, c))
        else:
            scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else None


# Re-run gate with all three retrievers
results_aaai = defaultdict(lambda: {'matches': 0, 'total': 0})
results_aaai_recency = defaultdict(lambda: {'matches': 0, 'total': 0})
results_csr = defaultdict(lambda: {'matches': 0, 'total': 0})

results_region_aaai = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_aaai_recency = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_csr = defaultdict(lambda: {'matches': 0, 'total': 0})

all_disagreements = []

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 15 == 0 and i > 0:
        elapsed = time.time() - start
        print(f"  [{i}/{len(test_resolutions)}] elapsed {elapsed:.0f}s")
    region = target['geopolitical_region']

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue

        # Three retrievers
        aaai_top = aaai_retrieve_with_tiebreak(target, ckg_pool, by='sort')
        aaai_recency_top = aaai_retrieve_with_tiebreak(target, ckg_pool, by='recency')
        csr_a, csr_n = csr_retrieve(target, nation_code,
                                      adopted_pool, nonadopted_pool,
                                      profile, k=1)
        csr_top = (csr_a + csr_n)[0] if (csr_a or csr_n) else None

        # Score each
        m_aaai = (aaai_top is not None and
                  coalition_match(target['coalition'], aaai_top['coalition'],
                                  exclude_nation=nation_code))
        m_aaai_r = (aaai_recency_top is not None and
                    coalition_match(target['coalition'],
                                    aaai_recency_top['coalition'],
                                    exclude_nation=nation_code))
        m_csr = (csr_top is not None and
                 coalition_match(target['coalition'], csr_top['coalition'],
                                 exclude_nation=nation_code))

        # Per nation
        for bucket, hit in [(results_aaai[nation_code], m_aaai),
                             (results_aaai_recency[nation_code], m_aaai_r),
                             (results_csr[nation_code], m_csr)]:
            bucket['total'] += 1
            if hit: bucket['matches'] += 1

        # Per region
        for bucket, hit in [(results_region_aaai[region], m_aaai),
                             (results_region_aaai_recency[region], m_aaai_r),
                             (results_region_csr[region], m_csr)]:
            bucket['total'] += 1
            if hit: bucket['matches'] += 1

        # Save disagreements between AAAI-recency and CSR (the fair fight)
        if (aaai_recency_top is not None and csr_top is not None
                and aaai_recency_top['resolution_id'] != csr_top['resolution_id']):
            all_disagreements.append({
                'target_id': target['resolution_id'],
                'target_date': str(target['date'].date()),
                'target_region': region,
                'predicting_nation': nation_code,
                'target_coalition': dict(target['coalition']),
                'aaai_id': aaai_recency_top['resolution_id'],
                'aaai_theme': aaai_recency_top['theme'],
                'aaai_coalition': dict(aaai_recency_top['coalition']),
                'aaai_match': m_aaai_r,
                'csr_id': csr_top['resolution_id'],
                'csr_theme': csr_top['theme'],
                'csr_coalition': dict(csr_top['coalition']),
                'csr_match': m_csr,
            })

elapsed = time.time() - start
print(f"\nCompleted in {elapsed:.0f}s")
print(f"Disagreements between AAAI-recency and CSR: {len(all_disagreements)}")


# === Per-nation table: 3 retrievers side by side ===
print("\n" + "=" * 80)
print("PER-NATION COALITION MATCH RATE (3 retrievers)")
print("=" * 80)
print(f"{'Nation':<6} {'AAAI raw':>10} {'AAAI+rec':>10} {'CSR':>10} "
      f"{'Δ vs raw':>10} {'Δ vs +rec':>11}")
print("-" * 80)
deltas_raw, deltas_recency = [], []
for nation in P5:
    a = results_aaai[nation]
    ar = results_aaai_recency[nation]
    c = results_csr[nation]
    if c['total'] == 0:
        continue
    a_mr = a['matches']/a['total']*100
    ar_mr = ar['matches']/ar['total']*100
    c_mr = c['matches']/c['total']*100
    d_raw = c_mr - a_mr
    d_rec = c_mr - ar_mr
    deltas_raw.append(d_raw)
    deltas_recency.append(d_rec)
    print(f"{nation:<6} {a_mr:>9.1f}% {ar_mr:>9.1f}% {c_mr:>9.1f}% "
          f"{d_raw:>+8.1f}pp {d_rec:>+9.1f}pp")
print("-" * 80)
print(f"{'Mean':<6} {'':>10} {'':>10} {'':>10} "
      f"{np.mean(deltas_raw):>+8.1f}pp {np.mean(deltas_recency):>+9.1f}pp")


# === Per-region table ===
print("\n" + "=" * 80)
print("PER-REGION COALITION MATCH RATE (CSR vs AAAI+recency)")
print("=" * 80)
print(f"{'Region':<25} {'AAAI+rec':>10} {'CSR':>10} {'Δ pp':>10} {'n':>5}")
print("-" * 80)
all_regions = sorted(results_region_csr.keys(),
                      key=lambda r: -results_region_csr[r]['total'])
for region in all_regions:
    ar = results_region_aaai_recency[region]
    c = results_region_csr[region]
    if c['total'] == 0:
        continue
    ar_mr = ar['matches']/ar['total']*100
    c_mr = c['matches']/c['total']*100
    delta = c_mr - ar_mr
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else ("·" if delta >= 0 else "✗")
    print(f"{region:<25} {ar_mr:>9.1f}% {c_mr:>9.1f}% "
          f"{delta:>+8.1f}pp {c['total']:>5}  {flag}")


# === Net advantage analysis ===
csr_wins = [d for d in all_disagreements if d['csr_match'] and not d['aaai_match']]
aaai_wins = [d for d in all_disagreements if d['aaai_match'] and not d['csr_match']]
both_wrong = [d for d in all_disagreements
              if not d['csr_match'] and not d['aaai_match']]
both_right = [d for d in all_disagreements
              if d['csr_match'] and d['aaai_match']]
print("\n" + "=" * 80)
print("NET ADVANTAGE (vs AAAI+recency)")
print("=" * 80)
print(f"CSR right, AAAI wrong:        {len(csr_wins):>4}")
print(f"AAAI right, CSR wrong:        {len(aaai_wins):>4}")
print(f"Both right (both retrieved different but matching coalitions): {len(both_right):>4}")
print(f"Both wrong:                   {len(both_wrong):>4}")
print(f"Net advantage to CSR:         {len(csr_wins) - len(aaai_wins):>+4}")


# === Stratified disagreement sampling ===
print("\n" + "=" * 80)
print("DISAGREEMENT SAMPLES (CSR right, AAAI wrong) — stratified by region")
print("=" * 80)
# Group csr_wins by region, sample 1 from each of top 4 regions
random.seed(7)
csr_wins_by_region = defaultdict(list)
for d in csr_wins:
    csr_wins_by_region[d['target_region']].append(d)
top_regions = sorted(csr_wins_by_region,
                      key=lambda r: -len(csr_wins_by_region[r]))[:4]
for region in top_regions:
    samples = csr_wins_by_region[region]
    d = random.choice(samples)
    print(f"\n--- {region} (n={len(samples)} CSR wins in this region) ---")
    print(f"Target {d['target_id']} ({d['target_date']}, "
          f"predicting {d['predicting_nation']})")
    print(f"  Target coalition:    {d['target_coalition']}")
    print(f"  AAAI retrieved:      {d['aaai_id']} (theme: {d['aaai_theme']})")
    print(f"    Coalition:         {d['aaai_coalition']}")
    print(f"  CSR retrieved:       {d['csr_id']} (theme: {d['csr_theme']})")
    print(f"    Coalition:         {d['csr_coalition']}")


# === Save ===
os.makedirs(os.path.join(WORKDIR, 'gate_results'), exist_ok=True)
with open(os.path.join(WORKDIR, 'gate_results/gate_v2.json'), 'w') as f:
    json.dump({
        'mean_delta_vs_raw': float(np.mean(deltas_raw)),
        'mean_delta_vs_recency': float(np.mean(deltas_recency)),
        'csr_wins': len(csr_wins),
        'aaai_wins': len(aaai_wins),
        'both_right': len(both_right),
        'both_wrong': len(both_wrong),
        'net_advantage': len(csr_wins) - len(aaai_wins),
        'per_nation': {n: {
            'aaai_raw': results_aaai[n]['matches']/results_aaai[n]['total']*100,
            'aaai_recency': results_aaai_recency[n]['matches']/results_aaai_recency[n]['total']*100,
            'csr': results_csr[n]['matches']/results_csr[n]['total']*100,
            'n': results_csr[n]['total'],
        } for n in P5 if results_csr[n]['total']},
        'per_region': {r: {
            'aaai_recency': results_region_aaai_recency[r]['matches']/results_region_aaai_recency[r]['total']*100,
            'csr': results_region_csr[r]['matches']/results_region_csr[r]['total']*100,
            'n': results_region_csr[r]['total'],
        } for r in all_regions},
    }, f, indent=2)
print(f"\n✓ Saved to gate_results/gate_v2.json")
print("✓ Cell 9.5 complete")

In [ ]:
# ============================================================
# CELL 10 — Diagnostic: is the AAAI top-50 coalition-diverse?
# ============================================================
# For each test resolution, retrieve AAAI's top-50 and check what
# fraction have unanimous-Y coalitions. If ≥30% of top-50 are
# non-unanimous, Path A (reranking) is viable. If <20%, Path A is dead
# and we go straight to Path B.

from collections import defaultdict, Counter
import numpy as np

def is_unanimous_y(coalition):
    """All 5 P5 voted Y."""
    return all(coalition.get(n) == 'Y' for n in P5)

def is_split(coalition):
    """At least 2 of 5 voted differently from the rest."""
    votes = [coalition.get(n) for n in P5 if coalition.get(n) is not None]
    return len(set(votes)) >= 2 if len(votes) >= 4 else False


# For each test resolution, get AAAI's top-50 and analyze its coalition diversity
top50_diversity = []
top50_split_counts = []
contested_in_top50 = []  # how many top-50 have non-unanimous coalitions

for target in test_resolutions:
    eligible = [c for c in ckg_pool if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        # recency tie-break
        tie = (target['date'] - c['date']).days * -1e-6
        scored.append((s + tie, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in scored[:50]]

    n_unanimous = sum(1 for c in top50 if is_unanimous_y(c['coalition']))
    n_split = sum(1 for c in top50 if is_split(c['coalition']))
    top50_diversity.append(n_unanimous / len(top50))
    top50_split_counts.append(n_split)
    contested_in_top50.append(n_split)


print("AAAI TOP-50 COALITION DIVERSITY ANALYSIS")
print("=" * 60)
print(f"Across {len(test_resolutions)} test resolutions, AAAI top-50:")
print(f"  Mean fraction unanimous-Y:  {np.mean(top50_diversity):.1%}")
print(f"  Median fraction unanimous-Y: {np.median(top50_diversity):.1%}")
print(f"  Mean # split coalitions:    {np.mean(top50_split_counts):.1f}")
print(f"  Median # split coalitions:  {np.median(top50_split_counts):.1f}")
print(f"  Min # split coalitions:     {min(top50_split_counts)}")
print(f"  Max # split coalitions:     {max(top50_split_counts)}")

# Histogram of how many test resolutions have N split candidates in top-50
print("\nDistribution: # split candidates in top-50, across 66 test resolutions")
buckets = Counter()
for n in top50_split_counts:
    if n == 0: buckets['0'] += 1
    elif n <= 5: buckets['1-5'] += 1
    elif n <= 15: buckets['6-15'] += 1
    elif n <= 30: buckets['16-30'] += 1
    else: buckets['31+'] += 1
for k in ['0', '1-5', '6-15', '16-30', '31+']:
    print(f"  {k:<8} split candidates: {buckets[k]:>3} test resolutions")


# Now: how often do we have a coalition-MATCHING candidate hidden in the top-50?
# This is the upper bound on what perfect reranking could achieve.
print("\n" + "=" * 60)
print("ORACLE RERANKING: best possible top-1 from AAAI's top-50")
print("=" * 60)
oracle_results = defaultdict(lambda: {'matches': 0, 'total': 0})
for target in test_resolutions:
    eligible = [c for c in ckg_pool if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        scored.append((s + tie, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in scored[:50]]

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue
        # Oracle: pick the top-50 candidate that maximizes coalition match
        best_match = any(coalition_match(target['coalition'], c['coalition'],
                                          exclude_nation=nation_code)
                          for c in top50)
        oracle_results[nation_code]['total'] += 1
        if best_match:
            oracle_results[nation_code]['matches'] += 1

print(f"\n{'Nation':<6} {'AAAI+rec MR':>14} {'Oracle MR':>12} {'Headroom':>10}")
print("-" * 50)
for nation in P5:
    o = oracle_results[nation]
    a = results_aaai_recency[nation]
    if o['total'] == 0:
        continue
    o_mr = o['matches']/o['total']*100
    a_mr = a['matches']/a['total']*100
    headroom = o_mr - a_mr
    print(f"{nation:<6} {a_mr:>13.1f}% {o_mr:>11.1f}% {headroom:>+9.1f}pp")

print("\nINTERPRETATION:")
print("  Headroom = max possible Δ from reranking the AAAI top-50.")
print("  If headroom is ≥+15pp on most nations, Path A (reranker) is viable.")
print("  If headroom is <+5pp, Path A cannot win and we go Path B.")

In [ ]:
# ============================================================
# CELL 11 — CSR v2: coalition-aware reranker
# ============================================================
# Architecture:
#   Stage 1 (recall): AAAI text scoring → top-50 candidates
#   Stage 2 (rerank): predict target coalition from theme history,
#                     rerank by Hamming match to predicted coalition.
#
# The reranker NEVER sees the target's actual coalition — only the
# theme-conditional historical distribution. This is what we'd have at
# test time when predicting an LLM's vote for a never-before-seen
# resolution.

import numpy as np


def predict_target_coalition(target, profile):
    """
    Predict the most likely P5 coalition for a target resolution from
    its theme/region alone, using the historical profile.

    Returns: dict {nation_code: 'Y'/'N'/'A'} representing the modal
    historical vote per nation conditional on theme (or region as fallback).

    This is the cleanest 'what would a theme-typical vote look like?'
    estimate. CSR ranks candidates by similarity to this predicted coalition.
    """
    theme = target.get('theme', 'Other')
    region = target.get('geopolitical_region', 'Other')
    predicted = {}
    for nation in P5:
        dist = profile.expected_distribution(theme, region, nation)
        predicted[nation] = max(dist, key=dist.get)
    return predicted


def coalition_hamming_similarity(coalition_a, coalition_b, exclude_nation=None):
    """
    Fraction of comparable P5 votes that match. Range [0, 1].
    Symmetric and handles None votes (skipped).
    """
    matches, total = 0, 0
    for nation in P5:
        if nation == exclude_nation:
            continue
        a, b = coalition_a.get(nation), coalition_b.get(nation)
        if a is None or b is None:
            continue
        if a == b:
            matches += 1
        total += 1
    return matches / total if total > 0 else 0.0


def csr_v2_retrieve(target, target_nation, ckg_pool, profile,
                    recall_k=50, final_k=1,
                    w_predicted=1.0, w_recency=0.2, w_text=0.1):
    """
    Two-stage CSR retrieval.

    Stage 1: AAAI text scoring → top recall_k candidates.
    Stage 2: Rerank by similarity to PREDICTED target coalition, plus
             small weights for recency and text (as tiebreakers).

    Args:
        target: dict with theme, region, keywords, target_nations, date.
        target_nation: which nation we're predicting (excluded from
            coalition similarity calculation, since that's what we're
            trying to predict).
        recall_k: candidates to pull from text-stage retrieval.
        final_k: candidates to return after reranking.
        w_predicted: weight on coalition match to predicted target.
        w_recency: weight on recency.
        w_text: small weight on original text score for tiebreaking.

    Returns: list of top final_k reranked candidates.
    """
    # Stage 1: text recall
    eligible = [c for c in ckg_pool if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        # Recency tiebreaker inside text retrieval
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, s, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = text_scored[:recall_k]
    if not top_recall:
        return []

    # Stage 2: predict target coalition, then rerank
    predicted_coalition = predict_target_coalition(target, profile)

    reranked = []
    for combined_score, raw_text_score, candidate in top_recall:
        # Primary: how well does candidate match predicted target coalition?
        # Exclude target_nation so we're not using info we'd want to predict.
        sim_to_predicted = coalition_hamming_similarity(
            predicted_coalition, candidate['coalition'],
            exclude_nation=target_nation
        )

        # Tiebreaker 1: recency (continuous, not just sign)
        days = (target['date'] - candidate['date']).days
        rec = 0.5 ** (days / 1825.0)

        # Tiebreaker 2: small text weight to break ties among candidates with
        # identical predicted-coalition match
        text_norm = min(raw_text_score / 5.0, 1.0)

        rerank_score = (w_predicted * sim_to_predicted
                        + w_recency * rec
                        + w_text * text_norm)
        reranked.append((rerank_score, candidate))

    reranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in reranked[:final_k]]


# === Run the gate against AAAI+recency baseline ===
print("CSR v2 GATE — two-stage rerank")
print("=" * 60)

results_csr_v2 = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_csr_v2 = defaultdict(lambda: {'matches': 0, 'total': 0})

import time
start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 15 == 0 and i > 0:
        elapsed = time.time() - start
        print(f"  [{i}/{len(test_resolutions)}] elapsed {elapsed:.0f}s")
    region = target['geopolitical_region']

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue

        csr_v2_top = csr_v2_retrieve(target, nation_code, ckg_pool, profile,
                                      recall_k=50, final_k=1)
        m = (csr_v2_top and
             coalition_match(target['coalition'], csr_v2_top[0]['coalition'],
                             exclude_nation=nation_code))

        for bucket in [results_csr_v2[nation_code], results_region_csr_v2[region]]:
            bucket['total'] += 1
            if m:
                bucket['matches'] += 1

elapsed = time.time() - start
print(f"\nCompleted in {elapsed:.0f}s")


# === Per-nation comparison: AAAI+recency vs CSR-v1 vs CSR-v2 vs Oracle ===
print("\n" + "=" * 80)
print("FOUR-WAY COMPARISON")
print("=" * 80)
print(f"{'Nation':<6} {'AAAI+rec':>10} {'CSR-v1':>10} {'CSR-v2':>10} "
      f"{'Oracle':>10} {'v2 capture':>12}")
print("-" * 80)
for nation in P5:
    a = results_aaai_recency[nation]
    v1 = results_csr[nation]
    v2 = results_csr_v2[nation]
    o = oracle_results[nation]
    if v2['total'] == 0:
        continue
    a_mr = a['matches']/a['total']*100
    v1_mr = v1['matches']/v1['total']*100
    v2_mr = v2['matches']/v2['total']*100
    o_mr = o['matches']/o['total']*100
    headroom = o_mr - a_mr
    captured = (v2_mr - a_mr) / headroom * 100 if headroom > 0 else 0
    print(f"{nation:<6} {a_mr:>9.1f}% {v1_mr:>9.1f}% {v2_mr:>9.1f}% "
          f"{o_mr:>9.1f}% {captured:>10.0f}% of headroom")

# Mean Δ
deltas_v2 = []
for nation in P5:
    a = results_aaai_recency[nation]
    v2 = results_csr_v2[nation]
    if v2['total'] == 0:
        continue
    deltas_v2.append(v2['matches']/v2['total']*100 - a['matches']/a['total']*100)
print("-" * 80)
print(f"Mean Δ (CSR-v2 vs AAAI+rec): {np.mean(deltas_v2):+.1f}pp")


# === Per-region ===
print("\n" + "=" * 80)
print("PER-REGION (CSR-v2 vs AAAI+recency)")
print("=" * 80)
print(f"{'Region':<25} {'AAAI+rec':>10} {'CSR-v2':>10} {'Δ pp':>10} {'n':>5}")
print("-" * 70)
for region in sorted(results_region_csr_v2.keys(),
                      key=lambda r: -results_region_csr_v2[r]['total']):
    ar = results_region_aaai_recency[region]
    v2 = results_region_csr_v2[region]
    if v2['total'] == 0:
        continue
    ar_mr = ar['matches']/ar['total']*100
    v2_mr = v2['matches']/v2['total']*100
    delta = v2_mr - ar_mr
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else ("·" if delta >= 0 else "✗")
    print(f"{region:<25} {ar_mr:>9.1f}% {v2_mr:>9.1f}% "
          f"{delta:>+8.1f}pp {v2['total']:>5}  {flag}")

print("\n✓ Cell 11 complete")

In [ ]:
# ============================================================
# CELL 12 — Diagnose the predicted-coalition collapse
# ============================================================
# Two questions:
# (1) For each test resolution, what does predict_target_coalition return?
#     Confirm hypothesis: it's almost always all-Y.
# (2) For each test resolution, what's the actual coalition? Where does
#     the actual coalition deviate from "all Y"? This shows us the
#     dissent pattern we need to model.

from collections import Counter

print("Q1: Predicted coalition distribution across 66 test resolutions")
print("=" * 60)
predicted_patterns = Counter()
for target in test_resolutions:
    pred = predict_target_coalition(target, profile)
    pattern = ''.join(pred[n] for n in P5)
    predicted_patterns[pattern] += 1

print(f"Predicted coalitions (P5 order = CHN, FRA, GBR, RUS, USA):")
for pat, count in predicted_patterns.most_common():
    print(f"  {pat}  →  {count:>3} resolutions")

# Confirm hypothesis
all_y = predicted_patterns.get('YYYYY', 0)
print(f"\nFraction predicted as all-Y: {all_y}/66 = {all_y/66:.0%}")


print("\n" + "=" * 60)
print("Q2: Actual coalition dissent patterns in test set")
print("=" * 60)
# For each test resolution, identify who dissented (voted not-Y)
dissent_patterns = Counter()
for target in test_resolutions:
    dissenters = []
    for n in P5:
        v = target['coalition'].get(n)
        if v is not None and v != 'Y':
            dissenters.append(n)
    if dissenters:
        dissent_patterns[tuple(sorted(dissenters))] += 1
    else:
        dissent_patterns[('none',)] += 1

print(f"Who dissented (voted N or A) in actual test resolutions:")
for pat, count in dissent_patterns.most_common(15):
    print(f"  {str(pat):<40}  →  {count:>3} resolutions")


print("\n" + "=" * 60)
print("Q3: For each test region, what's the typical dissent pattern?")
print("=" * 60)
# This tells us what 'expected dissent profile' should look like per region
region_dissent = defaultdict(Counter)
for target in test_resolutions:
    region = target['geopolitical_region']
    dissenters = tuple(sorted(n for n in P5
                               if target['coalition'].get(n) not in ('Y', None)))
    region_dissent[region][dissenters] += 1

for region in sorted(region_dissent, key=lambda r: -sum(region_dissent[r].values())):
    total = sum(region_dissent[region].values())
    if total < 3:
        continue
    print(f"\n{region} (n={total}):")
    for pat, c in region_dissent[region].most_common(5):
        pat_str = ', '.join(pat) if pat else 'unanimous-Y'
        print(f"  {pat_str:<35}  {c}/{total} = {c/total:.0%}")


print("\n" + "=" * 60)
print("Q4: Probability of each P5 nation dissenting, by region")
print("=" * 60)
# This is the signal we want: P(nation dissents | region) computed from
# the test set itself just to see the structure. We'll build this from
# pre-2013 history for the actual reranker.
print(f"\n{'Region':<25} {'CHN':>7} {'FRA':>7} {'GBR':>7} {'RUS':>7} {'USA':>7}  n")
print("-" * 70)
for region in sorted(region_dissent, key=lambda r: -sum(region_dissent[r].values())):
    total = sum(region_dissent[region].values())
    if total < 3:
        continue
    p_dissent = {n: 0 for n in P5}
    for target in test_resolutions:
        if target['geopolitical_region'] != region:
            continue
        for n in P5:
            v = target['coalition'].get(n)
            if v is not None and v != 'Y':
                p_dissent[n] += 1
    row = f"{region:<25}"
    for n in P5:
        row += f" {p_dissent[n]/total:>6.0%}"
    row += f"  {total}"
    print(row)


print("\n" + "=" * 60)
print("Q5: Same probability, computed from PRE-2013 history (the profile)")
print("=" * 60)
# This is what the reranker actually has access to — no leakage
print(f"\n{'Region':<25} {'CHN':>7} {'FRA':>7} {'GBR':>7} {'RUS':>7} {'USA':>7}")
print("-" * 70)
historical_dissent = defaultdict(lambda: {n: [0, 0] for n in P5})
for r in ckg_pool:
    if r['date'] >= datetime(2013, 1, 1):
        continue
    region = r['geopolitical_region']
    for n in P5:
        v = r['coalition'].get(n)
        if v is None:
            continue
        historical_dissent[region][n][1] += 1  # total
        if v != 'Y':
            historical_dissent[region][n][0] += 1  # dissented

# Map test regions to coarse buckets and look up
region_to_coarse = {
    'Middle East': 'Middle East',
    'Eastern Europe': 'Europe',
    'Global': 'Global',
    'West Africa': 'Africa',
    'Africa': 'Africa',
    'Sudan': 'Africa',
    'Venezuela': 'Americas',
    'North East Asia': 'Asia',
    'Bosnia and Herzegovina': 'Europe',
    'Europe': 'Europe',
    'Baltic Sea': 'Europe',
}
seen = set()
for test_region in region_dissent.keys():
    coarse = region_to_coarse.get(test_region, 'Other')
    if coarse in seen:
        continue
    seen.add(coarse)
    if coarse not in historical_dissent:
        continue
    row = f"{coarse:<25}"
    for n in P5:
        d, t = historical_dissent[coarse][n]
        row += f" {d/t:>6.0%}" if t > 0 else "    n/a"
    print(row)

In [ ]:
# ============================================================
# CELL 13 — Coalition entropy reranker (CSR v3)
# ============================================================
# Hypothesis: Don't predict the target's coalition. Just prefer
# coalition-diverse precedents. They teach the LLM that contested
# resolutions exist; unanimous precedents don't.
#
# Score = coalition_entropy(candidate) + small recency + small text
# No prediction needed → no era mismatch.

import math
from collections import Counter, defaultdict
import numpy as np


def coalition_entropy(coalition):
    """
    Shannon entropy over the P5 vote distribution within a coalition.
    Range [0, log2(3)] ≈ [0, 1.585].
    All-Y gets 0. A 3-2 split gets ~0.97. A 2-2-1 gets ~1.5.
    Higher entropy = more political contestation in the precedent.
    """
    votes = [coalition.get(n) for n in P5 if coalition.get(n) is not None]
    if not votes:
        return 0.0
    counts = Counter(votes)
    total = len(votes)
    entropy = 0.0
    for v, c in counts.items():
        p = c / total
        entropy -= p * math.log2(p)
    return entropy


def csr_v3_retrieve(target, target_nation, ckg_pool, profile,
                    recall_k=50, final_k=1,
                    w_entropy=1.0, w_recency=0.3, w_text=0.2):
    """
    Two-stage CSR with entropy-based reranking.

    Stage 1: AAAI text scoring → top recall_k candidates.
    Stage 2: Rerank by coalition entropy + recency + text tiebreaker.
    """
    eligible = [c for c in ckg_pool if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, s, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = text_scored[:recall_k]
    if not top_recall:
        return []

    # Normalize entropy to [0, 1] dividing by log2(3) ≈ 1.585
    MAX_ENTROPY = math.log2(3)

    reranked = []
    for combined_score, raw_text_score, candidate in top_recall:
        ent = coalition_entropy(candidate['coalition']) / MAX_ENTROPY
        days = (target['date'] - candidate['date']).days
        rec = 0.5 ** (days / 1825.0)
        text_norm = min(raw_text_score / 5.0, 1.0)
        rerank_score = w_entropy * ent + w_recency * rec + w_text * text_norm
        reranked.append((rerank_score, candidate))

    reranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in reranked[:final_k]]


# === Run gate ===
print("CSR v3 GATE — entropy-based rerank")
print("=" * 60)
results_csr_v3 = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_csr_v3 = defaultdict(lambda: {'matches': 0, 'total': 0})

import time
start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 15 == 0 and i > 0:
        print(f"  [{i}/{len(test_resolutions)}] elapsed {time.time()-start:.0f}s")
    region = target['geopolitical_region']
    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue
        v3_top = csr_v3_retrieve(target, nation_code, ckg_pool, profile,
                                  recall_k=50, final_k=1)
        m = (v3_top and
             coalition_match(target['coalition'], v3_top[0]['coalition'],
                             exclude_nation=nation_code))
        for bucket in [results_csr_v3[nation_code], results_region_csr_v3[region]]:
            bucket['total'] += 1
            if m:
                bucket['matches'] += 1
print(f"\nCompleted in {time.time()-start:.0f}s")


# === Five-way comparison ===
print("\n" + "=" * 90)
print("FIVE-WAY COMPARISON")
print("=" * 90)
print(f"{'Nation':<6} {'AAAI+rec':>10} {'CSR-v1':>10} {'CSR-v2':>10} "
      f"{'CSR-v3':>10} {'Oracle':>10} {'v3 capture':>12}")
print("-" * 90)
deltas_v3 = []
for nation in P5:
    a = results_aaai_recency[nation]
    v1 = results_csr[nation]
    v2 = results_csr_v2[nation]
    v3 = results_csr_v3[nation]
    o = oracle_results[nation]
    if v3['total'] == 0:
        continue
    a_mr = a['matches']/a['total']*100
    v1_mr = v1['matches']/v1['total']*100
    v2_mr = v2['matches']/v2['total']*100
    v3_mr = v3['matches']/v3['total']*100
    o_mr = o['matches']/o['total']*100
    headroom = o_mr - a_mr
    captured = (v3_mr - a_mr) / headroom * 100 if headroom > 0 else 0
    deltas_v3.append(v3_mr - a_mr)
    print(f"{nation:<6} {a_mr:>9.1f}% {v1_mr:>9.1f}% {v2_mr:>9.1f}% "
          f"{v3_mr:>9.1f}% {o_mr:>9.1f}% {captured:>10.0f}%")
print("-" * 90)
print(f"Mean Δ v3 vs AAAI+rec: {np.mean(deltas_v3):+.1f}pp")


# === Per-region ===
print("\n" + "=" * 70)
print("PER-REGION (CSR-v3 vs AAAI+recency)")
print("=" * 70)
print(f"{'Region':<25} {'AAAI+rec':>10} {'CSR-v3':>10} {'Δ pp':>10} {'n':>5}")
print("-" * 70)
for region in sorted(results_region_csr_v3.keys(),
                      key=lambda r: -results_region_csr_v3[r]['total']):
    ar = results_region_aaai_recency[region]
    v3 = results_region_csr_v3[region]
    if v3['total'] == 0:
        continue
    ar_mr = ar['matches']/ar['total']*100
    v3_mr = v3['matches']/v3['total']*100
    delta = v3_mr - ar_mr
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else ("·" if delta >= 0 else "✗")
    print(f"{region:<25} {ar_mr:>9.1f}% {v3_mr:>9.1f}% "
          f"{delta:>+8.1f}pp {v3['total']:>5}  {flag}")


# === Sanity: what does the entropy distribution look like in AAAI top-50? ===
print("\n" + "=" * 60)
print("SANITY CHECK: Entropy distribution in AAAI top-50")
print("=" * 60)
all_entropies = []
chosen_entropies_v3 = []
for target in test_resolutions:
    eligible = [c for c in ckg_pool if c['date'] < target['date']]
    scored = [(aaai_text_score(target['keywords'],
                                 target['geopolitical_region'],
                                 target['target_nations'], c), c)
               for c in eligible]
    scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in scored[:50]]
    for c in top50:
        all_entropies.append(coalition_entropy(c['coalition']))
    # What does v3 actually pick?
    v3_top = csr_v3_retrieve(target, 'RUS', ckg_pool, profile, recall_k=50, final_k=1)
    if v3_top:
        chosen_entropies_v3.append(coalition_entropy(v3_top[0]['coalition']))

print(f"All top-50 candidates (n={len(all_entropies)}):")
print(f"  Mean entropy:   {np.mean(all_entropies):.3f}")
print(f"  Median entropy: {np.median(all_entropies):.3f}")
print(f"  Max entropy:    {np.max(all_entropies):.3f}")
print(f"  % zero-entropy (unanimous): {sum(1 for e in all_entropies if e == 0)/len(all_entropies):.0%}")

print(f"\nv3 picks (n={len(chosen_entropies_v3)}):")
print(f"  Mean entropy:   {np.mean(chosen_entropies_v3):.3f}")
print(f"  Max entropy:    {np.max(chosen_entropies_v3):.3f}")
print(f"  % zero-entropy: {sum(1 for e in chosen_entropies_v3 if e == 0)/len(chosen_entropies_v3):.0%}")

In [ ]:
# ============================================================
# CELL 14 — Generate training pairs for cross-encoder
# ============================================================
# For each pre-2013 adopted resolution R in the CKG:
#   1. Score all earlier candidates with AAAI text scorer
#   2. Take top-5 candidates (mimics inference-time top-K recall)
#   3. Label each pair (R, candidate) with coalition Hamming similarity
#
# This produces ~10K-14K pairs whose distribution mirrors what the
# reranker will see at inference time. Hard negatives (high text score
# but low coalition match) are naturally included.

import os
import json
import numpy as np
from datetime import datetime
from collections import Counter

print("Generating cross-encoder training pairs...")
print("=" * 60)

# We use only pre-2013 adopted resolutions as TARGETS for training.
# The non-adopted (veto) entries have None votes for non-vetoers,
# which makes coalition similarity computation noisy. Use them only
# as candidates, not as targets.
training_targets = [r for r in ckg_pool
                     if r['is_adopted']
                     and r['date'] < datetime(2013, 1, 1)]
print(f"Training targets (pre-2013 adopted): {len(training_targets)}")

# Candidate pool for any target T = all CKG entries dated before T
# (mirrors the chronological constraint at inference time)


def coalition_hamming_continuous(coalition_a, coalition_b):
    """
    Continuous Hamming similarity over P5.
    Both coalitions must have non-None votes for at least 3 nations
    (otherwise the comparison is unreliable; we skip those pairs).

    Returns float in [0, 1] where 1.0 = perfect match, or None if
    too few comparable votes.
    """
    matches, total = 0, 0
    for nation in P5:
        a = coalition_a.get(nation)
        b = coalition_b.get(nation)
        if a is None or b is None:
            continue
        total += 1
        if a == b:
            matches += 1
    if total < 3:
        return None
    return matches / total


def build_target_text(entry):
    """
    Concatenate the textual fields a target/candidate exposes.
    For test resolutions we'll use context+summary. For CKG entries
    we use the description (richer than just title).
    """
    parts = []
    if entry.get('description'):
        parts.append(entry['description'])
    return ' '.join(parts)[:1500]  # truncate for tokenizer efficiency


# Generate pairs
training_pairs = []
n_skipped_label = 0

import time
start = time.time()

for i, target in enumerate(training_targets):
    if i % 500 == 0:
        elapsed = time.time() - start
        print(f"  [{i}/{len(training_targets)}] elapsed {elapsed:.0f}s, "
              f"pairs so far: {len(training_pairs)}")

    # Score all earlier candidates with AAAI text scorer
    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target['resolution_id']]
    if not eligible:
        continue

    scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        # recency tiebreaker
        tie = (target['date'] - c['date']).days * -1e-6
        scored.append((s + tie, c))
    scored.sort(key=lambda x: x[0], reverse=True)

    # Take top-5 candidates as training pairs for this target
    target_text = build_target_text(target)
    if not target_text:
        continue

    for _, candidate in scored[:5]:
        candidate_text = build_target_text(candidate)
        if not candidate_text:
            continue

        label = coalition_hamming_continuous(target['coalition'],
                                               candidate['coalition'])
        if label is None:
            n_skipped_label += 1
            continue

        training_pairs.append({
            'target_id': target['resolution_id'],
            'target_date': target['date'].isoformat(),
            'target_text': target_text,
            'candidate_id': candidate['resolution_id'],
            'candidate_text': candidate_text,
            'label': label,
            'target_theme': target['theme'],
            'target_region': target['geopolitical_region'],
        })

elapsed = time.time() - start
print(f"\nCompleted in {elapsed:.0f}s")
print(f"Total training pairs: {len(training_pairs)}")
print(f"Pairs skipped due to insufficient labels: {n_skipped_label}")


# === Sanity stats: label distribution ===
print("\n" + "=" * 60)
print("LABEL DISTRIBUTION")
print("=" * 60)
labels = [p['label'] for p in training_pairs]
print(f"Mean label:   {np.mean(labels):.3f}")
print(f"Median label: {np.median(labels):.3f}")
print(f"Std:          {np.std(labels):.3f}")
print(f"\nDistribution:")
buckets = Counter()
for l in labels:
    if l == 1.0: buckets['1.0 (perfect)'] += 1
    elif l >= 0.8: buckets['0.8-0.99'] += 1
    elif l >= 0.6: buckets['0.6-0.79'] += 1
    elif l >= 0.4: buckets['0.4-0.59'] += 1
    elif l >= 0.2: buckets['0.2-0.39'] += 1
    else: buckets['0.0-0.19'] += 1
for k in ['1.0 (perfect)', '0.8-0.99', '0.6-0.79', '0.4-0.59', '0.2-0.39', '0.0-0.19']:
    n = buckets[k]
    pct = n / len(labels) * 100
    bar = '█' * int(pct / 2)
    print(f"  {k:<18} {n:>5} ({pct:>5.1f}%) {bar}")


# === Sanity: hard cases ===
# A "hard negative" is a pair with high AAAI text score but low coalition match.
# We want to confirm these exist in our training data — they are why the
# heuristic rerankers failed and what we want the cross-encoder to learn.
print("\n" + "=" * 60)
print("HARD NEGATIVES SAMPLE (text-similar but coalition-different)")
print("=" * 60)
# Pairs in top-1 position with label < 0.6 — these are the cases AAAI
# would retrieve but where the coalition doesn't actually match
hard_negatives = [p for p in training_pairs if p['label'] < 0.6]
print(f"Hard negatives (label < 0.6): {len(hard_negatives)} "
      f"({len(hard_negatives)/len(training_pairs)*100:.1f}% of pairs)")

import random
random.seed(42)
print("\n3 random hard-negative examples:")
for p in random.sample(hard_negatives, min(3, len(hard_negatives))):
    print(f"\n  Target {p['target_id']} ({p['target_theme']}, {p['target_region']})")
    print(f"  Candidate {p['candidate_id']}")
    print(f"  Hamming label: {p['label']:.2f}")
    print(f"  Target text:    {p['target_text'][:120]}...")
    print(f"  Candidate text: {p['candidate_text'][:120]}...")


# === Save ===
os.makedirs(os.path.join(WORKDIR, 'cross_encoder'), exist_ok=True)
output_path = os.path.join(WORKDIR, 'cross_encoder/training_pairs.jsonl')
with open(output_path, 'w') as f:
    for p in training_pairs:
        f.write(json.dumps(p) + '\n')
print(f"\n✓ Saved to {output_path}")
print(f"✓ Cell 14 complete")

In [ ]:
# ============================================================
# CELL 15 — Train/val split with stratification by region
# ============================================================
# Stratify by target region so each split has same distribution.
# This is important because Middle East dominates the test set —
# we want the val set to look similar to inference distribution.

import json
import random
from collections import defaultdict

PAIRS_PATH = os.path.join(WORKDIR, 'cross_encoder/training_pairs.jsonl')
training_pairs = [json.loads(l) for l in open(PAIRS_PATH)]
print(f"Loaded {len(training_pairs)} training pairs")


# Group by target_id so all pairs with same target stay in same split.
# Otherwise the model might leak via memorizing target texts.
target_groups = defaultdict(list)
for p in training_pairs:
    target_groups[p['target_id']].append(p)
unique_targets = list(target_groups.keys())
print(f"Unique target resolutions: {len(unique_targets)}")


# Stratify by region
target_to_region = {p['target_id']: p['target_region'] for p in training_pairs}
target_to_region_collated = {tid: target_to_region[tid] for tid in unique_targets}

# Group targets by region, split each region 80/20
region_targets = defaultdict(list)
for tid, region in target_to_region_collated.items():
    region_targets[region].append(tid)

random.seed(42)
train_targets, val_targets = [], []
for region, targets in region_targets.items():
    random.shuffle(targets)
    n_val = max(1, int(len(targets) * 0.2))
    val_targets.extend(targets[:n_val])
    train_targets.extend(targets[n_val:])

print(f"\nSplit: {len(train_targets)} train targets, {len(val_targets)} val targets")

# Expand back to pairs
train_pairs = [p for tid in train_targets for p in target_groups[tid]]
val_pairs = [p for tid in val_targets for p in target_groups[tid]]
print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs:   {len(val_pairs)}")


# Verify region distribution matches across splits
print("\nRegion distribution check:")
print(f"{'Region':<20} {'Train %':>8} {'Val %':>8}")
print("-" * 40)
all_regions = set(target_to_region_collated.values())
for region in sorted(all_regions):
    train_n = sum(1 for p in train_pairs if p['target_region'] == region)
    val_n = sum(1 for p in val_pairs if p['target_region'] == region)
    train_pct = train_n / len(train_pairs) * 100 if train_pairs else 0
    val_pct = val_n / len(val_pairs) * 100 if val_pairs else 0
    print(f"  {region:<18} {train_pct:>7.1f}% {val_pct:>7.1f}%")


# Save splits
import json
train_path = os.path.join(WORKDIR, 'cross_encoder/train.jsonl')
val_path = os.path.join(WORKDIR, 'cross_encoder/val.jsonl')
with open(train_path, 'w') as f:
    for p in train_pairs:
        f.write(json.dumps(p) + '\n')
with open(val_path, 'w') as f:
    for p in val_pairs:
        f.write(json.dumps(p) + '\n')
print(f"\n✓ Saved {train_path}")
print(f"✓ Saved {val_path}")
print(f"✓ Cell 15 complete")

In [ ]:
# ============================================================
# CELL 16 — Fine-tune cross-encoder on coalition-similarity task
# ============================================================
# Architecture: cross-encoder/ms-marco-MiniLM-L-6-v2 (66M params).
# Pre-trained for query-passage relevance scoring on MS-MARCO; we
# fine-tune the regression head to predict P5 coalition Hamming
# similarity between two UNSC resolutions.
#
# Training notes:
# - WeightedRandomSampler balances perfect-match vs hard-case pairs.
# - MSE loss directly on the [0,1] coalition similarity target.
# - 3 epochs, batch 32, lr 2e-5 — standard for cross-encoder fine-tuning.
# - On free Colab T4 this takes ~30-45 min.

import json
import os
import time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load splits
train_pairs = [json.loads(l) for l in
               open(os.path.join(WORKDIR, 'cross_encoder/train.jsonl'))]
val_pairs = [json.loads(l) for l in
             open(os.path.join(WORKDIR, 'cross_encoder/val.jsonl'))]
print(f"\nTrain: {len(train_pairs)} pairs")
print(f"Val:   {len(val_pairs)} pairs")


# === Sampling weights (Fix A) ===
# Goal: each batch contains ~40% perfect, ~30% medium, ~30% hard.
# We compute weights per-pair so WeightedRandomSampler produces this mix.

def sample_weight(label):
    """Inverse-frequency-ish weight given the actual label distribution."""
    if label == 1.0:
        return 1.0          # 76% of data → weight 1
    elif label >= 0.8:
        return 4.0          # 12% of data → ~3x boost
    elif label >= 0.6:
        return 8.0          # 5% of data → ~5x boost
    elif label >= 0.4:
        return 15.0         # 2% of data → ~7x boost
    elif label >= 0.2:
        return 25.0         # 1% of data → ~10x boost
    else:
        return 20.0         # 3% of data → ~7x boost
    # Note: very low (0-0.19) gets slightly less weight than 0.2-0.39
    # because some are noise (procedural quirks); we don't want to
    # over-amplify them.


train_weights = [sample_weight(p['label']) for p in train_pairs]
print(f"\nSampling weight stats:")
print(f"  Min: {min(train_weights):.1f}, Max: {max(train_weights):.1f}")
print(f"  Effective batch composition:")
import bisect
weight_buckets = sorted(set(train_weights))
total_weight = sum(train_weights)
for w in weight_buckets:
    n = sum(1 for x in train_weights if x == w)
    contribution = (n * w) / total_weight * 100
    print(f"    weight={w:>4.1f}: {n:>5} pairs → "
          f"{contribution:>5.1f}% of expected batch")


# === Tokenizer + model ===
MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
print(f"\nLoading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


# === Dataset ===
class CoalitionPairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=256):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        # Cross-encoder input: [CLS] target [SEP] candidate [SEP]
        encoded = self.tokenizer(
            p['target_text'],
            p['candidate_text'],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label': torch.tensor(p['label'], dtype=torch.float32),
        }


train_dataset = CoalitionPairDataset(train_pairs, tokenizer)
val_dataset = CoalitionPairDataset(val_pairs, tokenizer)

train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_pairs),
    replacement=True
)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           sampler=train_sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")


# === Training loop ===
EPOCHS = 3
LR = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = torch.nn.MSELoss()

# Track metrics across epochs
history = {'epoch': [], 'train_loss': [], 'val_loss': [],
           'val_pearson': [], 'val_hard_neg_loss': []}


def evaluate(model, loader):
    model.eval()
    losses = []
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label = batch['label'].to(device)
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            pred = output.logits.squeeze(-1)
            loss = loss_fn(pred, label)
            losses.append(loss.item())
            preds.extend(pred.cpu().numpy())
            labels.extend(label.cpu().numpy())

    preds, labels = np.array(preds), np.array(labels)
    pearson = np.corrcoef(preds, labels)[0, 1] if len(preds) > 1 else 0
    # Hard-negative-specific loss (what we really care about)
    hard_mask = labels < 0.6
    hard_loss = ((preds[hard_mask] - labels[hard_mask]) ** 2).mean() if hard_mask.any() else 0
    return np.mean(losses), pearson, float(hard_loss)


# Initial eval (random model baseline)
print("\nInitial eval (untrained head)...")
val_loss, val_pearson, val_hard_loss = evaluate(model, val_loader)
print(f"  Val MSE: {val_loss:.4f}, Pearson: {val_pearson:.3f}, "
      f"Hard-case MSE: {val_hard_loss:.4f}")


# Train
print(f"\nTraining for {EPOCHS} epochs on {device}...")
print("=" * 60)
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        optimizer.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = output.logits.squeeze(-1)
        loss = loss_fn(pred, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_losses.append(loss.item())

        if batch_idx % 50 == 0:
            elapsed = time.time() - start
            print(f"  Epoch {epoch} batch {batch_idx}/{len(train_loader)}: "
                  f"loss={loss.item():.4f}, elapsed={elapsed:.0f}s")

    train_loss = np.mean(epoch_losses)
    val_loss, val_pearson, val_hard_loss = evaluate(model, val_loader)

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pearson'].append(val_pearson)
    history['val_hard_neg_loss'].append(val_hard_loss)

    print(f"\nEpoch {epoch} summary:")
    print(f"  Train MSE:     {train_loss:.4f}")
    print(f"  Val MSE:       {val_loss:.4f}")
    print(f"  Val Pearson:   {val_pearson:.3f}")
    print(f"  Hard-case MSE: {val_hard_loss:.4f}  (lower = learning hard cases)")
    print()


# === Save model ===
SAVE_PATH = os.path.join(WORKDIR, 'cross_encoder/model_v1')
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel saved to {SAVE_PATH}")


# === Save training history ===
with open(os.path.join(WORKDIR, 'cross_encoder/training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)


# === Final diagnostics ===
print("\n" + "=" * 60)
print("FINAL TRAINING SUMMARY")
print("=" * 60)
print(f"Total time: {(time.time()-start)/60:.1f} min")
print(f"\nMetrics across epochs:")
print(f"{'Epoch':<7} {'Train MSE':>10} {'Val MSE':>10} "
      f"{'Pearson':>10} {'Hard MSE':>10}")
for i in range(EPOCHS):
    print(f"{history['epoch'][i]:<7} "
          f"{history['train_loss'][i]:>10.4f} "
          f"{history['val_loss'][i]:>10.4f} "
          f"{history['val_pearson'][i]:>10.3f} "
          f"{history['val_hard_neg_loss'][i]:>10.4f}")

# Sanity: predictions on validation set
print("\n5 validation pairs with predictions vs actual:")
model.eval()
random.seed(42)
sample_pairs = random.sample(val_pairs, 5)
with torch.no_grad():
    for p in sample_pairs:
        encoded = tokenizer(p['target_text'], p['candidate_text'],
                             max_length=256, padding='max_length',
                             truncation=True, return_tensors='pt').to(device)
        pred = model(**encoded).logits.item()
        print(f"  Target {p['target_id']:<25} Cand {p['candidate_id']:<25} "
              f"actual={p['label']:.2f}, pred={pred:.2f}")

print("\n✓ Cell 16 complete")

In [ ]:
# ============================================================
# CELL 17 — Gate test for cross-encoder reranker
# ============================================================
# Pipeline: AAAI text scoring → top-50 candidates → cross-encoder
# rescores them → pick top-1 → check coalition match.
# We use raw logits for ranking (calibration to [0,1] doesn't matter
# for top-K selection). Compare against AAAI+recency baseline and
# all previous CSR versions.

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict
import numpy as np
import time

# Load fine-tuned model
SAVE_PATH = os.path.join(WORKDIR, 'cross_encoder/model_v1')
print(f"Loading fine-tuned model from {SAVE_PATH}...")
ce_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
ce_model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
ce_model = ce_model.to(device)
ce_model.eval()
print("✓ Model loaded")


def cross_encoder_score_batch(target_text, candidate_texts, batch_size=64,
                                max_length=256):
    """
    Score a batch of (target, candidate) pairs using the cross-encoder.
    Returns array of raw logit scores (higher = more coalition-similar).
    """
    scores = []
    with torch.no_grad():
        for i in range(0, len(candidate_texts), batch_size):
            batch = candidate_texts[i:i + batch_size]
            encoded = ce_tokenizer(
                [target_text] * len(batch),
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors='pt'
            ).to(device)
            output = ce_model(**encoded)
            scores.extend(output.logits.squeeze(-1).cpu().numpy().tolist())
    return scores


def csr_v4_retrieve(target, ckg_pool, recall_k=50, final_k=1):
    """
    Two-stage CSR-v4: AAAI text recall → cross-encoder rerank.
    Returns top-final_k candidates as dict objects.

    Note: this version doesn't use target_nation parameter because
    the cross-encoder operates purely on text similarity, not on
    nation-specific signal. We keep the signature consistent with
    earlier versions for ease of comparison.
    """
    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    if not eligible:
        return []

    # Stage 1: AAAI text scoring with recency tiebreaker
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = [c for _, c in text_scored[:recall_k]]

    # Stage 2: cross-encoder rerank
    target_text = build_target_text(target)
    if not target_text:
        return top_recall[:final_k]

    candidate_texts = [build_target_text(c) for c in top_recall]
    ce_scores = cross_encoder_score_batch(target_text, candidate_texts)

    reranked = sorted(zip(ce_scores, top_recall), key=lambda x: x[0],
                       reverse=True)
    return [c for _, c in reranked[:final_k]]


# === Build target text helper for test resolutions ===
def build_test_text(test_resolution):
    """Use context (full resolution text) for test resolutions since
    we have it; falls back to summary if context is empty."""
    text = test_resolution.get('context', '')[:1500]
    if not text:
        text = test_resolution.get('summary', '')[:1500]
    return text


# We need to override build_target_text for test resolutions because
# they have 'context' (full text) not 'description'. CKG candidates use
# 'description'. So at inference, target uses build_test_text() but
# candidates still use build_target_text().
#
# Let me wrap it cleanly:
def build_text_for_inference(item):
    """Universal text builder — works for test resolutions and CKG entries."""
    if 'context' in item and item['context']:
        return item['context'][:1500]
    if 'description' in item and item['description']:
        return item['description'][:1500]
    if 'summary' in item and item['summary']:
        return item['summary'][:1500]
    return ''


# Patch the retrieval to use the universal builder
def csr_v4_retrieve_fixed(target, ckg_pool, recall_k=50, final_k=1):
    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    if not eligible:
        return []

    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = [c for _, c in text_scored[:recall_k]]

    target_text = build_text_for_inference(target)
    if not target_text:
        return top_recall[:final_k]

    candidate_texts = [build_text_for_inference(c) for c in top_recall]
    ce_scores = cross_encoder_score_batch(target_text, candidate_texts)

    reranked = sorted(zip(ce_scores, top_recall), key=lambda x: x[0],
                       reverse=True)
    return [c for _, c in reranked[:final_k]]


# === Run the gate ===
print("\nRunning gate with cross-encoder reranker...")
print("=" * 60)
results_csr_v4 = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_csr_v4 = defaultdict(lambda: {'matches': 0, 'total': 0})

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 5 == 0 and i > 0:
        elapsed = time.time() - start
        eta = elapsed / i * (len(test_resolutions) - i)
        print(f"  [{i}/{len(test_resolutions)}] elapsed {elapsed:.0f}s, "
              f"ETA {eta:.0f}s")

    region = target['geopolitical_region']
    v4_top = csr_v4_retrieve_fixed(target, ckg_pool, recall_k=50, final_k=1)

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue
        # CE rerank doesn't depend on nation; same precedent for all 5 nations
        m = (v4_top and
             coalition_match(target['coalition'], v4_top[0]['coalition'],
                             exclude_nation=nation_code))
        for bucket in [results_csr_v4[nation_code], results_region_csr_v4[region]]:
            bucket['total'] += 1
            if m:
                bucket['matches'] += 1

print(f"\nCompleted in {time.time()-start:.0f}s")


# === Six-way comparison ===
print("\n" + "=" * 100)
print("SIX-WAY COMPARISON — CSR-v4 (cross-encoder) joins the lineup")
print("=" * 100)
print(f"{'Nation':<6} {'AAAI+rec':>10} {'CSR-v1':>9} {'CSR-v2':>9} "
      f"{'CSR-v3':>9} {'CSR-v4':>9} {'Oracle':>9} {'v4 capt':>9}")
print("-" * 100)
deltas_v4 = []
for nation in P5:
    a = results_aaai_recency[nation]
    v1 = results_csr[nation]
    v2 = results_csr_v2[nation]
    v3 = results_csr_v3[nation]
    v4 = results_csr_v4[nation]
    o = oracle_results[nation]
    if v4['total'] == 0:
        continue
    a_mr = a['matches']/a['total']*100
    v1_mr = v1['matches']/v1['total']*100
    v2_mr = v2['matches']/v2['total']*100
    v3_mr = v3['matches']/v3['total']*100
    v4_mr = v4['matches']/v4['total']*100
    o_mr = o['matches']/o['total']*100
    headroom = o_mr - a_mr
    captured = (v4_mr - a_mr) / headroom * 100 if headroom > 0 else 0
    deltas_v4.append(v4_mr - a_mr)
    print(f"{nation:<6} {a_mr:>9.1f}% {v1_mr:>8.1f}% {v2_mr:>8.1f}% "
          f"{v3_mr:>8.1f}% {v4_mr:>8.1f}% {o_mr:>8.1f}% {captured:>7.0f}%")
print("-" * 100)
print(f"Mean Δ v4 vs AAAI+rec: {np.mean(deltas_v4):+.1f}pp")


# === Per-region ===
print("\n" + "=" * 70)
print("PER-REGION (CSR-v4 vs AAAI+recency)")
print("=" * 70)
print(f"{'Region':<25} {'AAAI+rec':>10} {'CSR-v3':>10} {'CSR-v4':>10} "
      f"{'Δ v4':>10} {'n':>5}")
print("-" * 80)
for region in sorted(results_region_csr_v4.keys(),
                      key=lambda r: -results_region_csr_v4[r]['total']):
    ar = results_region_aaai_recency[region]
    v3 = results_region_csr_v3[region]
    v4 = results_region_csr_v4[region]
    if v4['total'] == 0:
        continue
    ar_mr = ar['matches']/ar['total']*100
    v3_mr = v3['matches']/v3['total']*100
    v4_mr = v4['matches']/v4['total']*100
    delta = v4_mr - ar_mr
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else ("·" if delta >= 0 else "✗")
    print(f"{region:<25} {ar_mr:>9.1f}% {v3_mr:>9.1f}% {v4_mr:>9.1f}% "
          f"{delta:>+8.1f}pp {v4['total']:>5}  {flag}")


# === Save ===
import json
with open(os.path.join(WORKDIR, 'gate_results/gate_v4_crossencoder.json'), 'w') as f:
    json.dump({
        'mean_delta_vs_aaai_recency': float(np.mean(deltas_v4)),
        'per_nation': {n: {
            'aaai_recency': results_aaai_recency[n]['matches']/results_aaai_recency[n]['total']*100,
            'csr_v4': results_csr_v4[n]['matches']/results_csr_v4[n]['total']*100,
            'oracle': oracle_results[n]['matches']/oracle_results[n]['total']*100,
        } for n in P5 if results_csr_v4[n]['total']},
        'per_region': {r: {
            'aaai_recency': results_region_aaai_recency[r]['matches']/results_region_aaai_recency[r]['total']*100,
            'csr_v4': results_region_csr_v4[r]['matches']/results_region_csr_v4[r]['total']*100,
        } for r in results_region_csr_v4},
    }, f, indent=2)
print("\n✓ Cell 17 complete")

In [ ]:
# ============================================================
# CELL 18 — Diagnose the cross-encoder's learned bias
# ============================================================
# For each test resolution:
#   1. Take AAAI's top-50 candidates
#   2. Score them with the cross-encoder
#   3. Split into two groups: unanimous-Y candidates vs split-coalition
#   4. Compare mean cross-encoder score per group
#
# If the model learned the right thing, split candidates should score
# HIGHER (we want it to surface contested precedents for contested targets).
# If the model learned the wrong thing, unanimous candidates score HIGHER.

import math
import numpy as np
import torch
from collections import defaultdict, Counter

print("Diagnosing cross-encoder's learned bias...")
print("=" * 60)


def is_unanimous_y(coalition):
    return all(coalition.get(n) == 'Y' for n in P5)


def coalition_entropy(coalition):
    votes = [coalition.get(n) for n in P5 if coalition.get(n) is not None]
    if not votes: return 0.0
    counts = Counter(votes)
    total = len(votes)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())


# For each test target, score top-50 with cross-encoder, then compare groups
unanimous_scores = []
split_scores = []
per_target_diff = []  # mean(unanimous_score) - mean(split_score) per target

ce_model.eval()
import time
start = time.time()

for i, target in enumerate(test_resolutions):
    if i % 10 == 0:
        print(f"  [{i}/{len(test_resolutions)}] elapsed {time.time()-start:.0f}s")

    eligible = [c for c in ckg_pool if c['date'] < target['date']]
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in text_scored[:50]]

    # Score all 50 with cross-encoder
    target_text = build_text_for_inference(target)
    cand_texts = [build_text_for_inference(c) for c in top50]
    if not target_text or not cand_texts:
        continue

    ce_scores = cross_encoder_score_batch(target_text, cand_texts)

    # Split
    unanimous_local = []
    split_local = []
    for c, s in zip(top50, ce_scores):
        if is_unanimous_y(c['coalition']):
            unanimous_local.append(s)
        else:
            split_local.append(s)
    unanimous_scores.extend(unanimous_local)
    split_scores.extend(split_local)
    if unanimous_local and split_local:
        per_target_diff.append(np.mean(unanimous_local) - np.mean(split_local))


print(f"\nCompleted in {time.time()-start:.0f}s")
print(f"\nGlobal stats across all top-50 candidates:")
print(f"  Unanimous-Y candidates: n={len(unanimous_scores)}")
print(f"    Mean CE score: {np.mean(unanimous_scores):.3f}")
print(f"    Median:        {np.median(unanimous_scores):.3f}")
print(f"  Split candidates:       n={len(split_scores)}")
print(f"    Mean CE score: {np.mean(split_scores):.3f}")
print(f"    Median:        {np.median(split_scores):.3f}")

mean_diff = np.mean(unanimous_scores) - np.mean(split_scores)
print(f"\n  Δ (unanimous - split): {mean_diff:+.3f}")
if mean_diff > 0.2:
    print("  ✗ DIAGNOSIS CONFIRMED: model prefers consensus → wrong learned signal")
elif mean_diff < -0.2:
    print("  ✓ Model prefers contested → correct learned signal")
else:
    print("  ≈ Model is roughly neutral on consensus vs contested")


# Per-target preference breakdown
diffs = np.array(per_target_diff)
print(f"\nPer-target preference (n={len(diffs)} targets):")
print(f"  Targets where unanimous candidates score HIGHER: "
      f"{(diffs > 0).sum()}/{len(diffs)} = {(diffs > 0).mean():.0%}")
print(f"  Targets where split candidates score HIGHER:     "
      f"{(diffs < 0).sum()}/{len(diffs)} = {(diffs < 0).mean():.0%}")


# === Spot check: 3 specific test cases ===
# For 3 contested test resolutions, show the top-5 reranked candidates
# and what the model selected.
print("\n" + "=" * 60)
print("3 TEST RESOLUTIONS: top-5 reranked candidates")
print("=" * 60)
import random
random.seed(42)
contested_targets = [t for t in test_resolutions
                      if any(v != 'Y' for v in t['coalition'].values()
                             if v is not None)]
sampled = random.sample(contested_targets, 3)

for target in sampled:
    print(f"\nTarget {target['resolution_id']} ({target['geopolitical_region']})")
    print(f"  Target coalition: {target['coalition']}")

    eligible = [c for c in ckg_pool if c['date'] < target['date']]
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in text_scored[:50]]

    target_text = build_text_for_inference(target)
    cand_texts = [build_text_for_inference(c) for c in top50]
    ce_scores = cross_encoder_score_batch(target_text, cand_texts)

    ranked = sorted(zip(ce_scores, top50), key=lambda x: x[0], reverse=True)[:5]
    print(f"  Top-5 reranked by cross-encoder:")
    for rank, (score, c) in enumerate(ranked, 1):
        ent = coalition_entropy(c['coalition'])
        unan = "UNANIMOUS-Y" if is_unanimous_y(c['coalition']) else f"split (entropy={ent:.2f})"
        print(f"    [{rank}] CE={score:>6.2f}  {c['resolution_id']:<20} "
              f"({unan}) coalition={c['coalition']}")

In [ ]:
# ============================================================
# CELL 19 — Generate contested-only training pairs
# ============================================================
# Filter training targets to resolutions whose own coalition is contested
# (at least one P5 voted not-Y). Pair each with top-50 AAAI candidates
# (not top-5) so we have more candidates per contested target.
#
# Hypothesis: training only on contested-as-target produces a model that
# learns to discriminate between matching and non-matching contested
# coalitions, instead of the consensus-prefer prior we got from full data.

import json
import os
import time
import numpy as np
from datetime import datetime
from collections import Counter
import math


def is_contested(coalition):
    """At least one P5 nation voted not-Y (and we have data for ≥3 nations)."""
    votes = [v for v in coalition.values() if v is not None]
    if len(votes) < 3:
        return False
    return any(v != 'Y' for v in votes)


# Filter pre-2013 targets to contested ones
all_pre2013_adopted = [r for r in ckg_pool
                        if r['is_adopted']
                        and r['date'] < datetime(2013, 1, 1)]
contested_targets = [r for r in all_pre2013_adopted if is_contested(r['coalition'])]
print(f"Pre-2013 adopted resolutions: {len(all_pre2013_adopted)}")
print(f"  → contested only: {len(contested_targets)} "
      f"({len(contested_targets)/len(all_pre2013_adopted):.0%})")

# Also include vetoed drafts as targets — they're always contested
contested_vetoes = [r for r in ckg_pool
                     if not r['is_adopted']
                     and r['date'] < datetime(2013, 1, 1)
                     and is_contested(r['coalition'])]
print(f"Pre-2013 veto events as additional targets: {len(contested_vetoes)}")

training_targets_v2 = contested_targets + contested_vetoes
print(f"Total training targets: {len(training_targets_v2)}")


def coalition_hamming_continuous(coalition_a, coalition_b):
    matches, total = 0, 0
    for nation in P5:
        a = coalition_a.get(nation)
        b = coalition_b.get(nation)
        if a is None or b is None:
            continue
        total += 1
        if a == b:
            matches += 1
    if total < 3:
        return None
    return matches / total


def build_target_text_v2(entry):
    parts = []
    if entry.get('description'):
        parts.append(entry['description'])
    return ' '.join(parts)[:1500]


# Generate pairs: top-15 AAAI candidates per contested target.
# Higher k than before (was 5) because we have fewer targets and want
# enough negative diversity per target for ranking learning.
training_pairs_v2 = []
n_skipped = 0

start = time.time()
for i, target in enumerate(training_targets_v2):
    if i % 100 == 0:
        elapsed = time.time() - start
        print(f"  [{i}/{len(training_targets_v2)}] elapsed {elapsed:.0f}s, "
              f"pairs so far: {len(training_pairs_v2)}")

    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target['resolution_id']]
    if not eligible:
        continue

    scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        scored.append((s + tie, c))
    scored.sort(key=lambda x: x[0], reverse=True)

    target_text = build_target_text_v2(target)
    if not target_text:
        continue

    for _, candidate in scored[:15]:
        candidate_text = build_target_text_v2(candidate)
        if not candidate_text:
            continue
        label = coalition_hamming_continuous(target['coalition'],
                                               candidate['coalition'])
        if label is None:
            n_skipped += 1
            continue
        training_pairs_v2.append({
            'target_id': target['resolution_id'],
            'target_date': target['date'].isoformat(),
            'target_text': target_text,
            'candidate_id': candidate['resolution_id'],
            'candidate_text': candidate_text,
            'label': label,
            'target_theme': target['theme'],
            'target_region': target['geopolitical_region'],
            'target_is_adopted': target['is_adopted'],
        })

print(f"\nCompleted in {time.time()-start:.0f}s")
print(f"Total pairs: {len(training_pairs_v2)}")
print(f"Skipped: {n_skipped}")


# === Label distribution ===
print("\n" + "=" * 60)
print("LABEL DISTRIBUTION (contested-only training)")
print("=" * 60)
labels = [p['label'] for p in training_pairs_v2]
print(f"Mean: {np.mean(labels):.3f}, Median: {np.median(labels):.3f}, "
      f"Std: {np.std(labels):.3f}")

buckets = Counter()
for l in labels:
    if l == 1.0: buckets['1.0 (perfect)'] += 1
    elif l >= 0.8: buckets['0.8-0.99'] += 1
    elif l >= 0.6: buckets['0.6-0.79'] += 1
    elif l >= 0.4: buckets['0.4-0.59'] += 1
    elif l >= 0.2: buckets['0.2-0.39'] += 1
    else: buckets['0.0-0.19'] += 1
for k in ['1.0 (perfect)', '0.8-0.99', '0.6-0.79', '0.4-0.59',
           '0.2-0.39', '0.0-0.19']:
    n = buckets[k]
    pct = n / len(labels) * 100 if labels else 0
    bar = '█' * int(pct / 2)
    print(f"  {k:<18} {n:>5} ({pct:>5.1f}%) {bar}")

# Compare to v1 distribution
print("\nComparison to v1 (full pre-2013 training):")
print(f"  v1 perfect-match rate: 76.1%")
v2_perfect = buckets['1.0 (perfect)'] / len(labels) * 100 if labels else 0
print(f"  v2 perfect-match rate: {v2_perfect:.1f}%")
print(f"  v2 hard cases (<0.6):  "
      f"{(buckets['0.4-0.59'] + buckets['0.2-0.39'] + buckets['0.0-0.19']) / len(labels) * 100:.1f}%")


# Save
import json
out_path = os.path.join(WORKDIR, 'cross_encoder/training_pairs_v2.jsonl')
with open(out_path, 'w') as f:
    for p in training_pairs_v2:
        f.write(json.dumps(p) + '\n')
print(f"\n✓ Saved to {out_path}")
print(f"✓ Cell 19 complete")

In [ ]:
# ============================================================
# CELL 20 — Train/val split for v2 (contested-only)
# ============================================================
# Same stratification logic as Cell 15. Use a separate val set so we
# can directly compare v2 model to v1 model.

import json
import random
from collections import defaultdict

PAIRS_PATH = os.path.join(WORKDIR, 'cross_encoder/training_pairs_v2.jsonl')
training_pairs_v2 = [json.loads(l) for l in open(PAIRS_PATH)]
print(f"Loaded {len(training_pairs_v2)} v2 training pairs")

target_groups = defaultdict(list)
for p in training_pairs_v2:
    target_groups[p['target_id']].append(p)
unique_targets = list(target_groups.keys())
print(f"Unique target resolutions: {len(unique_targets)}")

target_to_region = {p['target_id']: p['target_region']
                     for p in training_pairs_v2}
target_to_region_collated = {tid: target_to_region[tid] for tid in unique_targets}

region_targets = defaultdict(list)
for tid, region in target_to_region_collated.items():
    region_targets[region].append(tid)

random.seed(42)
train_targets, val_targets = [], []
for region, targets in region_targets.items():
    random.shuffle(targets)
    n_val = max(1, int(len(targets) * 0.2))
    val_targets.extend(targets[:n_val])
    train_targets.extend(targets[n_val:])

train_pairs = [p for tid in train_targets for p in target_groups[tid]]
val_pairs = [p for tid in val_targets for p in target_groups[tid]]
print(f"Train: {len(train_pairs)} pairs from {len(train_targets)} targets")
print(f"Val:   {len(val_pairs)} pairs from {len(val_targets)} targets")

# Save
with open(os.path.join(WORKDIR, 'cross_encoder/train_v2.jsonl'), 'w') as f:
    for p in train_pairs:
        f.write(json.dumps(p) + '\n')
with open(os.path.join(WORKDIR, 'cross_encoder/val_v2.jsonl'), 'w') as f:
    for p in val_pairs:
        f.write(json.dumps(p) + '\n')

# Region distribution check
print("\nRegion distribution check:")
print(f"{'Region':<20} {'Train %':>8} {'Val %':>8}")
print("-" * 40)
all_regions = set(target_to_region_collated.values())
for region in sorted(all_regions):
    train_n = sum(1 for p in train_pairs if p['target_region'] == region)
    val_n = sum(1 for p in val_pairs if p['target_region'] == region)
    train_pct = train_n / len(train_pairs) * 100 if train_pairs else 0
    val_pct = val_n / len(val_pairs) * 100 if val_pairs else 0
    print(f"  {region:<18} {train_pct:>7.1f}% {val_pct:>7.1f}%")

print("\n✓ Cell 20 complete")

In [ ]:
# ============================================================
# CELL 21 — Fine-tune cross-encoder v2 on contested-only data
# ============================================================
# Same architecture as v1 but two changes:
#   1. Sigmoid output + BCE-with-logits loss (calibrates outputs to [0,1])
#   2. Sampling weights recalibrated for the v2 label distribution
#
# Training data is now contested-target pairs, so the model learns
# discriminative coalition matching instead of consensus prior.

import json
import os
import time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load v2 splits
train_pairs = [json.loads(l) for l in
               open(os.path.join(WORKDIR, 'cross_encoder/train_v2.jsonl'))]
val_pairs = [json.loads(l) for l in
             open(os.path.join(WORKDIR, 'cross_encoder/val_v2.jsonl'))]
print(f"Train: {len(train_pairs)} pairs")
print(f"Val:   {len(val_pairs)} pairs")


# === Recalibrated sampling weights for v2 distribution ===
# Old distribution had 76% perfect. New is 25% perfect, 33% near-perfect (0.8-0.99),
# 17% (0.6-0.79), 7% (0.4-0.59), 6% (0.2-0.39), 12% (0.0-0.19).
# Goal: roughly balanced exposure across all bins.

def sample_weight_v2(label):
    if label == 1.0:
        return 1.0           # 25% of data → 25% effective batch
    elif label >= 0.8:
        return 0.8           # 33% of data → 26% effective batch
    elif label >= 0.6:
        return 1.5           # 17% of data → 25% effective batch
    elif label >= 0.4:
        return 3.5           # 7% of data → 24% effective batch
    elif label >= 0.2:
        return 4.0           # 6% of data → 24% effective batch
    else:
        return 2.0           # 12% of data → 24% effective batch
    # All bins now contribute roughly equally per batch.


train_weights = [sample_weight_v2(p['label']) for p in train_pairs]
print(f"\nSampling weight composition:")
weight_buckets = sorted(set(train_weights))
total_weight = sum(train_weights)
for w in weight_buckets:
    n = sum(1 for x in train_weights if x == w)
    contribution = (n * w) / total_weight * 100
    print(f"  weight={w:>4.1f}: {n:>5} pairs → {contribution:>5.1f}% of expected batch")


# === Load model ===
MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


# === Dataset ===
class CoalitionPairDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=256):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        encoded = self.tokenizer(
            p['target_text'], p['candidate_text'],
            max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label': torch.tensor(p['label'], dtype=torch.float32),
        }


train_dataset = CoalitionPairDataset(train_pairs, tokenizer)
val_dataset = CoalitionPairDataset(val_pairs, tokenizer)

train_sampler = WeightedRandomSampler(weights=train_weights,
                                        num_samples=len(train_pairs),
                                        replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           sampler=train_sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# === Training with BCE-with-logits + sigmoid output ===
EPOCHS = 4   # one extra epoch since dataset is smaller
LR = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
loss_fn = torch.nn.BCEWithLogitsLoss()  # expects raw logits + targets in [0,1]

history = {'epoch': [], 'train_loss': [], 'val_loss': [],
           'val_pearson': [], 'val_hard_loss': [], 'val_hard_acc': []}


def evaluate(model, loader):
    """Compute val loss, Pearson, and hard-case-specific metrics."""
    model.eval()
    losses = []
    raw_logits, sigmoid_preds, labels = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            label = batch['label'].to(device)
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = output.logits.squeeze(-1)
            loss = loss_fn(logits, label)
            losses.append(loss.item())
            raw_logits.extend(logits.cpu().numpy())
            sigmoid_preds.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(label.cpu().numpy())

    raw_logits = np.array(raw_logits)
    sigmoid_preds = np.array(sigmoid_preds)
    labels = np.array(labels)
    pearson = np.corrcoef(sigmoid_preds, labels)[0, 1] if len(labels) > 1 else 0

    # Hard-case metrics
    hard_mask = labels < 0.6
    if hard_mask.any():
        hard_loss = ((sigmoid_preds[hard_mask] - labels[hard_mask]) ** 2).mean()
        # "Hard accuracy": for hard cases (label<0.6), did we predict <0.6?
        hard_acc = (sigmoid_preds[hard_mask] < 0.6).mean()
    else:
        hard_loss, hard_acc = 0, 0
    return np.mean(losses), pearson, float(hard_loss), float(hard_acc)


# Initial eval
print("\nInitial eval (untrained head)...")
val_loss, pearson, hard_loss, hard_acc = evaluate(model, val_loader)
print(f"  BCE: {val_loss:.4f}, Pearson: {pearson:.3f}, "
      f"Hard MSE: {hard_loss:.4f}, Hard acc: {hard_acc:.0%}")

# Train
print(f"\nTraining for {EPOCHS} epochs on {device}...")
print("=" * 60)
start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        label = batch['label'].to(device)

        optimizer.zero_grad()
        output = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = output.logits.squeeze(-1)
        loss = loss_fn(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_losses.append(loss.item())

        if batch_idx % 30 == 0:
            print(f"  E{epoch} batch {batch_idx}/{len(train_loader)}: "
                  f"loss={loss.item():.4f}, elapsed={time.time()-start:.0f}s")

    train_loss = np.mean(epoch_losses)
    val_loss, pearson, hard_loss, hard_acc = evaluate(model, val_loader)
    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_pearson'].append(pearson)
    history['val_hard_loss'].append(hard_loss)
    history['val_hard_acc'].append(hard_acc)

    print(f"\nEpoch {epoch} summary:")
    print(f"  Train BCE:     {train_loss:.4f}")
    print(f"  Val BCE:       {val_loss:.4f}")
    print(f"  Val Pearson:   {pearson:.3f}")
    print(f"  Hard MSE:      {hard_loss:.4f}  (lower = better)")
    print(f"  Hard accuracy: {hard_acc:.0%}     (predicting <0.6 on hard cases)")
    print()


# Save model v2
SAVE_PATH = os.path.join(WORKDIR, 'cross_encoder/model_v2')
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"\nModel saved to {SAVE_PATH}")

with open(os.path.join(WORKDIR, 'cross_encoder/training_history_v2.json'), 'w') as f:
    json.dump(history, f, indent=2)


# Final summary
print("\n" + "=" * 60)
print("FINAL TRAINING SUMMARY (v2)")
print("=" * 60)
print(f"Total time: {(time.time()-start)/60:.1f} min")
print(f"\n{'Epoch':<7} {'Train BCE':>10} {'Val BCE':>10} "
      f"{'Pearson':>9} {'Hard MSE':>10} {'Hard Acc':>10}")
for i in range(EPOCHS):
    print(f"{history['epoch'][i]:<7} "
          f"{history['train_loss'][i]:>10.4f} "
          f"{history['val_loss'][i]:>10.4f} "
          f"{history['val_pearson'][i]:>9.3f} "
          f"{history['val_hard_loss'][i]:>10.4f} "
          f"{history['val_hard_acc'][i]:>9.0%}")


# Sanity check: 5 val predictions
print("\n5 val pairs (sigmoid predictions vs actual):")
import random
random.seed(42)
sample_pairs = random.sample(val_pairs, 5)
model.eval()
with torch.no_grad():
    for p in sample_pairs:
        encoded = tokenizer(p['target_text'], p['candidate_text'],
                             max_length=256, padding='max_length',
                             truncation=True, return_tensors='pt').to(device)
        logits = model(**encoded).logits.item()
        pred = 1 / (1 + np.exp(-logits))  # sigmoid
        print(f"  {p['target_id']:<22} {p['candidate_id']:<22} "
              f"actual={p['label']:.2f}  sigmoid={pred:.3f}  raw={logits:+.2f}")

print("\n✓ Cell 21 complete")

In [ ]:
# ============================================================
# CELL 22 — Gate test for cross-encoder v2 (contested-trained)
# ============================================================
# Same pipeline as Cell 17 but loads the v2 model. We rerank AAAI's
# top-50 candidates per target, take top-1, check coalition match.

import torch
import numpy as np
import time
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import defaultdict

# Load v2 model
SAVE_PATH = os.path.join(WORKDIR, 'cross_encoder/model_v2')
print(f"Loading v2 model from {SAVE_PATH}...")
ce_tokenizer_v2 = AutoTokenizer.from_pretrained(SAVE_PATH)
ce_model_v2 = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
ce_model_v2 = ce_model_v2.to(device)
ce_model_v2.eval()
print("✓ Model loaded")


def cross_encoder_score_batch_v2(target_text, candidate_texts,
                                   batch_size=64, max_length=256):
    """Score using v2 model. Returns sigmoid-calibrated probabilities."""
    scores = []
    with torch.no_grad():
        for i in range(0, len(candidate_texts), batch_size):
            batch = candidate_texts[i:i + batch_size]
            encoded = ce_tokenizer_v2(
                [target_text] * len(batch), batch,
                padding=True, truncation=True,
                max_length=max_length, return_tensors='pt'
            ).to(device)
            output = ce_model_v2(**encoded)
            probs = torch.sigmoid(output.logits.squeeze(-1))
            scores.extend(probs.cpu().numpy().tolist())
    return scores


def build_text_for_inference(item):
    """Universal text builder — works for test resolutions and CKG entries."""
    if 'context' in item and item['context']:
        return item['context'][:1500]
    if 'description' in item and item['description']:
        return item['description'][:1500]
    if 'summary' in item and item['summary']:
        return item['summary'][:1500]
    return ''


def csr_v5_retrieve(target, ckg_pool, recall_k=50, final_k=1):
    """CSR-v5: AAAI text recall → cross-encoder v2 rerank."""
    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    if not eligible:
        return []

    # Stage 1: AAAI text scoring
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = [c for _, c in text_scored[:recall_k]]

    # Stage 2: cross-encoder v2 rerank
    target_text = build_text_for_inference(target)
    if not target_text:
        return top_recall[:final_k]
    candidate_texts = [build_text_for_inference(c) for c in top_recall]
    ce_scores = cross_encoder_score_batch_v2(target_text, candidate_texts)
    reranked = sorted(zip(ce_scores, top_recall), key=lambda x: x[0],
                       reverse=True)
    return [c for _, c in reranked[:final_k]]


# === Run gate ===
print("\nRunning gate with cross-encoder v2 (contested-trained)...")
print("=" * 60)
results_csr_v5 = defaultdict(lambda: {'matches': 0, 'total': 0})
results_region_csr_v5 = defaultdict(lambda: {'matches': 0, 'total': 0})

# Track which (target, csr_top) pairs disagreed with which retrievers
all_v5_disagreements = []

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 10 == 0:
        elapsed = time.time() - start
        eta = elapsed / max(i, 1) * (len(test_resolutions) - i) if i > 0 else 0
        print(f"  [{i}/{len(test_resolutions)}] elapsed {elapsed:.0f}s, "
              f"ETA {eta:.0f}s")

    region = target['geopolitical_region']
    v5_top = csr_v5_retrieve(target, ckg_pool, recall_k=50, final_k=1)

    for nation_code in P5:
        if target['coalition'].get(nation_code) is None:
            continue
        m = (v5_top and
             coalition_match(target['coalition'], v5_top[0]['coalition'],
                             exclude_nation=nation_code))
        for bucket in [results_csr_v5[nation_code], results_region_csr_v5[region]]:
            bucket['total'] += 1
            if m:
                bucket['matches'] += 1

print(f"\nCompleted in {time.time()-start:.0f}s")


# === Seven-way comparison ===
print("\n" + "=" * 110)
print("SEVEN-WAY COMPARISON — CSR-v5 (cross-encoder, contested-trained)")
print("=" * 110)
print(f"{'Nation':<6} {'AAAI+rec':>10} {'v1':>7} {'v2':>7} {'v3':>7} "
      f"{'v4':>7} {'v5':>7} {'Oracle':>8} {'v5 capt':>9}")
print("-" * 110)
deltas_v5 = []
for nation in P5:
    a = results_aaai_recency[nation]
    v1 = results_csr[nation]
    v2 = results_csr_v2[nation]
    v3 = results_csr_v3[nation]
    v4 = results_csr_v4[nation]
    v5 = results_csr_v5[nation]
    o = oracle_results[nation]
    if v5['total'] == 0:
        continue
    a_mr = a['matches']/a['total']*100
    v1_mr = v1['matches']/v1['total']*100
    v2_mr = v2['matches']/v2['total']*100
    v3_mr = v3['matches']/v3['total']*100
    v4_mr = v4['matches']/v4['total']*100
    v5_mr = v5['matches']/v5['total']*100
    o_mr = o['matches']/o['total']*100
    headroom = o_mr - a_mr
    captured = (v5_mr - a_mr) / headroom * 100 if headroom > 0 else 0
    deltas_v5.append(v5_mr - a_mr)
    print(f"{nation:<6} {a_mr:>9.1f}% {v1_mr:>6.1f}% {v2_mr:>6.1f}% "
          f"{v3_mr:>6.1f}% {v4_mr:>6.1f}% {v5_mr:>6.1f}% "
          f"{o_mr:>7.1f}% {captured:>7.0f}%")
print("-" * 110)
print(f"Mean Δ v5 vs AAAI+rec: {np.mean(deltas_v5):+.1f}pp")


# === Per-region with v3 and v5 side by side ===
print("\n" + "=" * 80)
print("PER-REGION (AAAI vs v3 entropy vs v5 cross-encoder)")
print("=" * 80)
print(f"{'Region':<25} {'AAAI+rec':>10} {'v3':>8} {'v5':>8} "
      f"{'Δ v5':>10} {'n':>5}")
print("-" * 80)
for region in sorted(results_region_csr_v5.keys(),
                      key=lambda r: -results_region_csr_v5[r]['total']):
    ar = results_region_aaai_recency[region]
    v3 = results_region_csr_v3[region]
    v5 = results_region_csr_v5[region]
    if v5['total'] == 0:
        continue
    ar_mr = ar['matches']/ar['total']*100
    v3_mr = v3['matches']/v3['total']*100
    v5_mr = v5['matches']/v5['total']*100
    delta = v5_mr - ar_mr
    flag = "✓" if delta >= 10 else "≈" if delta >= 5 else ("·" if delta >= 0 else "✗")
    print(f"{region:<25} {ar_mr:>9.1f}% {v3_mr:>7.1f}% {v5_mr:>7.1f}% "
          f"{delta:>+8.1f}pp {v5['total']:>5}  {flag}")


# === Diagnostic: re-check consensus-bias on v2 model ===
print("\n" + "=" * 60)
print("CONSENSUS-BIAS DIAGNOSTIC (v2 model)")
print("=" * 60)
unanimous_scores_v2 = []
split_scores_v2 = []
per_target_diff_v2 = []

for target in test_resolutions:
    eligible = [c for c in ckg_pool if c['date'] < target['date']]
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top50 = [c for _, c in text_scored[:50]]
    target_text = build_text_for_inference(target)
    cand_texts = [build_text_for_inference(c) for c in top50]
    if not target_text or not cand_texts:
        continue
    ce_scores = cross_encoder_score_batch_v2(target_text, cand_texts)
    unan_local, split_local = [], []
    for c, s in zip(top50, ce_scores):
        if all(c['coalition'].get(n) == 'Y' for n in P5):
            unan_local.append(s)
        else:
            split_local.append(s)
    unanimous_scores_v2.extend(unan_local)
    split_scores_v2.extend(split_local)
    if unan_local and split_local:
        per_target_diff_v2.append(np.mean(unan_local) - np.mean(split_local))

print(f"Mean CE-v2 score on UNANIMOUS-Y candidates: "
      f"{np.mean(unanimous_scores_v2):.3f}")
print(f"Mean CE-v2 score on SPLIT candidates:        "
      f"{np.mean(split_scores_v2):.3f}")
print(f"Δ (unanimous - split): "
      f"{np.mean(unanimous_scores_v2) - np.mean(split_scores_v2):+.3f}")
diffs = np.array(per_target_diff_v2)
print(f"\nTargets where unanimous scored higher: "
      f"{(diffs > 0).sum()}/{len(diffs)} = {(diffs > 0).mean():.0%}")
print(f"  (v1 cross-encoder: 79% — biased toward unanimous)")


# Save
with open(os.path.join(WORKDIR, 'gate_results/gate_v5_crossencoder_v2.json'), 'w') as f:
    json.dump({
        'mean_delta_vs_aaai_recency': float(np.mean(deltas_v5)),
        'per_nation': {n: {
            'aaai_recency': results_aaai_recency[n]['matches']/results_aaai_recency[n]['total']*100,
            'csr_v5': results_csr_v5[n]['matches']/results_csr_v5[n]['total']*100,
            'oracle': oracle_results[n]['matches']/oracle_results[n]['total']*100,
        } for n in P5 if results_csr_v5[n]['total']},
        'per_region': {r: {
            'aaai_recency': results_region_aaai_recency[r]['matches']/results_region_aaai_recency[r]['total']*100,
            'csr_v5': results_region_csr_v5[r]['matches']/results_region_csr_v5[r]['total']*100,
        } for r in results_region_csr_v5},
        'unanimous_higher_pct_v1': 0.79,
        'unanimous_higher_pct_v2': float((diffs > 0).mean()) if len(diffs) > 0 else 0,
    }, f, indent=2)
print("\n✓ Cell 22 complete")

In [ ]:
# ============================================================
# CELL 22.5 — Save canonical state for LLM experiments
# ============================================================
# Persists the artifacts we'll need for all downstream LLM work
# in a single pickle so future sessions can skip cells 1-22.

import pickle
import os
import json
from datetime import datetime

CANONICAL_PATH = os.path.join(WORKDIR, 'canonical_state_v1.pkl')

canonical_state = {
    'ckg_pool': ckg_pool,
    'test_resolutions': test_resolutions,
    'profile_cutoff_date': datetime(2013, 1, 1),
    'P5': P5,
    'NATION_AAAI_TO_CODE': NATION_AAAI_TO_CODE,
    'NATION_CODE_TO_AAAI': NATION_CODE_TO_AAAI,
    'REGION_HIERARCHY': REGION_HIERARCHY,
    'THEME_TO_REGION': THEME_TO_REGION,
    'COUNTRY_PATTERNS': COUNTRY_PATTERNS,
    'created_at': datetime.now().isoformat(),
    'version': 'v1_csr_v3_canonical',
}

with open(CANONICAL_PATH, 'wb') as f:
    pickle.dump(canonical_state, f)

print(f"Saved canonical state: {CANONICAL_PATH}")
print(f"Size: {os.path.getsize(CANONICAL_PATH) / 1e6:.1f} MB")
print(f"\nContents:")
print(f"  ckg_pool:          {len(ckg_pool)} entries")
print(f"  test_resolutions:  {len(test_resolutions)} entries")
print(f"  profile_cutoff:    {datetime(2013, 1, 1).date()}")
print(f"  P5 nations:        {P5}")
print(f"  Regions:           {len(REGION_HIERARCHY)}")
print(f"  Themes mapped:     {len(THEME_TO_REGION)}")
print(f"  Country patterns:  {len(COUNTRY_PATTERNS)}")
print(f"\n✓ Future sessions can run Cell 0 (below) instead of Cells 1-22")

In [ ]:
# ============================================================
# CELL 0 — Bootstrap from canonical state (future sessions only)
# ============================================================
# Run this ONCE at the start of any new Colab session to restore
# everything we need for LLM experiments. Replaces Cells 1-22.

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import pickle
import json
import numpy as np
import pandas as pd
import math
from datetime import datetime, timedelta
from collections import defaultdict, Counter

WORKDIR = '/content/drive/MyDrive/CoalitionRAG'
os.chdir(WORKDIR)

# Load canonical state
CANONICAL_PATH = os.path.join(WORKDIR, 'canonical_state_v1.pkl')
with open(CANONICAL_PATH, 'rb') as f:
    state = pickle.load(f)

ckg_pool = state['ckg_pool']
test_resolutions = state['test_resolutions']
P5 = state['P5']
NATION_AAAI_TO_CODE = state['NATION_AAAI_TO_CODE']
NATION_CODE_TO_AAAI = state['NATION_CODE_TO_AAAI']
REGION_HIERARCHY = state['REGION_HIERARCHY']
THEME_TO_REGION = state['THEME_TO_REGION']
COUNTRY_PATTERNS = state['COUNTRY_PATTERNS']
PROFILE_CUTOFF = state['profile_cutoff_date']

print(f"Loaded canonical state from {state['created_at']}")
print(f"  ckg_pool:         {len(ckg_pool)} entries")
print(f"  test_resolutions: {len(test_resolutions)} entries")


# Rebuild profile (fast, ~2 sec, doesn't need to be saved)
class CoalitionThemeProfile:
    def __init__(self, ckg_resolutions, cutoff_date, min_samples=10):
        self.min_samples = min_samples
        self.theme_profile = defaultdict(lambda: defaultdict(Counter))
        self.region_profile = defaultdict(lambda: defaultdict(Counter))
        self.nation_profile = defaultdict(Counter)
        for r in ckg_resolutions:
            if r['date'] >= cutoff_date:
                continue
            for nation_code, vote in r['coalition'].items():
                if vote is None:
                    continue
                self.theme_profile[r['theme']][nation_code][vote] += 1
                self.region_profile[r['geopolitical_region']][nation_code][vote] += 1
                self.nation_profile[nation_code][vote] += 1
        self.profiles = {}
        for theme, nations in self.theme_profile.items():
            for nation, counter in nations.items():
                total = sum(counter.values())
                if total == 0:
                    continue
                self.profiles[(theme, nation)] = {
                    v: counter.get(v, 0) / total for v in ['Y', 'N', 'A']
                }

    @staticmethod
    def _to_dist(counter):
        total = sum(counter.values())
        if total == 0:
            return None
        return {v: counter.get(v, 0) / total for v in ['Y', 'N', 'A']}

    def expected_distribution(self, theme, region, nation):
        c = self.theme_profile[theme][nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        c = self.region_profile[region][nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        c = self.nation_profile[nation]
        if sum(c.values()) >= self.min_samples:
            return self._to_dist(c)
        return {'Y': 1/3, 'N': 1/3, 'A': 1/3}


profile = CoalitionThemeProfile(ckg_pool, PROFILE_CUTOFF)


# Rebuild canonical CSR-v3 functions (fast, no GPU needed)

def normalize_region(region):
    if not region:
        return 'Other'
    return REGION_HIERARCHY.get(region, 'Other')


def regions_match(region_a, region_b):
    if not region_a or not region_b:
        return False
    if region_a == region_b:
        return True
    return normalize_region(region_a) == normalize_region(region_b)


def aaai_text_score(target_keywords, target_region, target_nations, candidate):
    score = 0.0
    if regions_match(target_region, candidate.get('geopolitical_region', '')):
        score += 2.0
    cand_nations = set(candidate.get('target_nations', []))
    score += 1.0 * len(set(target_nations) & cand_nations)
    cand_keywords = set(candidate.get('keywords', []))
    score += 0.1 * len(set(target_keywords) & cand_keywords)
    return score


def coalition_entropy(coalition):
    votes = [coalition.get(n) for n in P5 if coalition.get(n) is not None]
    if not votes:
        return 0.0
    counts = Counter(votes)
    total = len(votes)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())


def csr_v3_retrieve(target, ckg_pool, recall_k=50, final_k=1,
                     w_entropy=1.0, w_recency=0.3, w_text=0.2):
    """
    Canonical CSR-v3: AAAI text recall → entropy-based rerank.
    This is the chosen retrieval method for LLM experiments.
    """
    eligible = [c for c in ckg_pool
                if c['date'] < target['date']
                and c['resolution_id'] != target.get('resolution_id')]
    if not eligible:
        return []
    text_scored = []
    for c in eligible:
        s = aaai_text_score(target['keywords'],
                             target['geopolitical_region'],
                             target['target_nations'], c)
        tie = (target['date'] - c['date']).days * -1e-6
        text_scored.append((s + tie, s, c))
    text_scored.sort(key=lambda x: x[0], reverse=True)
    top_recall = text_scored[:recall_k]
    if not top_recall:
        return []

    MAX_ENTROPY = math.log2(3)
    reranked = []
    for combined_score, raw_text_score, candidate in top_recall:
        ent = coalition_entropy(candidate['coalition']) / MAX_ENTROPY
        days = (target['date'] - candidate['date']).days
        rec = 0.5 ** (days / 1825.0)
        text_norm = min(raw_text_score / 5.0, 1.0)
        rerank_score = w_entropy * ent + w_recency * rec + w_text * text_norm
        reranked.append((rerank_score, candidate))
    reranked.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in reranked[:final_k]]


# Quick verification
print(f"\nProfile rebuilt:")
print(f"  (theme, nation) pairs: {len(profile.profiles)}")
print(f"\nCSR-v3 smoke test on test_resolutions[0]:")
test_target = test_resolutions[0]
result = csr_v3_retrieve(test_target, ckg_pool, recall_k=50, final_k=1)
if result:
    print(f"  Target: {test_target['resolution_id']} ({test_target['geopolitical_region']})")
    print(f"  Retrieved: {result[0]['resolution_id']} (theme: {result[0]['theme']})")
    print(f"  Coalition match: {result[0]['coalition']}")
print("\n✓ Cell 0 complete — ready for LLM experiments")

In [ ]:
# ============================================================
# CELL 22.6 — Inspect AAAI pipeline structure for integration
# ============================================================
# Read the key AAAI files so Cell 23 can integrate cleanly.
# This is a one-time inspection cell, not part of the pipeline.

import os

REPO_DIR = os.path.join(WORKDIR, 'Nation-Level_Bias')

files_to_inspect = [
    'Framework/RAG_RFLX_GRAPH.py',
    '[submission]Debiasing_Framework.py',
    '[submission]Vote_simulation.py',
    'util/templates/VoteTemplates.py',
    'util/Helper/Helper.py',
    'util/CustomClass/CustomClass.py',
    'util/CustomClass/CustomGraphState.py',
    'util/DatasetLoader/DatasetLoader.py',
]

for rel_path in files_to_inspect:
    full_path = os.path.join(REPO_DIR, rel_path)
    if not os.path.exists(full_path):
        print(f"✗ MISSING: {rel_path}")
        continue
    size = os.path.getsize(full_path)
    with open(full_path) as f:
        content = f.read()
    print(f"\n{'='*70}")
    print(f"FILE: {rel_path} ({size:,} bytes, {len(content.splitlines())} lines)")
    print(f"{'='*70}")
    print(content)

In [ ]:
# ============================================================
# CELL 23.5 — Rebuild AAAI integration state + save canonical v2
# ============================================================
# Run this AFTER Cell 0 (canonical state load).
# It rebuilds AAAI candidate dicts with coalition cross-reference,
# saves everything to canonical_state_v2.pkl, and prints verification.
# Future sessions just need Cell 0 + Cell 23.5-load (defined at end).

import os
import sys
import json
import re
import math
import pickle
from datetime import datetime
from collections import Counter

REPO_DIR = os.path.join(WORKDIR, 'Nation-Level_Bias')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# === Load AAAI datasets ===
ADOPTED_PATH = os.path.join(REPO_DIR, 'Dataset/adopted_resolutions_dataset.json')
NONADOPTED_PATH = os.path.join(REPO_DIR, 'Dataset/not_adopted_resolutions_dataset.json')

with open(ADOPTED_PATH) as f:
    AAAI_ADOPTED_DATASET = json.load(f)
with open(NONADOPTED_PATH) as f:
    AAAI_NONADOPTED_DATASET = json.load(f)
print(f"AAAI ADOPTED:     {len(AAAI_ADOPTED_DATASET)}")
print(f"AAAI NON_ADOPTED: {len(AAAI_NONADOPTED_DATASET)}")


# === Helper functions ===
def aaai_vote_to_coalition(vote_dict):
    """Convert AAAI vote dict to {CHN: 'Y', ...} format."""
    coalition = {}
    for code in P5:
        aaai_name = NATION_CODE_TO_AAAI[code]
        if aaai_name in vote_dict.get('favour', []):
            coalition[code] = 'Y'
        elif aaai_name in vote_dict.get('against', []):
            coalition[code] = 'N'
        elif aaai_name in vote_dict.get('abstention', []):
            coalition[code] = 'A'
        else:
            coalition[code] = None
    return coalition


def to_list_safe(x):
    """Coerce string/list/None to list."""
    if x is None:
        return []
    if isinstance(x, list):
        return [str(item).strip() for item in x if item]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if ',' in s:
            return [item.strip() for item in s.split(',') if item.strip()]
        return [s]
    return [str(x)]


def extract_resolution_number(res_no_str):
    """Extract bare resolution number from any AAAI format variant."""
    s = str(res_no_str).strip()
    if s.isdigit():
        return s
    m = re.match(r'S/RES/(\d+)', s)
    if m:
        return m.group(1)
    return None


# Build CKG index keyed by bare resolution number
ckg_by_number = {}
for r in ckg_pool:
    rid = r['resolution_id']
    m = re.match(r'S/RES/(\d+)\(\d+\)', rid)
    if m:
        ckg_by_number[m.group(1)] = r
    m = re.match(r'S/(\d{4})/(\d+)', rid)
    if m:
        ckg_by_number[rid] = r
print(f"CKG index built: {len(ckg_by_number)} keys")


def lookup_adopted_coalition(aaai_entry):
    """Lookup adopted entry's coalition via CKG cross-reference."""
    bare_number = extract_resolution_number(aaai_entry['res_no'])
    if bare_number is None:
        return None
    if bare_number in ckg_by_number:
        return dict(ckg_by_number[bare_number]['coalition'])
    return None


def build_aaai_candidate(entry, is_adopted):
    """Convert AAAI entry to candidate dict with recovered coalition."""
    if is_adopted:
        coalition = lookup_adopted_coalition(entry)
        if coalition is None:
            coalition = {n: None for n in P5}
    else:
        coalition = aaai_vote_to_coalition(entry['vote'])
    return {
        'resolution_id': entry['res_no'],
        'date': datetime.fromisoformat(entry['date']),
        'description': entry.get('summary', '') or entry.get('context', '')[:1500],
        'theme': entry.get('geopolitical_region', 'Other'),
        'geopolitical_region': entry.get('geopolitical_region', 'Other'),
        'target_nations': to_list_safe(entry.get('target_nations_list')),
        'keywords': to_list_safe(entry.get('keyword')),
        'coalition': coalition,
        'is_adopted': is_adopted,
        '_aaai_entry': entry,
    }


# Build candidates
AAAI_ADOPTED_CANDIDATES = [build_aaai_candidate(e, True) for e in AAAI_ADOPTED_DATASET]
AAAI_NONADOPTED_CANDIDATES = [build_aaai_candidate(e, False) for e in AAAI_NONADOPTED_DATASET]

n_recovered = sum(1 for c in AAAI_ADOPTED_CANDIDATES
                   if any(v is not None for v in c['coalition'].values()))
print(f"\nAdopted coalition recovery: {n_recovered}/{len(AAAI_ADOPTED_CANDIDATES)} "
      f"({n_recovered/len(AAAI_ADOPTED_CANDIDATES):.0%})")


# === CSR retriever class ===
class CSRv3Retriever:
    """Drop-in replacement for AAAI's KeywordRetriever using CSR-v3 entropy rerank."""

    def __init__(self, adopted_candidates, nonadopted_candidates,
                 profile, recall_k=50,
                 w_entropy=1.0, w_recency=0.3, w_text=0.2):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates
        self.profile = profile
        self.recall_k = recall_k
        self.w_entropy = w_entropy
        self.w_recency = w_recency
        self.w_text = w_text
        self.MAX_ENTROPY = math.log2(3)

    def _retrieve_pool(self, target, pool, k):
        target_date = target['date']
        target_id = target.get('res_no')
        eligible = [c for c in pool
                    if c['date'] < target_date
                    and c['resolution_id'] != target_id]
        if not eligible:
            return []
        text_scored = []
        for c in eligible:
            s = aaai_text_score(target.get('keyword', []),
                                 target.get('geopolitical_region', ''),
                                 target.get('target_nations_list', []),
                                 c)
            tie = (target_date - c['date']).days * -1e-6
            text_scored.append((s + tie, s, c))
        text_scored.sort(key=lambda x: x[0], reverse=True)
        top_recall = text_scored[:self.recall_k]

        reranked = []
        for combined_score, raw_text_score, candidate in top_recall:
            ent = coalition_entropy(candidate['coalition']) / self.MAX_ENTROPY
            days = (target_date - candidate['date']).days
            rec = 0.5 ** (days / 1825.0)
            text_norm = min(raw_text_score / 5.0, 1.0)
            rerank_score = (self.w_entropy * ent
                            + self.w_recency * rec
                            + self.w_text * text_norm)
            reranked.append((rerank_score, candidate))
        reranked.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in reranked[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        target = {
            'date': datetime.fromisoformat(input_dict['date']),
            'res_no': input_dict.get('res_no'),
            'keyword': to_list_safe(input_dict.get('keyword')),
            'geopolitical_region': input_dict.get('geopolitical_region', ''),
            'target_nations_list': to_list_safe(
                input_dict.get('target_nations_list')),
        }
        adopted_picks = self._retrieve_pool(target, self.adopted, k)
        nonadopted_picks = self._retrieve_pool(target, self.nonadopted, k)

        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]

        return to_aaai_format(adopted_picks), to_aaai_format(nonadopted_picks)


csr_retriever = CSRv3Retriever(
    AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES, profile
)
print("\n✓ CSRv3Retriever instantiated")


# === Save to canonical state v2 ===
# We save the candidates (which contain the cross-referenced coalitions)
# and the retriever's hyperparameters so future sessions can rebuild
# the retriever quickly. We don't pickle CSRv3Retriever directly because
# it references the candidates by Python identity.

CANONICAL_V2_PATH = os.path.join(WORKDIR, 'canonical_state_v2.pkl')
canonical_v2 = {
    'AAAI_ADOPTED_CANDIDATES': AAAI_ADOPTED_CANDIDATES,
    'AAAI_NONADOPTED_CANDIDATES': AAAI_NONADOPTED_CANDIDATES,
    'AAAI_ADOPTED_DATASET': AAAI_ADOPTED_DATASET,
    'AAAI_NONADOPTED_DATASET': AAAI_NONADOPTED_DATASET,
    'ckg_by_number': ckg_by_number,
    'csr_hyperparams': {
        'recall_k': 50,
        'w_entropy': 1.0,
        'w_recency': 0.3,
        'w_text': 0.2,
    },
    'created_at': datetime.now().isoformat(),
    'version': 'v2_csr_with_aaai_integration',
}
with open(CANONICAL_V2_PATH, 'wb') as f:
    pickle.dump(canonical_v2, f)
print(f"\nSaved canonical state v2: {CANONICAL_V2_PATH}")
print(f"Size: {os.path.getsize(CANONICAL_V2_PATH) / 1e6:.1f} MB")
print(f"\n✓ Cell 23.5 complete")
print(f"\nNext sessions: run Cell 0 then Cell 0b (defined below) to skip rebuild.")

In [ ]:
# ============================================================
# CELL 0b — Load AAAI integration state (future sessions)
# ============================================================
# Run AFTER Cell 0. Loads AAAI candidates and CSR retriever instantly.

import pickle
import math
import sys
import os
import re
from datetime import datetime
from collections import Counter

REPO_DIR = os.path.join(WORKDIR, 'Nation-Level_Bias')
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CANONICAL_V2_PATH = os.path.join(WORKDIR, 'canonical_state_v2.pkl')
with open(CANONICAL_V2_PATH, 'rb') as f:
    state_v2 = pickle.load(f)

AAAI_ADOPTED_CANDIDATES = state_v2['AAAI_ADOPTED_CANDIDATES']
AAAI_NONADOPTED_CANDIDATES = state_v2['AAAI_NONADOPTED_CANDIDATES']
AAAI_ADOPTED_DATASET = state_v2['AAAI_ADOPTED_DATASET']
AAAI_NONADOPTED_DATASET = state_v2['AAAI_NONADOPTED_DATASET']
ckg_by_number = state_v2['ckg_by_number']
csr_hyperparams = state_v2['csr_hyperparams']

print(f"Loaded AAAI integration state from {state_v2['created_at']}")
print(f"  ADOPTED candidates:     {len(AAAI_ADOPTED_CANDIDATES)}")
print(f"  NON_ADOPTED candidates: {len(AAAI_NONADOPTED_CANDIDATES)}")


# Re-define the helper functions (they're not pickled because they reference
# global state like ckg_by_number)
def aaai_vote_to_coalition(vote_dict):
    coalition = {}
    for code in P5:
        aaai_name = NATION_CODE_TO_AAAI[code]
        if aaai_name in vote_dict.get('favour', []):
            coalition[code] = 'Y'
        elif aaai_name in vote_dict.get('against', []):
            coalition[code] = 'N'
        elif aaai_name in vote_dict.get('abstention', []):
            coalition[code] = 'A'
        else:
            coalition[code] = None
    return coalition


def to_list_safe(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(item).strip() for item in x if item]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if ',' in s:
            return [item.strip() for item in s.split(',') if item.strip()]
        return [s]
    return [str(x)]


def extract_resolution_number(res_no_str):
    s = str(res_no_str).strip()
    if s.isdigit():
        return s
    m = re.match(r'S/RES/(\d+)', s)
    if m:
        return m.group(1)
    return None


def lookup_adopted_coalition(aaai_entry):
    bare_number = extract_resolution_number(aaai_entry['res_no'])
    if bare_number is None:
        return None
    if bare_number in ckg_by_number:
        return dict(ckg_by_number[bare_number]['coalition'])
    return None


# Re-define CSR retriever class
class CSRv3Retriever:
    def __init__(self, adopted_candidates, nonadopted_candidates,
                 profile, recall_k=50,
                 w_entropy=1.0, w_recency=0.3, w_text=0.2):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates
        self.profile = profile
        self.recall_k = recall_k
        self.w_entropy = w_entropy
        self.w_recency = w_recency
        self.w_text = w_text
        self.MAX_ENTROPY = math.log2(3)

    def _retrieve_pool(self, target, pool, k):
        target_date = target['date']
        target_id = target.get('res_no')
        eligible = [c for c in pool
                    if c['date'] < target_date
                    and c['resolution_id'] != target_id]
        if not eligible:
            return []
        text_scored = []
        for c in eligible:
            s = aaai_text_score(target.get('keyword', []),
                                 target.get('geopolitical_region', ''),
                                 target.get('target_nations_list', []),
                                 c)
            tie = (target_date - c['date']).days * -1e-6
            text_scored.append((s + tie, s, c))
        text_scored.sort(key=lambda x: x[0], reverse=True)
        top_recall = text_scored[:self.recall_k]
        reranked = []
        for combined_score, raw_text_score, candidate in top_recall:
            ent = coalition_entropy(candidate['coalition']) / self.MAX_ENTROPY
            days = (target_date - candidate['date']).days
            rec = 0.5 ** (days / 1825.0)
            text_norm = min(raw_text_score / 5.0, 1.0)
            rerank_score = (self.w_entropy * ent
                            + self.w_recency * rec
                            + self.w_text * text_norm)
            reranked.append((rerank_score, candidate))
        reranked.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in reranked[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        target = {
            'date': datetime.fromisoformat(input_dict['date']),
            'res_no': input_dict.get('res_no'),
            'keyword': to_list_safe(input_dict.get('keyword')),
            'geopolitical_region': input_dict.get('geopolitical_region', ''),
            'target_nations_list': to_list_safe(
                input_dict.get('target_nations_list')),
        }
        adopted_picks = self._retrieve_pool(target, self.adopted, k)
        nonadopted_picks = self._retrieve_pool(target, self.nonadopted, k)
        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]
        return to_aaai_format(adopted_picks), to_aaai_format(nonadopted_picks)


csr_retriever = CSRv3Retriever(
    AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES, profile,
    **csr_hyperparams
)
print("✓ CSRv3Retriever rebuilt and ready")
print("✓ Cell 0b complete")

In [ ]:
# ============================================================
# CELL 23b — End-to-end smoke test with mock LLM
# ============================================================
# Re-applies the CSR monkey-patch (kernel-restart-safe), builds
# pipeline with mock LLM, runs one test resolution end-to-end.
# Verifies entire AAAI pipeline plumbing without API spend.

import os
import sys
import time
from langchain_core.runnables import Runnable
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver

# AAAI uses relative imports — must be in repo dir
original_cwd = os.getcwd()
os.chdir(REPO_DIR)

# Force reimport of AAAI module to pick up any changes since last session
import importlib
import Framework.RAG_RFLX_GRAPH as rag_module
importlib.reload(rag_module)
from Framework.RAG_RFLX_GRAPH import RAGRflxGraph
from util.Helper.Helper import (
    input_dict_generator, get_vote_result,
    formatter_adopted_docs, formatter_not_adopted_docs
)
from util.CustomClass.CustomGraphState import CustomGraphState


# === Define the CSR retriever method ===
def csr_retrieve_document(self, state):
    """CSR-v3 replacement for AAAI's retrieve_document.

    Uses the global `csr_retriever` (instantiated in Cell 0b) to perform
    two-stage CSR retrieval, then formats results in the structure AAAI's
    downstream nodes expect.
    """
    error_flag = False
    country = state["country"]

    input_dict = {
        "summary": state["question_summary"],
        "res_no": state["res_id"],
        "date": state["question_date"],
        "geopolitical_region": state["question_geographic_keywords"],
        "target_nations_list": state["question_target_nation_keywords"],
        "keyword": state["question_keywords"],
    }

    k = state["k"]
    adopted_retrieved_docs, not_adopted_retrieved_docs = csr_retriever.retrieve_by_keyword(
        input_dict, k, threshold=3
    )

    retrieved_docs_dict = {}
    if adopted_retrieved_docs:
        for (key, res_content) in adopted_retrieved_docs:
            retrieved_docs_dict[key] = {
                "date": res_content["date"],
                "adopted": True,
            }
    if not_adopted_retrieved_docs:
        for (key, res_content) in not_adopted_retrieved_docs:
            retrieved_docs_dict[key] = {
                "date": res_content["date"],
                "adopted": False,
            }

    retrieved_docs_dict = dict(sorted(
        retrieved_docs_dict.items(),
        key=lambda item: item[1]["date"]
    ))
    retrieved_docs_order = list(retrieved_docs_dict.keys())

    if self.Verbal:
        print("## CSR-v3 RETRIEVER // sorted document dict :", retrieved_docs_dict)

    retrieved_docs_dictionary = {}
    if adopted_retrieved_docs:
        adopted_res_no_list, adopted_docs_formatted, adopted_speeches_formatted, adopted_summaries = formatter_adopted_docs(
            adopted_retrieved_docs, country
        )
        for res_key, doc, speech, summary in zip(
            adopted_res_no_list, adopted_docs_formatted,
            adopted_speeches_formatted, adopted_summaries
        ):
            retrieved_docs_dictionary[res_key] = {
                "adopted": True,
                "context": doc,
                "speech": speech,
                "summary": summary,
            }
    if not_adopted_retrieved_docs:
        not_adopted_res_no_list, not_adopted_docs_formatted, vote_list, not_adopted_speeches_formatted, not_adopted_summaries = formatter_not_adopted_docs(
            not_adopted_retrieved_docs, country
        )
        for res_key, doc, speech, vote, summary in zip(
            not_adopted_res_no_list, not_adopted_docs_formatted,
            not_adopted_speeches_formatted, vote_list, not_adopted_summaries
        ):
            retrieved_docs_dictionary[res_key] = {
                "adopted": False,
                "context": doc,
                "speech": speech,
                "vote": vote,
                "summary": summary,
            }

    retrieved_docs_dictionary_sorted = {
        key: retrieved_docs_dictionary[key]
        for key in retrieved_docs_order
        if key in retrieved_docs_dictionary
    }

    return CustomGraphState(
        retrieved_docs_dictionary_sorted=retrieved_docs_dictionary_sorted,
        retrieved_doc_no=len(retrieved_docs_order),
        retrieved_docs_order=retrieved_docs_order,
        curr_idx_of_execution=0,
        error_flag=error_flag,
    )


# Apply the monkey-patch
RAGRflxGraph.retrieve_document = csr_retrieve_document
print(f"✓ Monkey-patch applied. retrieve_document is now: "
      f"{RAGRflxGraph.retrieve_document.__name__}")


# === Mock LLM (Runnable-compliant) ===
class MockLLM(Runnable):
    """LangChain-compatible mock LLM for pipeline plumbing tests."""

    def __init__(self):
        self.call_count = 0
        self.call_log = []

    def invoke(self, prompt_input, config=None, **kwargs):
        self.call_count += 1
        log_entry = {
            'call': self.call_count,
            'input_type': type(prompt_input).__name__,
        }
        try:
            log_entry['input_preview'] = str(prompt_input)[:120]
        except Exception:
            log_entry['input_preview'] = '(unprintable)'
        self.call_log.append(log_entry)

        return AIMessage(content=(
            '{"vote": "abstention", "rationale": '
            '"Mock rationale for pipeline plumbing verification."}'
        ))


# === Build pipeline ===
mock_llm = MockLLM()
workflow_builder = RAGRflxGraph(mock_llm, debug_flag=False, Verbal=True)
workflow = workflow_builder.building_workflow()
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)
print("✓ Pipeline compiled with MockLLM\n")


# === Run smoke test ===
print("=" * 70)
print("SMOKE TEST — full pipeline on test entry [15] (S/2017/970)")
print("=" * 70)

test_entry = AAAI_NONADOPTED_DATASET[15]
persona = "Russian Federation"

input_dict = input_dict_generator(test_entry, persona, k=1)
true_vote = get_vote_result(test_entry['vote'], persona)
print(f"\nTarget: {test_entry['res_no']} ({test_entry['date']})")
print(f"Region: {test_entry['geopolitical_region']}")
print(f"Persona: {persona}")
print(f"True vote: {true_vote}\n")

config = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": f"smoke_{persona}_{test_entry['res_no']}_{int(time.time())}"}
)

start_time = time.time()
try:
    for event in app.stream(input_dict, config=config):
        pass
    elapsed = time.time() - start_time
    final_state = app.get_state(config).values

    print("\n" + "=" * 60)
    print(f"✓ Pipeline completed in {elapsed:.1f}s")
    print("=" * 60)
    print(f"  Mock LLM call count:        {mock_llm.call_count}")
    print(f"  Final answer vote:          {final_state['answer'].vote}")
    print(f"  Final rationale:            {final_state['answer'].rationale[:80]}")
    print(f"  Error flag:                 {final_state['error_flag']}")
    print(f"  Retrieved docs:             {list(final_state['retrieved_docs_dictionary_sorted'].keys())}")
    print(f"\n  Mock LLM call sequence:")
    for log in mock_llm.call_log:
        prev = log.get('input_preview', '').replace('\n', ' ')[:80]
        print(f"    [{log['call']}] {log['input_type']}: {prev}...")
except Exception as e:
    elapsed = time.time() - start_time
    print(f"\n✗ Pipeline failed after {elapsed:.1f}s")
    print(f"  Error type: {type(e).__name__}")
    print(f"  Error msg:  {e}")
    import traceback
    traceback.print_exc()

os.chdir(original_cwd)
print(f"\n✓ Cell 23b complete")

In [ ]:
# Ensure rank_bm25 is installed
!pip install -q rank_bm25

In [ ]:
import os
import re
from rank_bm25 import BM25Okapi
from datetime import datetime
from collections import Counter



def build_bm25_text(aaai_entry):
    """Concatenate summary + action_item + keyword for BM25 indexing.
    Avoids raw context's length bias while using AAAI's curated signal."""
    parts = []
    s = aaai_entry.get('summary', '')
    a = aaai_entry.get('action_item', '')
    k = aaai_entry.get('keyword', '')
    if s and isinstance(s, str):
        parts.append(s)
    if a and isinstance(a, str):
        parts.append(a)
    if k and isinstance(k, str):
        parts.append(k)
    return ' '.join(parts).strip()


def simple_tokenize(text):
    """Lowercase, alphanumeric tokens of length >= 2.
    Standard for BM25 — matches rank_bm25 examples."""
    return re.findall(r'[a-z0-9]+', text.lower())


class BM25Retriever:
    """Standard sparse-retrieval baseline using BM25Okapi.

    Indexes adopted and non-adopted pools separately (matching AAAI's
    pool-separated retrieval). Returns top-k from each in the same
    format as CSRv3Retriever for clean comparison.
    """

    def __init__(self, adopted_candidates, nonadopted_candidates):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates

        # Build per-pool corpora and BM25 indices
        self.adopted_texts = [build_bm25_text(c['_aaai_entry']) for c in adopted_candidates]
        self.nonadopted_texts = [build_bm25_text(c['_aaai_entry']) for c in nonadopted_candidates]

        adopted_tokens = [simple_tokenize(t) for t in self.adopted_texts]
        nonadopted_tokens = [simple_tokenize(t) for t in self.nonadopted_texts]

        # rank_bm25 needs at least one non-empty doc
        self.bm25_adopted = BM25Okapi(adopted_tokens)
        self.bm25_nonadopted = BM25Okapi(nonadopted_tokens)

    def _retrieve_pool(self, target_text, target_date, target_id,
                       pool, bm25, k):
        """Score pool with BM25, filter by chronology, return top-k."""
        target_tokens = simple_tokenize(target_text)
        if not target_tokens:
            return []

        scores = bm25.get_scores(target_tokens)

        # Filter eligible (chronology + self-exclusion)
        scored = []
        for i, c in enumerate(pool):
            if c['date'] >= target_date:
                continue
            if c['resolution_id'] == target_id:
                continue
            # Recency tiebreaker on top of BM25 score
            tie = (target_date - c['date']).days * -1e-9
            scored.append((scores[i] + tie, c))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        """AAAI-compatible signature. Builds query text from target's
        summary + action_item + keyword, scores both pools, returns top-k."""

        # Construct target's BM25 query text the same way we built corpus
        # so query and document distributions match
        target_text = build_bm25_text(input_dict)
        target_date = datetime.fromisoformat(input_dict['date'])
        target_id = input_dict.get('res_no')

        adopted_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            self.adopted, self.bm25_adopted, k
        )
        nonadopted_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            self.nonadopted, self.bm25_nonadopted, k
        )

        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]

        return to_aaai_format(adopted_picks), to_aaai_format(nonadopted_picks)


# === Build BM25 retriever ===
print("Building BM25 indices...")
bm25_retriever = BM25Retriever(
    AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES
)
print(f"✓ BM25 indexed {len(AAAI_ADOPTED_CANDIDATES)} adopted "
      f"+ {len(AAAI_NONADOPTED_CANDIDATES)} non-adopted candidates")


# === Smoke test on test entry [15] (same target as CSR smoke test) ===
test_entry = AAAI_NONADOPTED_DATASET[15]
print(f"\n{'=' * 70}")
print(f"BM25 SMOKE TEST — {test_entry['res_no']} ({test_entry['date']})")
print(f"Region: {test_entry['geopolitical_region']}")
print(f"Target's actual coalition: {aaai_vote_to_coalition(test_entry['vote'])}")
print('=' * 70)

input_dict = {
    'summary': test_entry['summary'],
    'action_item': test_entry['action_item'],
    'keyword': test_entry['keyword'],
    'res_no': test_entry['res_no'],
    'date': test_entry['date'],
    'geopolitical_region': test_entry['geopolitical_region'],
    'target_nations_list': test_entry['target_nations_list'],
}

adopted, nonadopted = bm25_retriever.retrieve_by_keyword(input_dict, k=1)

print(f"\nBM25 adopted retrieval ({len(adopted)}):")
for res_id, content in adopted:
    coalition = lookup_adopted_coalition(content['doc']) or {n: None for n in P5}
    print(f"  {res_id} ({content['date']})  Coalition: {coalition}")

print(f"\nBM25 non-adopted retrieval ({len(nonadopted)}):")
for res_id, content in nonadopted:
    coalition = aaai_vote_to_coalition(content['doc']['vote'])
    print(f"  {res_id} ({content['date']})  Coalition: {coalition}")


# === Quick comparison: BM25 vs CSR-v3 on same target ===
csr_adopted, csr_nonadopted = csr_retriever.retrieve_by_keyword(input_dict, k=1)
print(f"\n{'=' * 70}")
print(f"BM25 vs CSR-v3 on this target")
print('=' * 70)
print(f"  BM25     adopted: {[r[0] for r in adopted]}")
print(f"  CSR-v3   adopted: {[r[0] for r in csr_adopted]}")
print(f"  BM25     non-adopted: {[r[0] for r in nonadopted]}")
print(f"  CSR-v3   non-adopted: {[r[0] for r in csr_nonadopted]}")

print("\n✓ Cell 24 complete")

In [ ]:
# ============================================================
# CELL 25 — Dense retriever (BGE-small) baseline
# ============================================================
# Uses BAAI/bge-small-en-v1.5 to embed candidate texts and target
# queries. Scores by cosine similarity. Same pool-separated structure
# and output format as BM25Retriever and CSRv3Retriever.
#
# Embeddings are cached on disk (Drive) so we only build them once.

import os
import pickle
import torch
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer

DENSE_MODEL_NAME = 'BAAI/bge-small-en-v1.5'
EMBEDDINGS_CACHE = os.path.join(WORKDIR, 'dense_embeddings_v1.pkl')


def load_or_build_embeddings():
    """Load cached embeddings or compute fresh. Cached on Drive so
    future sessions don't repeat the ~30s embedding work."""
    if os.path.exists(EMBEDDINGS_CACHE):
        print(f"Loading cached embeddings from {EMBEDDINGS_CACHE}")
        with open(EMBEDDINGS_CACHE, 'rb') as f:
            cache = pickle.load(f)
        print(f"  Adopted:     {cache['adopted_emb'].shape}")
        print(f"  Non-adopted: {cache['nonadopted_emb'].shape}")
        return cache

    print(f"Building dense embeddings with {DENSE_MODEL_NAME}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")
    model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

    # Same text source as BM25 — summary + action_item + keyword
    adopted_texts = [build_bm25_text(c['_aaai_entry']) for c in AAAI_ADOPTED_CANDIDATES]
    nonadopted_texts = [build_bm25_text(c['_aaai_entry']) for c in AAAI_NONADOPTED_CANDIDATES]

    # BGE recommends "Represent this sentence for searching relevant passages: "
    # prefix on QUERIES (not documents). We won't add it to the corpus,
    # only to query texts at inference time.
    print(f"  Encoding {len(adopted_texts)} adopted...")
    adopted_emb = model.encode(adopted_texts, show_progress_bar=False,
                                normalize_embeddings=True,
                                batch_size=32)
    print(f"  Encoding {len(nonadopted_texts)} non-adopted...")
    nonadopted_emb = model.encode(nonadopted_texts, show_progress_bar=False,
                                   normalize_embeddings=True,
                                   batch_size=32)

    cache = {
        'adopted_emb': adopted_emb,
        'nonadopted_emb': nonadopted_emb,
        'model_name': DENSE_MODEL_NAME,
        'created_at': datetime.now().isoformat(),
    }
    with open(EMBEDDINGS_CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f"  Cached to {EMBEDDINGS_CACHE}")
    return cache


# Load model and embeddings
embeddings_cache = load_or_build_embeddings()
DENSE_MODEL = SentenceTransformer(DENSE_MODEL_NAME,
                                    device='cuda' if torch.cuda.is_available() else 'cpu')


# BGE query prefix — recommended by the model authors
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


class DenseRetriever:
    """Dense retrieval baseline using BGE-small embeddings + cosine similarity."""

    def __init__(self, adopted_candidates, nonadopted_candidates,
                 adopted_emb, nonadopted_emb, model):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates
        self.adopted_emb = adopted_emb       # (n_adopted, dim), normalized
        self.nonadopted_emb = nonadopted_emb # (n_nonadopted, dim), normalized
        self.model = model

    def _retrieve_pool(self, target_text, target_date, target_id,
                       pool, pool_emb, k):
        """Encode target, score all docs in pool by cosine, filter, return top-k."""
        if not target_text.strip():
            return []

        # Embed target with BGE query prefix
        query_text = BGE_QUERY_PREFIX + target_text
        query_emb = self.model.encode([query_text],
                                       normalize_embeddings=True,
                                       show_progress_bar=False)[0]

        # Cosine similarity = dot product (since normalized)
        scores = pool_emb @ query_emb  # (n,)

        # Filter eligible
        scored = []
        for i, c in enumerate(pool):
            if c['date'] >= target_date:
                continue
            if c['resolution_id'] == target_id:
                continue
            tie = (target_date - c['date']).days * -1e-9
            scored.append((float(scores[i]) + tie, c))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        """AAAI-compatible signature."""
        target_text = build_bm25_text(input_dict)  # same text builder as BM25
        target_date = datetime.fromisoformat(input_dict['date'])
        target_id = input_dict.get('res_no')

        adopted_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            self.adopted, self.adopted_emb, k
        )
        nonadopted_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            self.nonadopted, self.nonadopted_emb, k
        )

        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]

        return to_aaai_format(adopted_picks), to_aaai_format(nonadopted_picks)


# Build dense retriever
dense_retriever = DenseRetriever(
    AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES,
    embeddings_cache['adopted_emb'],
    embeddings_cache['nonadopted_emb'],
    DENSE_MODEL,
)
print(f"\n✓ Dense retriever ready ({DENSE_MODEL_NAME})")


# === Smoke test on test entry [15] ===
test_entry = AAAI_NONADOPTED_DATASET[15]
print(f"\n{'=' * 70}")
print(f"DENSE RETRIEVER SMOKE TEST — {test_entry['res_no']} ({test_entry['date']})")
print(f"Region: {test_entry['geopolitical_region']}")
print(f"Target's actual coalition: {aaai_vote_to_coalition(test_entry['vote'])}")
print('=' * 70)

input_dict = {
    'summary': test_entry['summary'],
    'action_item': test_entry['action_item'],
    'keyword': test_entry['keyword'],
    'res_no': test_entry['res_no'],
    'date': test_entry['date'],
    'geopolitical_region': test_entry['geopolitical_region'],
    'target_nations_list': test_entry['target_nations_list'],
}

dense_adopted, dense_nonadopted = dense_retriever.retrieve_by_keyword(input_dict, k=1)

print(f"\nDense adopted retrieval ({len(dense_adopted)}):")
for res_id, content in dense_adopted:
    coalition = lookup_adopted_coalition(content['doc']) or {n: None for n in P5}
    print(f"  {res_id} ({content['date']})  Coalition: {coalition}")

print(f"\nDense non-adopted retrieval ({len(dense_nonadopted)}):")
for res_id, content in dense_nonadopted:
    coalition = aaai_vote_to_coalition(content['doc']['vote'])
    print(f"  {res_id} ({content['date']})  Coalition: {coalition}")


# === Three-way comparison ===
bm25_adopted, bm25_nonadopted = bm25_retriever.retrieve_by_keyword(input_dict, k=1)
csr_adopted, csr_nonadopted = csr_retriever.retrieve_by_keyword(input_dict, k=1)

print(f"\n{'=' * 70}")
print(f"THREE-WAY COMPARISON on {test_entry['res_no']}")
print('=' * 70)
print(f"{'Method':<12} {'Adopted':<15} {'Non-adopted':<20}")
print('-' * 50)
print(f"{'BM25':<12} {bm25_adopted[0][0]:<15} {bm25_nonadopted[0][0] if bm25_nonadopted else '-':<20}")
print(f"{'Dense':<12} {dense_adopted[0][0]:<15} {dense_nonadopted[0][0] if dense_nonadopted else '-':<20}")
print(f"{'CSR-v3':<12} {csr_adopted[0][0]:<15} {csr_nonadopted[0][0] if csr_nonadopted else '-':<20}")

print("\n✓ Cell 25 complete")

In [ ]:
# ============================================================
# CELL 26 — Four-way retrieval comparison: AAAI vs BM25 vs Dense vs CSR
# ============================================================
# Runs all four retrievers on 66 test resolutions × 5 P5 nations,
# computes coalition match rate at k=1 per nation and per region.
# Output: the main retrieval-quality table for the paper.

import time
import json
import os
import numpy as np
from collections import defaultdict
from datetime import datetime


# === AAAI text retriever wrapper (for fair comparison) ===
# We re-implement AAAI's text scoring as a retriever class using the same
# code path as in our diagnostic gate cells. This is the "AAAI baseline."

class AAAITextRetriever:
    """AAAI's region+nation+keyword text scorer with recency tiebreaking."""

    def __init__(self, adopted_candidates, nonadopted_candidates):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates

    def _retrieve_pool(self, target, pool, k):
        target_date = datetime.fromisoformat(target['date'])
        target_id = target.get('res_no')
        eligible = [c for c in pool
                    if c['date'] < target_date
                    and c['resolution_id'] != target_id]
        scored = []
        for c in eligible:
            s = aaai_text_score(
                to_list_safe(target.get('keyword', [])),
                target.get('geopolitical_region', ''),
                to_list_safe(target.get('target_nations_list', [])),
                c
            )
            tie = (target_date - c['date']).days * -1e-6
            scored.append((s + tie, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        adopted_picks = self._retrieve_pool(input_dict, self.adopted, k)
        nonadopted_picks = self._retrieve_pool(input_dict, self.nonadopted, k)

        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]

        return to_aaai_format(adopted_picks), to_aaai_format(nonadopted_picks)


aaai_retriever = AAAITextRetriever(AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES)
print("✓ AAAI text retriever instantiated for comparison")


# === Coalition match rate function ===
def coalition_match(target_coalition, retrieved_coalition,
                    exclude_nation=None, threshold=0.75):
    """≥threshold fraction of comparable P5 votes match."""
    matches, total = 0, 0
    for nation in P5:
        if nation == exclude_nation:
            continue
        tv = target_coalition.get(nation)
        rv = retrieved_coalition.get(nation)
        if tv is None or rv is None:
            continue
        if tv == rv:
            matches += 1
        total += 1
    return total > 0 and matches / total >= threshold


def get_retrieved_coalition(retrieved_doc, is_adopted):
    """Extract coalition from a retrieved doc, handling both pool types."""
    if is_adopted:
        coalition = lookup_adopted_coalition(retrieved_doc)
        if coalition is None:
            coalition = {n: None for n in P5}
        return coalition
    else:
        return aaai_vote_to_coalition(retrieved_doc['vote'])


# === Run all four retrievers on all 66 test resolutions × 5 P5 nations ===
retrievers = {
    'AAAI':    aaai_retriever,
    'BM25':    bm25_retriever,
    'Dense':   dense_retriever,
    'CSR-v3':  csr_retriever,
}

# Per-nation, per-method counters (combining adopted + non-adopted top-1 results)
# We score: did either retrieved precedent (adopted top-1 or non-adopted top-1)
# coalition-match the target?
results = {m: defaultdict(lambda: {'matches': 0, 'total': 0}) for m in retrievers}
results_region = {m: defaultdict(lambda: {'matches': 0, 'total': 0}) for m in retrievers}
results_overall = {m: {'matches': 0, 'total': 0} for m in retrievers}

# Also save first 5 disagreement examples per method-pair for paper analysis
disagreements_to_save = []

print(f"\nRunning four-way comparison on {len(test_resolutions)} test resolutions...")
start = time.time()

for i, target in enumerate(test_resolutions):
    if i % 10 == 0 and i > 0:
        elapsed = time.time() - start
        eta = elapsed / i * (len(test_resolutions) - i)
        print(f"  [{i}/{len(test_resolutions)}] {elapsed:.0f}s elapsed, ETA {eta:.0f}s")

    region = target['geopolitical_region']
    target_coalition = target['coalition']

    # Build input_dict (matches what AAAI pipeline produces)
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary':              aaai_entry['summary'],
        'action_item':          aaai_entry['action_item'],
        'keyword':              aaai_entry['keyword'],
        'res_no':               aaai_entry['res_no'],
        'date':                 aaai_entry['date'],
        'geopolitical_region':  aaai_entry['geopolitical_region'],
        'target_nations_list':  aaai_entry['target_nations_list'],
    }

    for method, retriever in retrievers.items():
        adopted_results, nonadopted_results = retriever.retrieve_by_keyword(
            input_dict, k=1
        )

        # We score against the BEST of the two retrieved coalitions:
        # if EITHER pool returns a matching precedent, we count it as success.
        # This mirrors what the LLM actually sees (both pools' picks are
        # provided as context).
        retrieved_coalitions = []
        if adopted_results:
            retrieved_coalitions.append(
                get_retrieved_coalition(adopted_results[0][1]['doc'], is_adopted=True)
            )
        if nonadopted_results:
            retrieved_coalitions.append(
                get_retrieved_coalition(nonadopted_results[0][1]['doc'], is_adopted=False)
            )

        for nation_code in P5:
            if target_coalition.get(nation_code) is None:
                continue
            # Either retrieved coalition matches?
            best_match = any(
                coalition_match(target_coalition, rc, exclude_nation=nation_code)
                for rc in retrieved_coalitions
            )
            results[method][nation_code]['total'] += 1
            results_region[method][region]['total'] += 1
            results_overall[method]['total'] += 1
            if best_match:
                results[method][nation_code]['matches'] += 1
                results_region[method][region]['matches'] += 1
                results_overall[method]['matches'] += 1

elapsed = time.time() - start
print(f"\nCompleted in {elapsed:.0f}s")


# === Print results table: per-nation ===
print(f"\n{'=' * 80}")
print("RETRIEVAL QUALITY: COALITION MATCH RATE AT k=1, PER NATION")
print('=' * 80)
print(f"{'Nation':<6} {'AAAI':>10} {'BM25':>10} {'Dense':>10} {'CSR-v3':>10} {'Best':>8}")
print('-' * 80)
for nation in P5:
    n = results['AAAI'][nation]['total']
    rates = {}
    for method in retrievers:
        r = results[method][nation]
        rates[method] = r['matches'] / r['total'] * 100 if r['total'] else 0
    best = max(rates, key=rates.get)
    print(f"{nation:<6} {rates['AAAI']:>9.1f}% {rates['BM25']:>9.1f}% "
          f"{rates['Dense']:>9.1f}% {rates['CSR-v3']:>9.1f}% {best:>8}")

# Overall mean
print('-' * 80)
mean_rates = {}
for method in retrievers:
    r = results_overall[method]
    mean_rates[method] = r['matches'] / r['total'] * 100 if r['total'] else 0
print(f"{'Overall':<6} {mean_rates['AAAI']:>9.1f}% {mean_rates['BM25']:>9.1f}% "
      f"{mean_rates['Dense']:>9.1f}% {mean_rates['CSR-v3']:>9.1f}%")


# === Per-region ===
print(f"\n{'=' * 90}")
print("PER-REGION COALITION MATCH RATE")
print('=' * 90)
print(f"{'Region':<25} {'n':>5} {'AAAI':>10} {'BM25':>10} {'Dense':>10} {'CSR-v3':>10}")
print('-' * 90)
all_regions = sorted(results_region['AAAI'].keys(),
                      key=lambda r: -results_region['AAAI'][r]['total'])
for region in all_regions:
    n = results_region['AAAI'][region]['total']
    if n == 0:
        continue
    rates = {}
    for method in retrievers:
        r = results_region[method][region]
        rates[method] = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{region:<25} {n:>5} {rates['AAAI']:>9.1f}% {rates['BM25']:>9.1f}% "
          f"{rates['Dense']:>9.1f}% {rates['CSR-v3']:>9.1f}%")


# === Save results to disk ===
os.makedirs(os.path.join(WORKDIR, 'paper_results'), exist_ok=True)
output_path = os.path.join(WORKDIR, 'paper_results/retrieval_quality_v1.json')
with open(output_path, 'w') as f:
    json.dump({
        'methods': list(retrievers.keys()),
        'per_nation': {n: {m: results[m][n] for m in retrievers}
                       for n in P5 if results['AAAI'][n]['total']},
        'per_region': {r: {m: results_region[m][r] for m in retrievers}
                       for r in all_regions if results_region['AAAI'][r]['total']},
        'overall': results_overall,
        'mean_rates': mean_rates,
        'created_at': datetime.now().isoformat(),
        'notes': 'Coalition match: ≥75% of OTHER P5 votes match between target and either retrieved precedent (adopted or non-adopted top-1).',
    }, f, indent=2)
print(f"\n✓ Results saved to {output_path}")
print("✓ Cell 26 complete")

In [ ]:
# ============================================================
# CELL 26.5 — Per-pool diagnostic decomposition
# ============================================================
# Cell 26 used "either pool top-1 matches" which inflates rates.
# This cell separates adopted-pool top-1 success from non-adopted-pool
# top-1 success, so we can see which retrievers win where.

import time
from collections import defaultdict

# Per-pool, per-method counters
adopted_results = {m: defaultdict(lambda: {'matches': 0, 'total': 0}) for m in retrievers}
nonadopted_results = {m: defaultdict(lambda: {'matches': 0, 'total': 0}) for m in retrievers}
adopted_overall = {m: {'matches': 0, 'total': 0} for m in retrievers}
nonadopted_overall = {m: {'matches': 0, 'total': 0} for m in retrievers}

# Track agreement: how often do all 4 methods pick the same precedent?
adopted_picks_per_target = defaultdict(dict)  # target_id -> {method: pick_id}
nonadopted_picks_per_target = defaultdict(dict)

print(f"\nRunning per-pool decomposition on {len(test_resolutions)} test resolutions...")
start = time.time()

for i, target in enumerate(test_resolutions):
    if i % 15 == 0 and i > 0:
        elapsed = time.time() - start
        print(f"  [{i}/{len(test_resolutions)}] {elapsed:.0f}s")

    target_coalition = target['coalition']

    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary':              aaai_entry['summary'],
        'action_item':          aaai_entry['action_item'],
        'keyword':              aaai_entry['keyword'],
        'res_no':               aaai_entry['res_no'],
        'date':                 aaai_entry['date'],
        'geopolitical_region':  aaai_entry['geopolitical_region'],
        'target_nations_list':  aaai_entry['target_nations_list'],
    }

    for method, retriever in retrievers.items():
        ad_picks, nad_picks = retriever.retrieve_by_keyword(input_dict, k=1)

        # Adopted pool scoring
        if ad_picks:
            ad_id, ad_content = ad_picks[0]
            ad_coalition = get_retrieved_coalition(ad_content['doc'], is_adopted=True)
            adopted_picks_per_target[target['resolution_id']][method] = ad_id
            for nation_code in P5:
                if target_coalition.get(nation_code) is None:
                    continue
                ok = coalition_match(target_coalition, ad_coalition,
                                     exclude_nation=nation_code)
                adopted_results[method][nation_code]['total'] += 1
                adopted_overall[method]['total'] += 1
                if ok:
                    adopted_results[method][nation_code]['matches'] += 1
                    adopted_overall[method]['matches'] += 1

        # Non-adopted pool scoring
        if nad_picks:
            nad_id, nad_content = nad_picks[0]
            nad_coalition = get_retrieved_coalition(nad_content['doc'], is_adopted=False)
            nonadopted_picks_per_target[target['resolution_id']][method] = nad_id
            for nation_code in P5:
                if target_coalition.get(nation_code) is None:
                    continue
                ok = coalition_match(target_coalition, nad_coalition,
                                     exclude_nation=nation_code)
                nonadopted_results[method][nation_code]['total'] += 1
                nonadopted_overall[method]['total'] += 1
                if ok:
                    nonadopted_results[method][nation_code]['matches'] += 1
                    nonadopted_overall[method]['matches'] += 1

print(f"Completed in {time.time()-start:.0f}s")


# === Print: Adopted pool only ===
print(f"\n{'=' * 80}")
print("ADOPTED POOL ONLY: top-1 coalition match rate")
print('=' * 80)
print(f"{'Nation':<6} {'AAAI':>10} {'BM25':>10} {'Dense':>10} {'CSR-v3':>10}")
print('-' * 80)
for nation in P5:
    rates = {}
    for m in retrievers:
        r = adopted_results[m][nation]
        rates[m] = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{nation:<6} {rates['AAAI']:>9.1f}% {rates['BM25']:>9.1f}% "
          f"{rates['Dense']:>9.1f}% {rates['CSR-v3']:>9.1f}%")
print('-' * 80)
print(f"{'Mean':<6} ", end='')
for m in retrievers:
    r = adopted_overall[m]
    rate = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{rate:>9.1f}% ", end='')
print()


# === Print: Non-adopted pool only ===
print(f"\n{'=' * 80}")
print("NON-ADOPTED POOL ONLY: top-1 coalition match rate")
print('=' * 80)
print(f"{'Nation':<6} {'AAAI':>10} {'BM25':>10} {'Dense':>10} {'CSR-v3':>10}")
print('-' * 80)
for nation in P5:
    rates = {}
    for m in retrievers:
        r = nonadopted_results[m][nation]
        rates[m] = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{nation:<6} {rates['AAAI']:>9.1f}% {rates['BM25']:>9.1f}% "
          f"{rates['Dense']:>9.1f}% {rates['CSR-v3']:>9.1f}%")
print('-' * 80)
print(f"{'Mean':<6} ", end='')
for m in retrievers:
    r = nonadopted_overall[m]
    rate = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{rate:>9.1f}% ", end='')
print()


# === Inter-method agreement ===
print(f"\n{'=' * 70}")
print("INTER-METHOD AGREEMENT: how often do methods pick the same precedent?")
print('=' * 70)

method_pairs = [('AAAI', 'BM25'), ('AAAI', 'Dense'), ('AAAI', 'CSR-v3'),
                ('BM25', 'Dense'), ('BM25', 'CSR-v3'), ('Dense', 'CSR-v3')]

print("\nADOPTED POOL agreement (out of 66 targets):")
for m1, m2 in method_pairs:
    agree = sum(1 for tid in adopted_picks_per_target
                if (m1 in adopted_picks_per_target[tid]
                    and m2 in adopted_picks_per_target[tid]
                    and adopted_picks_per_target[tid][m1] == adopted_picks_per_target[tid][m2]))
    print(f"  {m1:<8} vs {m2:<8}: {agree}/66 ({agree/66*100:.0f}%)")

print("\nNON-ADOPTED POOL agreement:")
for m1, m2 in method_pairs:
    agree = sum(1 for tid in nonadopted_picks_per_target
                if (m1 in nonadopted_picks_per_target[tid]
                    and m2 in nonadopted_picks_per_target[tid]
                    and nonadopted_picks_per_target[tid][m1] == nonadopted_picks_per_target[tid][m2]))
    total = len(nonadopted_picks_per_target)
    print(f"  {m1:<8} vs {m2:<8}: {agree}/{total} ({agree/total*100:.0f}%)" if total else "  no targets")

print("\n✓ Cell 26.5 complete")

In [ ]:
# ============================================================
# CELL 26.6 — Verify Western finding + build hybrid retriever
# ============================================================
# Two purposes:
# 1. Diagnose: is the FRA/GBR/USA win on adopted pool driven by a few
#    resolutions or is it consistent across the test set?
# 2. Build a hybrid retriever (CSR-v3 adopted + Dense non-adopted)
#    and compare it to all four single methods on combined-pool metric.

import time
from collections import defaultdict, Counter


# === Diagnostic 1: Per-resolution contribution to adopted-pool wins ===
# For each test resolution, check whether CSR-v3 retrieved an adopted
# precedent that matched USA's coalition (where CSR wins biggest).
print("=" * 70)
print("DIAGNOSTIC: Where does CSR-v3 win on USA adopted-pool?")
print("=" * 70)

usa_csr_wins = []
usa_csr_losses = []

for target in test_resolutions:
    if target['coalition'].get('USA') is None:
        continue

    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    csr_ad, _ = csr_retriever.retrieve_by_keyword(input_dict, k=1)
    aaai_ad, _ = aaai_retriever.retrieve_by_keyword(input_dict, k=1)

    if not csr_ad or not aaai_ad:
        continue

    csr_coal = get_retrieved_coalition(csr_ad[0][1]['doc'], is_adopted=True)
    aaai_coal = get_retrieved_coalition(aaai_ad[0][1]['doc'], is_adopted=True)

    csr_match = coalition_match(target['coalition'], csr_coal, exclude_nation='USA')
    aaai_match = coalition_match(target['coalition'], aaai_coal, exclude_nation='USA')

    record = {
        'target': target['resolution_id'],
        'region': target['geopolitical_region'],
        'target_usa_vote': target['coalition'].get('USA'),
        'csr_pick': csr_ad[0][0],
        'csr_match': csr_match,
        'aaai_pick': aaai_ad[0][0],
        'aaai_match': aaai_match,
    }
    if csr_match and not aaai_match:
        usa_csr_wins.append(record)
    elif aaai_match and not csr_match:
        usa_csr_losses.append(record)

print(f"\nCSR wins (CSR matches, AAAI doesn't): {len(usa_csr_wins)}")
print(f"CSR losses (AAAI matches, CSR doesn't): {len(usa_csr_losses)}")
print(f"Net: {len(usa_csr_wins) - len(usa_csr_losses)}")

print(f"\nCSR-win regions distribution:")
for region, count in Counter(r['region'] for r in usa_csr_wins).most_common():
    print(f"  {region}: {count}")

print(f"\nFirst 5 CSR wins on USA adopted-pool:")
for r in usa_csr_wins[:5]:
    print(f"  {r['target']} ({r['region']}) USA vote: {r['target_usa_vote']}")
    print(f"    CSR  picked: {r['csr_pick']}  → match")
    print(f"    AAAI picked: {r['aaai_pick']}  → no match")


# === Build hybrid retriever ===
class HybridRetriever:
    """CSR-v3 on adopted pool + Dense on non-adopted pool.
    Best of both: coalition-aware reranking for consensus-heavy adopted pool,
    semantic similarity for already-contested non-adopted pool."""

    def __init__(self, csr, dense):
        self.csr = csr
        self.dense = dense

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        # Get CSR's adopted top-k
        csr_ad, _ = self.csr.retrieve_by_keyword(input_dict, k, threshold)
        # Get Dense's non-adopted top-k
        _, dense_nad = self.dense.retrieve_by_keyword(input_dict, k, threshold)
        return csr_ad, dense_nad


hybrid_retriever = HybridRetriever(csr_retriever, dense_retriever)
print("\n✓ HybridRetriever instantiated (CSR adopted + Dense non-adopted)")


# === Re-run combined-pool comparison (matches Cell 26 metric) with hybrid added ===
print(f"\n{'=' * 80}")
print("FIVE-WAY COMPARISON: combined-pool 'either match' rate")
print('=' * 80)

retrievers_with_hybrid = {**retrievers, 'Hybrid': hybrid_retriever}
results_v2 = {m: defaultdict(lambda: {'matches': 0, 'total': 0})
              for m in retrievers_with_hybrid}
overall_v2 = {m: {'matches': 0, 'total': 0} for m in retrievers_with_hybrid}

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 20 == 0 and i > 0:
        print(f"  [{i}/{len(test_resolutions)}] {time.time()-start:.0f}s")

    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    for method, retriever in retrievers_with_hybrid.items():
        ad, nad = retriever.retrieve_by_keyword(input_dict, k=1)
        retrieved_coals = []
        if ad:
            retrieved_coals.append(get_retrieved_coalition(ad[0][1]['doc'], is_adopted=True))
        if nad:
            retrieved_coals.append(get_retrieved_coalition(nad[0][1]['doc'], is_adopted=False))

        for nation_code in P5:
            if target_coalition.get(nation_code) is None:
                continue
            best = any(coalition_match(target_coalition, rc,
                                         exclude_nation=nation_code)
                       for rc in retrieved_coals)
            results_v2[method][nation_code]['total'] += 1
            overall_v2[method]['total'] += 1
            if best:
                results_v2[method][nation_code]['matches'] += 1
                overall_v2[method]['matches'] += 1

print(f"Completed in {time.time()-start:.0f}s")


print(f"\nCOMBINED-POOL 'EITHER MATCH' RATE (per nation):")
print(f"{'Nation':<6} {'AAAI':>9} {'BM25':>9} {'Dense':>9} {'CSR-v3':>9} {'Hybrid':>9}")
print('-' * 70)
for nation in P5:
    rates = {}
    for m in retrievers_with_hybrid:
        r = results_v2[m][nation]
        rates[m] = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{nation:<6} {rates['AAAI']:>8.1f}% {rates['BM25']:>8.1f}% "
          f"{rates['Dense']:>8.1f}% {rates['CSR-v3']:>8.1f}% {rates['Hybrid']:>8.1f}%")
print('-' * 70)
print(f"{'Mean':<6}", end='')
for m in retrievers_with_hybrid:
    r = overall_v2[m]
    rate = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f" {rate:>8.1f}%", end='')
print()


# === Save expanded results ===
import os
import json
output_path = os.path.join(WORKDIR, 'paper_results/retrieval_quality_v2.json')
with open(output_path, 'w') as f:
    json.dump({
        'methods': list(retrievers_with_hybrid.keys()),
        'per_nation_combined': {n: {m: results_v2[m][n] for m in retrievers_with_hybrid}
                                 for n in P5},
        'overall_combined': overall_v2,
        'csr_usa_wins_count': len(usa_csr_wins),
        'csr_usa_losses_count': len(usa_csr_losses),
        'created_at': datetime.now().isoformat(),
    }, f, indent=2)
print(f"\n✓ Saved to {output_path}")
print("✓ Cell 26.6 complete")

In [ ]:
# ============================================================
# CELL 26.7 — Adaptive per-nation hybrid retriever
# ============================================================
# Routes adopted-pool retrieval per target nation:
#   FRA, GBR, USA → CSR-v3 (entropy reranker captures Western dissent)
#   CHN, RUS      → Dense   (consensus-aligned, semantic similarity wins)
# Non-adopted pool → always Dense (already-contested, semantic wins)
#
# This isn't a runtime decision the LLM makes — it's a design choice
# baked into retrieval. Paper would justify with the adopted-pool
# per-nation analysis from Cell 26.5.

import time
from collections import defaultdict

WESTERN_NATIONS = {'FRA', 'GBR', 'USA'}
EASTERN_NATIONS = {'CHN', 'RUS'}


class AdaptiveHybridRetriever:
    """Routes retrieval per target nation:
    - Adopted pool: CSR-v3 for Western, Dense for Eastern
    - Non-adopted pool: always Dense

    The 'target nation' here means the nation the LLM is simulating.
    The retriever needs to know which nation is being predicted to
    route correctly."""

    def __init__(self, csr, dense):
        self.csr = csr
        self.dense = dense

    def retrieve_by_keyword(self, input_dict, k, threshold=3,
                              target_nation=None):
        """target_nation: code like 'USA' indicating who we're simulating.
        If None, defaults to Dense (safe baseline)."""
        # Adopted pool routing
        if target_nation in WESTERN_NATIONS:
            ad_picks, _ = self.csr.retrieve_by_keyword(input_dict, k, threshold)
        else:
            ad_picks, _ = self.dense.retrieve_by_keyword(input_dict, k, threshold)
        # Non-adopted pool always Dense
        _, nad_picks = self.dense.retrieve_by_keyword(input_dict, k, threshold)
        return ad_picks, nad_picks


adaptive_retriever = AdaptiveHybridRetriever(csr_retriever, dense_retriever)
print("✓ AdaptiveHybridRetriever instantiated")


# === Run six-way comparison: AAAI, BM25, Dense, CSR-v3, Hybrid (CSR+Dense), Adaptive ===
all_retrievers = {
    'AAAI':     aaai_retriever,
    'BM25':     bm25_retriever,
    'Dense':    dense_retriever,
    'CSR-v3':   csr_retriever,
    'Hybrid':   hybrid_retriever,       # CSR adopted + Dense non-adopted
    'Adaptive': adaptive_retriever,     # Per-nation routing
}

results_v3 = {m: defaultdict(lambda: {'matches': 0, 'total': 0})
              for m in all_retrievers}
overall_v3 = {m: {'matches': 0, 'total': 0} for m in all_retrievers}

print(f"\nRunning six-way comparison on {len(test_resolutions)} test resolutions...")
start = time.time()

for i, target in enumerate(test_resolutions):
    if i % 20 == 0 and i > 0:
        print(f"  [{i}/{len(test_resolutions)}] {time.time()-start:.0f}s")

    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    for nation_code in P5:
        if target_coalition.get(nation_code) is None:
            continue

        for method, retriever in all_retrievers.items():
            # Adaptive needs to know the target nation; others ignore it
            if method == 'Adaptive':
                ad, nad = retriever.retrieve_by_keyword(
                    input_dict, k=1, target_nation=nation_code
                )
            else:
                ad, nad = retriever.retrieve_by_keyword(input_dict, k=1)

            retrieved_coals = []
            if ad:
                retrieved_coals.append(
                    get_retrieved_coalition(ad[0][1]['doc'], is_adopted=True)
                )
            if nad:
                retrieved_coals.append(
                    get_retrieved_coalition(nad[0][1]['doc'], is_adopted=False)
                )
            best = any(coalition_match(target_coalition, rc, exclude_nation=nation_code)
                       for rc in retrieved_coals)
            results_v3[method][nation_code]['total'] += 1
            overall_v3[method]['total'] += 1
            if best:
                results_v3[method][nation_code]['matches'] += 1
                overall_v3[method]['matches'] += 1

print(f"Completed in {time.time()-start:.0f}s")


# === Print clean results table ===
print(f"\n{'=' * 95}")
print("SIX-WAY COMPARISON: combined-pool 'either match' rate at k=1")
print('=' * 95)
print(f"{'Nation':<6} {'AAAI':>9} {'BM25':>9} {'Dense':>9} {'CSR-v3':>9} {'Hybrid':>9} {'Adaptive':>10}")
print('-' * 95)
for nation in P5:
    rates = {}
    for m in all_retrievers:
        r = results_v3[m][nation]
        rates[m] = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f"{nation:<6} {rates['AAAI']:>8.1f}% {rates['BM25']:>8.1f}% "
          f"{rates['Dense']:>8.1f}% {rates['CSR-v3']:>8.1f}% "
          f"{rates['Hybrid']:>8.1f}% {rates['Adaptive']:>9.1f}%")
print('-' * 95)
print(f"{'Mean':<6}", end='')
for m in all_retrievers:
    r = overall_v3[m]
    rate = r['matches'] / r['total'] * 100 if r['total'] else 0
    print(f" {rate:>8.1f}%", end='')
print()


# === Decision criteria ===
print(f"\n{'=' * 70}")
print("DECISION CRITERIA")
print('=' * 70)
dense_rate = overall_v3['Dense']['matches'] / overall_v3['Dense']['total'] * 100
adaptive_rate = overall_v3['Adaptive']['matches'] / overall_v3['Adaptive']['total'] * 100
delta = adaptive_rate - dense_rate
print(f"Dense overall:    {dense_rate:.1f}%")
print(f"Adaptive overall: {adaptive_rate:.1f}%")
print(f"Δ Adaptive vs Dense: {delta:+.1f}pp")
print()
if delta >= 5:
    print("✓ Adaptive is meaningfully better (+5pp). Defendable contribution.")
elif delta >= 2:
    print("≈ Adaptive marginally better (+2-5pp). Mention but don't headline.")
elif delta >= 0:
    print("· Adaptive ties Dense. Use Dense as primary baseline, document CSR finding.")
else:
    print("✗ Adaptive worse than Dense. Use Dense as primary, drop hybrid framing.")


# === Save ===
import os, json
output_path = os.path.join(WORKDIR, 'paper_results/retrieval_quality_v3.json')
with open(output_path, 'w') as f:
    json.dump({
        'methods': list(all_retrievers.keys()),
        'per_nation': {n: {m: results_v3[m][n] for m in all_retrievers}
                       for n in P5},
        'overall': overall_v3,
        'dense_vs_adaptive_delta': delta,
        'created_at': datetime.now().isoformat(),
    }, f, indent=2)
print(f"\n✓ Saved to {output_path}")
print("✓ Cell 26.7 complete")

In [ ]:
# ============================================================
# CELL 26.X — Diagnostic 1: Which pool's retrieval matters more?
# ============================================================
# Reasoning: AAAI pipeline rehearses both retrieved precedents
# separately (history_predict + reflexion run twice), then uses
# both rehearsals + final query for the answer.
#
# So we need to know: when the adopted-pool retrieval is good but
# the non-adopted is bad, is that better/worse than the reverse?
# This tells us which retrieval to optimize hardest.

import time
from collections import defaultdict

# For each test resolution × nation, classify the four cases:
#   AA: adopted match + non-adopted match
#   AN: adopted match, non-adopted no-match
#   NA: adopted no-match, non-adopted match
#   NN: neither match
# Do this for each retriever and see the distribution.

case_counts = {m: defaultdict(int) for m in retrievers}

for target in test_resolutions:
    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    for method, retriever in retrievers.items():
        ad, nad = retriever.retrieve_by_keyword(input_dict, k=1)
        ad_coal = (get_retrieved_coalition(ad[0][1]['doc'], is_adopted=True)
                   if ad else None)
        nad_coal = (get_retrieved_coalition(nad[0][1]['doc'], is_adopted=False)
                    if nad else None)

        for nation in P5:
            if target_coalition.get(nation) is None:
                continue
            ad_match = (ad_coal is not None
                        and coalition_match(target_coalition, ad_coal,
                                              exclude_nation=nation))
            nad_match = (nad_coal is not None
                         and coalition_match(target_coalition, nad_coal,
                                               exclude_nation=nation))
            case = ('A' if ad_match else 'N') + ('A' if nad_match else 'N')
            case_counts[method][case] += 1

print(f"{'=' * 70}")
print("CASE BREAKDOWN: adopted/non-adopted match patterns")
print('=' * 70)
print(f"{'Method':<10} {'AA':>8} {'AN':>8} {'NA':>8} {'NN':>8} {'Total':>8}")
print('-' * 60)
for method in retrievers:
    c = case_counts[method]
    total = sum(c.values())
    print(f"{method:<10} {c['AA']:>8} {c['AN']:>8} {c['NA']:>8} {c['NN']:>8} {total:>8}")

print(f"\n{'Method':<10} {'AA%':>8} {'AN%':>8} {'NA%':>8} {'NN%':>8}")
print('-' * 50)
for method in retrievers:
    c = case_counts[method]
    total = sum(c.values())
    print(f"{method:<10} {c['AA']/total*100:>7.1f}% {c['AN']/total*100:>7.1f}% "
          f"{c['NA']/total*100:>7.1f}% {c['NN']/total*100:>7.1f}%")

print(f"\nINTERPRETATION:")
print(f"  AA (both match):        gold standard — both rehearsals reinforce target")
print(f"  AN (only adopted):      adopted-pool rehearsal is the useful signal")
print(f"  NA (only non-adopted):  non-adopted-pool rehearsal is the useful signal")
print(f"  NN (neither):           no useful coalition signal — pipeline is text-only")

In [ ]:
# ============================================================
# CELL 26.Y — Diagnostic 2: Speech text as untapped signal
# ============================================================
# Each test resolution has speeches from P5 nations. AAAI's pipeline
# uses these in Reflexion but not in retrieval. Let's check:
# - How long is the target nation's speech on each test resolution?
# - Does speech length correlate with vote-against?
# - Sample a few speeches from contested-vote cases to see signal
#   we're throwing away in retrieval.

import numpy as np

print(f"{'=' * 70}")
print("SPEECH LENGTH ANALYSIS BY VOTE TYPE (test set, target P5 nations)")
print('=' * 70)

# For each (test resolution, P5 nation, vote), gather speech length
data = []
for entry in AAAI_NONADOPTED_DATASET:
    speeches = entry.get('speech', {})
    if not isinstance(speeches, dict):
        continue
    coalition = aaai_vote_to_coalition(entry['vote'])
    for code, nation_name in NATION_CODE_TO_AAAI.items():
        speech_list = speeches.get(nation_name, [])
        if not speech_list:
            continue
        speech_text = ' '.join(speech_list) if isinstance(speech_list, list) else str(speech_list)
        word_count = len(speech_text.split())
        vote = coalition.get(code)
        if vote:
            data.append({'nation': code, 'vote': vote,
                         'words': word_count,
                         'res_no': entry['res_no']})

# Aggregate
by_vote = defaultdict(list)
for d in data:
    by_vote[d['vote']].append(d['words'])

print(f"\nSpeech lengths by vote (across all P5 × resolutions):")
for vote in ['Y', 'N', 'A']:
    if vote in by_vote:
        lens = by_vote[vote]
        print(f"  {vote}: n={len(lens):>4}, "
              f"mean={np.mean(lens):>5.0f} words, "
              f"median={np.median(lens):>5.0f}, "
              f"max={max(lens):>5}")


# Sample one speech each from different vote types on contested resolutions
print(f"\n{'=' * 70}")
print("SPEECH SAMPLES (truncated to 250 chars)")
print('=' * 70)

# Find a resolution where USA voted AGAINST (rare, signals real opposition)
import random
random.seed(42)
against_examples = [d for d in data if d['vote'] == 'N' and d['nation'] == 'USA']
if against_examples:
    sample = random.choice(against_examples)
    entry = next(e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == sample['res_no'])
    speech = entry['speech']['United States']
    speech_text = ' '.join(speech) if isinstance(speech, list) else str(speech)
    print(f"\nUSA AGAINST on {sample['res_no']} ({entry['geopolitical_region']}):")
    print(f"  {speech_text[:300]}...")

# Russia AGAINST
against_examples = [d for d in data if d['vote'] == 'N' and d['nation'] == 'RUS']
if against_examples:
    sample = random.choice(against_examples)
    entry = next(e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == sample['res_no'])
    speech = entry['speech']['Russian Federation']
    speech_text = ' '.join(speech) if isinstance(speech, list) else str(speech)
    print(f"\nRUS AGAINST on {sample['res_no']} ({entry['geopolitical_region']}):")
    print(f"  {speech_text[:300]}...")

# Russia ABSTAIN
abstain_examples = [d for d in data if d['vote'] == 'A' and d['nation'] == 'RUS']
if abstain_examples:
    sample = random.choice(abstain_examples)
    entry = next(e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == sample['res_no'])
    speech = entry['speech']['Russian Federation']
    speech_text = ' '.join(speech) if isinstance(speech, list) else str(speech)
    print(f"\nRUS ABSTAIN on {sample['res_no']} ({entry['geopolitical_region']}):")
    print(f"  {speech_text[:300]}...")

In [ ]:
# ============================================================
# CELL 26.Z — Diagnostic 3: Coalition-only-distinguishing cases
# ============================================================
# For each test resolution, find cases where:
# - Dense's adopted top-1 has WRONG coalition (text-similar, coalition-different)
# - SOME pre-target precedent in the adopted pool has RIGHT coalition
# - That right-coalition precedent also has decent text similarity
#
# These are the cases where coalition signal can win. Count them per nation.

import numpy as np

print(f"{'=' * 70}")
print("COALITION-DISCRIMINATIVE CASES (per nation, adopted pool)")
print('=' * 70)
print("Cases where Dense's top-1 has wrong coalition AND a coalition-matching")
print("precedent exists in Dense's top-10 (i.e., recoverable by reranking).")
print()

# For each test target × nation, compute:
# - Dense's top-1 adopted
# - Dense's top-10 adopted
# - Whether top-1 matches; whether any of top-2..10 matches
recoverable = defaultdict(int)
not_recoverable = defaultdict(int)
already_correct = defaultdict(int)

for target in test_resolutions:
    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    # Dense top-10 adopted
    ad_top10, _ = dense_retriever.retrieve_by_keyword(input_dict, k=10)

    for nation in P5:
        if target_coalition.get(nation) is None:
            continue
        if not ad_top10:
            continue

        top1_coal = get_retrieved_coalition(ad_top10[0][1]['doc'], is_adopted=True)
        top1_match = coalition_match(target_coalition, top1_coal,
                                       exclude_nation=nation)
        if top1_match:
            already_correct[nation] += 1
            continue

        # Top-1 didn't match — check if any of top-2..10 do
        rest_match = False
        for res_id, content in ad_top10[1:]:
            coal = get_retrieved_coalition(content['doc'], is_adopted=True)
            if coalition_match(target_coalition, coal, exclude_nation=nation):
                rest_match = True
                break
        if rest_match:
            recoverable[nation] += 1
        else:
            not_recoverable[nation] += 1


print(f"{'Nation':<6} {'Top-1 OK':>10} {'Recoverable':>12} {'Unrecoverable':>14}")
print('-' * 50)
for nation in P5:
    a = already_correct[nation]
    r = recoverable[nation]
    nr = not_recoverable[nation]
    total = a + r + nr
    print(f"{nation:<6} {a:>9} ({a/total*100:>3.0f}%) {r:>9} ({r/total*100:>3.0f}%) "
          f"{nr:>10} ({nr/total*100:>3.0f}%)")

print(f"\nINTERPRETATION:")
print(f"  Top-1 OK:       Dense already retrieves a coalition-matching precedent")
print(f"  Recoverable:    A reranker with the right signal could fix this case")
print(f"  Unrecoverable:  Even Dense's top-10 has no matching coalition — pool limit")

In [ ]:
# ============================================================
# CELL 27 — Speech-augmented retrieval prototype
# ============================================================
# For each candidate, embed the target nation's speech (when present).
# At query time, embed the test target's resolution text.
# Match: target's resolution text → candidates' speeches by same nation.
#
# Hypothesis: a nation's past speeches contain political positioning
# language that disambiguates regime (Russia-vetoes Western proposals
# vs Russia-supports anti-imperial proposals). Resolution text alone
# doesn't carry this signal in the way speech text does.

import time
import numpy as np
import torch
import pickle
import os
from collections import defaultdict
from datetime import datetime
from sentence_transformers import SentenceTransformer

SPEECH_EMB_CACHE = os.path.join(WORKDIR, 'speech_embeddings_v1.pkl')


def get_nation_speech(aaai_entry, nation_aaai_name):
    """Extract a nation's speech text from an AAAI entry, joining if list."""
    speeches = aaai_entry.get('speech', {})
    if not isinstance(speeches, dict):
        return ''
    s = speeches.get(nation_aaai_name, '')
    if isinstance(s, list):
        return ' '.join(s).strip()
    return str(s).strip()


def build_or_load_speech_embeddings():
    """Embed each (candidate × nation) speech with BGE-small.
    Returns dict: {pool_name: {nation_code: (emb_array, has_speech_mask)}}
    """
    if os.path.exists(SPEECH_EMB_CACHE):
        print(f"Loading speech embeddings from {SPEECH_EMB_CACHE}")
        with open(SPEECH_EMB_CACHE, 'rb') as f:
            return pickle.load(f)

    print("Building speech embeddings...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(DENSE_MODEL_NAME, device=device)

    cache = {}
    for pool_name, candidates in [
        ('adopted', AAAI_ADOPTED_CANDIDATES),
        ('nonadopted', AAAI_NONADOPTED_CANDIDATES),
    ]:
        cache[pool_name] = {}
        for nation_code in P5:
            nation_aaai = NATION_CODE_TO_AAAI[nation_code]
            speeches = []
            has_speech = []
            for c in candidates:
                speech = get_nation_speech(c['_aaai_entry'], nation_aaai)
                # Truncate to 1500 words to keep embedding tractable
                speech_truncated = ' '.join(speech.split()[:1500])
                speeches.append(speech_truncated if speech_truncated
                                else 'no comments')
                has_speech.append(bool(speech_truncated))

            print(f"  Embedding {len(speeches)} {pool_name} speeches for {nation_code} "
                  f"(have speech: {sum(has_speech)}/{len(speeches)})...")
            embs = model.encode(speeches, normalize_embeddings=True,
                                 batch_size=32, show_progress_bar=False)
            cache[pool_name][nation_code] = (embs, np.array(has_speech))

    with open(SPEECH_EMB_CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f"  Saved to {SPEECH_EMB_CACHE}")
    return cache


# Build speech embeddings (cached after first run)
speech_embs = build_or_load_speech_embeddings()
print(f"\nSpeech embeddings ready.")


# Resolution-text embeddings already exist (for Dense retrieval queries)
# We embed the QUERY (target's resolution) into the same space.
DENSE_MODEL = SentenceTransformer(DENSE_MODEL_NAME,
                                    device='cuda' if torch.cuda.is_available() else 'cpu')
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "


class SpeechAugmentedRetriever:
    """For target nation N and test target T:
    Score each candidate C by:
      similarity( embed(T's resolution text), embed(C's speech by N) )
    where C's speech is the nation N's past statement when voting on C.

    Intuition: candidates where nation N gave a politically-positioning
    speech in response to similar resolution content are stronger
    coalition-prediction precedents than candidates that just look
    textually similar.
    """

    def __init__(self, adopted_candidates, nonadopted_candidates,
                 speech_embs, model):
        self.adopted = adopted_candidates
        self.nonadopted = nonadopted_candidates
        self.speech_embs = speech_embs
        self.model = model

    def _retrieve_pool(self, target_text, target_date, target_id,
                       target_nation, pool, pool_name, k):
        if not target_text.strip():
            return []
        # Encode target with query prefix
        query = BGE_QUERY_PREFIX + target_text
        query_emb = self.model.encode([query], normalize_embeddings=True,
                                        show_progress_bar=False)[0]

        # Get speech embeddings for this nation in this pool
        speech_emb_array, has_speech_mask = self.speech_embs[pool_name][target_nation]
        scores = speech_emb_array @ query_emb  # cosine since normalized

        # Filter: chronology + self-exclusion + must have speech
        scored = []
        for i, c in enumerate(pool):
            if c['date'] >= target_date:
                continue
            if c['resolution_id'] == target_id:
                continue
            if not has_speech_mask[i]:
                # If no speech, fall back to a low score (don't pick)
                continue
            tie = (target_date - c['date']).days * -1e-9
            scored.append((float(scores[i]) + tie, c))

        scored.sort(key=lambda x: x[0], reverse=True)
        return [c for _, c in scored[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3,
                              target_nation=None):
        if target_nation is None:
            target_nation = 'USA'  # safe default

        # Use target's summary + action_item + keyword as query
        target_text = build_bm25_text(input_dict)
        target_date = datetime.fromisoformat(input_dict['date'])
        target_id = input_dict.get('res_no')

        ad_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            target_nation, self.adopted, 'adopted', k
        )
        nad_picks = self._retrieve_pool(
            target_text, target_date, target_id,
            target_nation, self.nonadopted, 'nonadopted', k
        )

        def to_aaai_format(picks):
            return [(c['resolution_id'],
                     {'doc': c['_aaai_entry'],
                      'date': c['_aaai_entry']['date']})
                    for c in picks]

        return to_aaai_format(ad_picks), to_aaai_format(nad_picks)


speech_retriever = SpeechAugmentedRetriever(
    AAAI_ADOPTED_CANDIDATES, AAAI_NONADOPTED_CANDIDATES,
    speech_embs, DENSE_MODEL,
)
print("✓ SpeechAugmentedRetriever ready")


# === Run gate test against Dense ===
print(f"\n{'=' * 80}")
print("GATE TEST: SpeechAugmented vs Dense (per-nation, combined-pool)")
print('=' * 80)

speech_results = defaultdict(lambda: {'matches': 0, 'total': 0})
dense_results_v2 = defaultdict(lambda: {'matches': 0, 'total': 0})

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 20 == 0 and i > 0:
        print(f"  [{i}/{len(test_resolutions)}] {time.time()-start:.0f}s")

    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    for nation_code in P5:
        if target_coalition.get(nation_code) is None:
            continue

        # Speech-augmented (nation-specific)
        sp_ad, sp_nad = speech_retriever.retrieve_by_keyword(
            input_dict, k=1, target_nation=nation_code
        )
        sp_coals = []
        if sp_ad:
            sp_coals.append(get_retrieved_coalition(sp_ad[0][1]['doc'], is_adopted=True))
        if sp_nad:
            sp_coals.append(get_retrieved_coalition(sp_nad[0][1]['doc'], is_adopted=False))
        sp_match = any(coalition_match(target_coalition, c, exclude_nation=nation_code)
                       for c in sp_coals)
        speech_results[nation_code]['total'] += 1
        if sp_match:
            speech_results[nation_code]['matches'] += 1

        # Dense for comparison
        d_ad, d_nad = dense_retriever.retrieve_by_keyword(input_dict, k=1)
        d_coals = []
        if d_ad:
            d_coals.append(get_retrieved_coalition(d_ad[0][1]['doc'], is_adopted=True))
        if d_nad:
            d_coals.append(get_retrieved_coalition(d_nad[0][1]['doc'], is_adopted=False))
        d_match = any(coalition_match(target_coalition, c, exclude_nation=nation_code)
                      for c in d_coals)
        dense_results_v2[nation_code]['total'] += 1
        if d_match:
            dense_results_v2[nation_code]['matches'] += 1

print(f"Completed in {time.time()-start:.0f}s")


# === Print comparison ===
print(f"\n{'Nation':<6} {'Dense':>10} {'Speech-Aug':>12} {'Δ':>8}")
print('-' * 50)
deltas = []
for nation in P5:
    d_rate = dense_results_v2[nation]['matches'] / dense_results_v2[nation]['total'] * 100
    s_rate = speech_results[nation]['matches'] / speech_results[nation]['total'] * 100
    delta = s_rate - d_rate
    deltas.append(delta)
    flag = "✓" if delta >= 5 else "≈" if delta >= 0 else "✗"
    print(f"{nation:<6} {d_rate:>9.1f}% {s_rate:>11.1f}% {delta:>+6.1f}pp {flag}")

mean_delta = np.mean(deltas)
print('-' * 50)
print(f"{'Mean Δ':<19} {mean_delta:>+6.1f}pp")

print(f"\nDECISION:")
if mean_delta >= 5:
    print(f"✓ Speech augmentation works (+{mean_delta:.1f}pp). Real CSR contribution.")
elif mean_delta >= 0:
    print(f"≈ Marginal gain ({mean_delta:+.1f}pp). May be worth exploring per-nation.")
else:
    print(f"✗ Speech-aug worse ({mean_delta:+.1f}pp). Path B: accept Dense as primary.")

print("\n✓ Cell 27 complete")

In [ ]:
# ============================================================
# CELL 28 — Save canonical state v3 with all retrievers ready
# ============================================================
# Persists everything we need for LLM experiments:
#   - All four retrievers (AAAI, BM25, Dense, CSR-v3) and their state
#   - Dense embeddings (cached)
#   - Speech embeddings (cached)
#   - Full retrieval results from Cell 26 and 26.5

import pickle
import os
import json
from datetime import datetime

CANONICAL_V3_PATH = os.path.join(WORKDIR, 'canonical_state_v3.pkl')

state_v3 = {
    # All AAAI integration state from v2
    'AAAI_ADOPTED_CANDIDATES': AAAI_ADOPTED_CANDIDATES,
    'AAAI_NONADOPTED_CANDIDATES': AAAI_NONADOPTED_CANDIDATES,
    'AAAI_ADOPTED_DATASET': AAAI_ADOPTED_DATASET,
    'AAAI_NONADOPTED_DATASET': AAAI_NONADOPTED_DATASET,
    'ckg_by_number': ckg_by_number,
    'csr_hyperparams': {
        'recall_k': 50, 'w_entropy': 1.0, 'w_recency': 0.3, 'w_text': 0.2,
    },

    # Path references for embedding caches (loaded separately by future cells)
    'dense_embeddings_path': os.path.join(WORKDIR, 'dense_embeddings_v1.pkl'),
    'speech_embeddings_path': os.path.join(WORKDIR, 'speech_embeddings_v1.pkl'),
    'dense_model_name': DENSE_MODEL_NAME,

    # Retrieval evaluation results — ready for the paper
    'retrieval_results': {
        'combined_pool_either_match': {
            'AAAI':   {n: dict(results_v3['AAAI'][n])    for n in P5},
            'BM25':   {n: dict(results_v3['BM25'][n])    for n in P5},
            'Dense':  {n: dict(results_v3['Dense'][n])   for n in P5},
            'CSR-v3': {n: dict(results_v3['CSR-v3'][n])  for n in P5},
            'Hybrid': {n: dict(results_v3['Hybrid'][n])  for n in P5},
            'Adaptive': {n: dict(results_v3['Adaptive'][n]) for n in P5},
        } if 'results_v3' in dir() else None,
        'speech_aug_loss_per_nation': {
            n: {
                'speech_rate': speech_results[n]['matches']/speech_results[n]['total']*100,
                'dense_rate': dense_results_v2[n]['matches']/dense_results_v2[n]['total']*100,
            } for n in P5
        },
    },

    'paper_decisions': {
        'primary_retriever': 'Dense',
        'csr_specialty': 'Adopted-pool Western-nation retrieval',
        'experiments_to_run': ['AAAI_text', 'BM25', 'Dense', 'CSR-v3'],
        'rationale': (
            'Dense is strongest single method. CSR-v3 has documented Western-'
            'nation adopted-pool advantage. We run both to characterize when '
            'coalition-aware retrieval helps downstream debiasing.'
        ),
    },

    'created_at': datetime.now().isoformat(),
    'version': 'v3_retrieval_finalized',
}

with open(CANONICAL_V3_PATH, 'wb') as f:
    pickle.dump(state_v3, f)

print(f"Saved canonical state v3: {CANONICAL_V3_PATH}")
print(f"Size: {os.path.getsize(CANONICAL_V3_PATH) / 1e6:.1f} MB")
print()
print("Paper decisions captured:")
print(f"  Primary retriever:  {state_v3['paper_decisions']['primary_retriever']}")
print(f"  CSR specialty:      {state_v3['paper_decisions']['csr_specialty']}")
print(f"  Experiments to run: {state_v3['paper_decisions']['experiments_to_run']}")
print()
print("Future sessions can run Cell 0 → Cell 0b → load this v3 state directly")
print("\n✓ Cell 28 complete")

In [ ]:
# ============================================================
# CELL 29 — Reciprocal Rank Fusion (RRF) of multiple retrievers
# ============================================================
# Combines rankings from Dense + CSR-v3 + BM25 + AAAI using RRF.
# Score: sum over retrievers of 1/(k + rank). Higher = candidates
# ranked well by multiple retrievers.
#
# Standard practice in production search; adds signal when retrievers
# capture different aspects of relevance (orthogonal signal).
#
# We retrieve top-K from each retriever, then fuse, then take final top-1.

import time
from collections import defaultdict
import numpy as np


class RRFRetriever:
    """Reciprocal Rank Fusion of multiple retrievers.

    For each pool, get top-recall_k from each constituent retriever.
    Compute RRF score per candidate: sum over retrievers of 1/(k+rank).
    Return top-final_k by RRF score.
    """

    def __init__(self, retrievers, recall_k=20, k_rrf=60):
        """retrievers: dict of {name: retriever_instance}
        recall_k: how many to take from each retriever before fusion
        k_rrf: RRF damping constant (60 is standard from the original paper)
        """
        self.retrievers = retrievers
        self.recall_k = recall_k
        self.k_rrf = k_rrf

    def _fuse_pool(self, target_input, pool_attr, k):
        """Get top-recall_k from each retriever for one pool, fuse via RRF."""
        # Track RRF scores per resolution_id
        rrf_scores = defaultdict(float)
        # Also keep a reference to the candidate object for each id
        candidate_by_id = {}

        for name, retriever in self.retrievers.items():
            # Each retriever's retrieve_by_keyword returns
            # (adopted_picks, nonadopted_picks)
            ad, nad = retriever.retrieve_by_keyword(
                target_input, k=self.recall_k
            )
            picks = ad if pool_attr == 'adopted' else nad

            for rank, (res_id, content) in enumerate(picks):
                rrf_scores[res_id] += 1.0 / (self.k_rrf + rank + 1)
                if res_id not in candidate_by_id:
                    candidate_by_id[res_id] = content

        # Sort by RRF score, return top-k
        sorted_ids = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i],
                             reverse=True)
        return [(rid, candidate_by_id[rid]) for rid in sorted_ids[:k]]

    def retrieve_by_keyword(self, input_dict, k, threshold=3):
        adopted_picks = self._fuse_pool(input_dict, 'adopted', k)
        nonadopted_picks = self._fuse_pool(input_dict, 'nonadopted', k)
        return adopted_picks, nonadopted_picks


# Build RRF with all four retrievers
rrf_all = RRFRetriever({
    'AAAI':   aaai_retriever,
    'BM25':   bm25_retriever,
    'Dense':  dense_retriever,
    'CSR-v3': csr_retriever,
}, recall_k=20, k_rrf=60)

# Also build a 2-method RRF (Dense + CSR-v3) — test whether more
# retrievers helps or hurts
rrf_dense_csr = RRFRetriever({
    'Dense':  dense_retriever,
    'CSR-v3': csr_retriever,
}, recall_k=20, k_rrf=60)

print("✓ RRF retrievers built (4-way and 2-way Dense+CSR)")


# === Run gate against Dense and CSR-v3 ===
print(f"\n{'=' * 80}")
print("GATE TEST: RRF variants vs Dense (combined-pool either-match)")
print('=' * 80)

methods_to_compare = {
    'Dense':       dense_retriever,
    'CSR-v3':      csr_retriever,
    'RRF-2way':    rrf_dense_csr,
    'RRF-4way':    rrf_all,
}

results = {m: defaultdict(lambda: {'matches': 0, 'total': 0})
           for m in methods_to_compare}

start = time.time()
for i, target in enumerate(test_resolutions):
    if i % 20 == 0 and i > 0:
        print(f"  [{i}/{len(test_resolutions)}] {time.time()-start:.0f}s")

    target_coalition = target['coalition']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }

    for method, retriever in methods_to_compare.items():
        ad, nad = retriever.retrieve_by_keyword(input_dict, k=1)
        coals = []
        if ad:
            coals.append(get_retrieved_coalition(ad[0][1]['doc'], is_adopted=True))
        if nad:
            coals.append(get_retrieved_coalition(nad[0][1]['doc'], is_adopted=False))

        for nation in P5:
            if target_coalition.get(nation) is None:
                continue
            best = any(coalition_match(target_coalition, c, exclude_nation=nation)
                       for c in coals)
            results[method][nation]['total'] += 1
            if best:
                results[method][nation]['matches'] += 1

print(f"Completed in {time.time()-start:.0f}s")


# === Print results ===
print(f"\n{'Nation':<6} {'Dense':>9} {'CSR-v3':>9} {'RRF-2way':>10} {'RRF-4way':>10}")
print('-' * 60)
deltas_2way = []
deltas_4way = []
for nation in P5:
    rates = {}
    for m in methods_to_compare:
        r = results[m][nation]
        rates[m] = r['matches'] / r['total'] * 100 if r['total'] else 0
    deltas_2way.append(rates['RRF-2way'] - rates['Dense'])
    deltas_4way.append(rates['RRF-4way'] - rates['Dense'])
    print(f"{nation:<6} {rates['Dense']:>8.1f}% {rates['CSR-v3']:>8.1f}% "
          f"{rates['RRF-2way']:>9.1f}% {rates['RRF-4way']:>9.1f}%")

print('-' * 60)
print(f"{'Mean Δ vs Dense':<19} {np.mean(deltas_2way):>+8.1f}pp {np.mean(deltas_4way):>+9.1f}pp")


# === Per-region breakdown for the winning RRF (whichever) ===
best_rrf_name = 'RRF-2way' if np.mean(deltas_2way) >= np.mean(deltas_4way) else 'RRF-4way'
best_rrf = methods_to_compare[best_rrf_name]
print(f"\n{best_rrf_name} per-region breakdown:")

results_region = {m: defaultdict(lambda: {'matches': 0, 'total': 0})
                   for m in ['Dense', best_rrf_name]}
for target in test_resolutions:
    target_coalition = target['coalition']
    region = target['geopolitical_region']
    aaai_entry = next(
        (e for e in AAAI_NONADOPTED_DATASET if e['res_no'] == target['resolution_id']),
        None
    )
    if aaai_entry is None:
        continue
    input_dict = {
        'summary': aaai_entry['summary'],
        'action_item': aaai_entry['action_item'],
        'keyword': aaai_entry['keyword'],
        'res_no': aaai_entry['res_no'],
        'date': aaai_entry['date'],
        'geopolitical_region': aaai_entry['geopolitical_region'],
        'target_nations_list': aaai_entry['target_nations_list'],
    }
    for method, retriever in [('Dense', dense_retriever),
                                (best_rrf_name, best_rrf)]:
        ad, nad = retriever.retrieve_by_keyword(input_dict, k=1)
        coals = []
        if ad: coals.append(get_retrieved_coalition(ad[0][1]['doc'], is_adopted=True))
        if nad: coals.append(get_retrieved_coalition(nad[0][1]['doc'], is_adopted=False))
        for nation in P5:
            if target_coalition.get(nation) is None: continue
            best = any(coalition_match(target_coalition, c, exclude_nation=nation)
                       for c in coals)
            results_region[method][region]['total'] += 1
            if best:
                results_region[method][region]['matches'] += 1

print(f"\n{'Region':<25} {'Dense':>8} {best_rrf_name:>10} {'Δ':>8} {'n':>5}")
print('-' * 60)
for region in sorted(results_region['Dense'].keys(),
                      key=lambda r: -results_region['Dense'][r]['total']):
    n = results_region['Dense'][region]['total']
    if n == 0: continue
    d = results_region['Dense'][region]['matches'] / n * 100
    r = results_region[best_rrf_name][region]['matches'] / n * 100
    print(f"{region:<25} {d:>7.1f}% {r:>9.1f}% {r-d:>+6.1f}pp {n:>5}")


# === Decision ===
print(f"\n{'=' * 60}")
print("DECISION CRITERIA")
print('=' * 60)
mean_2way = np.mean(deltas_2way)
mean_4way = np.mean(deltas_4way)
best = max(mean_2way, mean_4way)
if best >= 5:
    print(f"✓ RRF beats Dense by {best:+.1f}pp. New CSR contribution found.")
elif best >= 2:
    print(f"≈ RRF marginal ({best:+.1f}pp). Mention as analysis but don't headline.")
elif best >= 0:
    print(f"· RRF ties Dense ({best:+.1f}pp). No real gain from fusion.")
else:
    print(f"✗ RRF worse than Dense ({best:+.1f}pp). Confirms retrieval ceiling.")

print("\n✓ Cell 29 complete")

In [ ]:
# ============================================================
# CELL 30 — Full LLM experiment runner
# ============================================================
# Runs B0 / B1 / B2 / B3 × 3 models × 5 nations × 66 resolutions × 3 runs
# Saves after every resolution — crash-safe.
# Prerequisites: Cell 0 + Cell 0b + Cell 0c already run.
#
# Cost estimate:
#   GPT-4o-mini    ~$5  (OpenAI API)
#   Llama-3.3-70B  ~$4  (Together.ai)
#   Mistral-24B    ~$3  (Together.ai)
#   Total:         ~$12
# ============================================================

import os, sys, json, time, random, importlib
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_together import Together
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.runnables import RunnableConfig

# ── API keys (fill in before running) ────────────────────────
OPENAI_API_KEY    = "sk-..."          # OpenAI
TOGETHER_API_KEY  = "..."             # Together.ai (Llama + Mistral)
# ─────────────────────────────────────────────────────────────

os.environ["OPENAI_API_KEY"]   = OPENAI_API_KEY
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY

RESULTS_DIR = os.path.join(WORKDIR, "llm_results")
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Model registry ────────────────────────────────────────────
def build_llm(model_key):
    if model_key == "gpt4o_mini":
        return ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=500)
    elif model_key == "llama70b":
        return Together(
            model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
            temperature=0, max_tokens=500,
            together_api_key=TOGETHER_API_KEY
        )
    elif model_key == "mistral24b":
        return Together(
            model="mistralai/Mistral-Small-24B-Instruct-2501",
            temperature=0, max_tokens=500,
            together_api_key=TOGETHER_API_KEY
        )
    else:
        raise ValueError(f"Unknown model key: {model_key}")

MODEL_REGISTRY = {
    "gpt4o_mini":  "GPT-4o-mini",
    "llama70b":    "Llama-3.3-70B",
    "mistral24b":  "Mistral-Small-24B",
}

NATIONS = ["China", "France", "United Kingdom",
           "Russian Federation", "United States"]

# ── AAAI imports ─────────────────────────────────────────────
original_cwd = os.getcwd()
os.chdir(REPO_DIR)
import Framework.RAG_RFLX_GRAPH as rag_module
importlib.reload(rag_module)
from Framework.RAG_RFLX_GRAPH import RAGRflxGraph
from util.Helper.Helper import (
    input_dict_generator, get_vote_result,
    formatter_adopted_docs, formatter_not_adopted_docs,
    evaluate_confusion_matrix
)
from util.CustomClass.CustomGraphState import CustomGraphState, BasicGraphState
from util.CustomClass.CustomClass import AnswerFormat
from util.templates.VoteTemplates import basic_voting_template
from util.Helper.Helper import vote_normalizer_v2
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langgraph.graph import END, StateGraph
os.chdir(original_cwd)
print("✓ AAAI framework imported")


# ─────────────────────────────────────────────────────────────
# Retriever monkey-patches
# ─────────────────────────────────────────────────────────────

def make_csr_retrieve(retriever_instance):
    """Returns a retrieve_document method using the given retriever."""
    def _retrieve(self, state):
        error_flag = False
        country = state["country"]
        input_dict = {
            "summary":            state["question_summary"],
            "res_no":             state["res_id"],
            "date":               state["question_date"],
            "geopolitical_region": state["question_geographic_keywords"],
            "target_nations_list": state["question_target_nation_keywords"],
            "keyword":            state["question_keywords"],
        }
        k = state["k"]
        adopted_docs, nonadopted_docs = retriever_instance.retrieve_by_keyword(
            input_dict, k, threshold=3
        )
        retrieved_docs_dict = {}
        if adopted_docs:
            for key, rc in adopted_docs:
                retrieved_docs_dict[key] = {"date": rc["date"], "adopted": True}
        if nonadopted_docs:
            for key, rc in nonadopted_docs:
                retrieved_docs_dict[key] = {"date": rc["date"], "adopted": False}
        retrieved_docs_dict = dict(sorted(
            retrieved_docs_dict.items(), key=lambda x: x[1]["date"]))
        retrieved_docs_order = list(retrieved_docs_dict.keys())

        retrieved_docs_dictionary = {}
        if adopted_docs:
            rno, dfmt, sfmt, sumfmt = formatter_adopted_docs(adopted_docs, country)
            for k_, d, s, sm in zip(rno, dfmt, sfmt, sumfmt):
                retrieved_docs_dictionary[k_] = {
                    "adopted": True, "context": d, "speech": s, "summary": sm}
        if nonadopted_docs:
            rno, dfmt, vlist, sfmt, sumfmt = formatter_not_adopted_docs(
                nonadopted_docs, country)
            for k_, d, s, v, sm in zip(rno, dfmt, sfmt, vlist, sumfmt):
                retrieved_docs_dictionary[k_] = {
                    "adopted": False, "context": d,
                    "speech": s, "vote": v, "summary": sm}

        sorted_dict = {
            k_: retrieved_docs_dictionary[k_]
            for k_ in retrieved_docs_order if k_ in retrieved_docs_dictionary
        }
        return CustomGraphState(
            retrieved_docs_dictionary_sorted=sorted_dict,
            retrieved_doc_no=len(retrieved_docs_order),
            retrieved_docs_order=retrieved_docs_order,
            curr_idx_of_execution=0,
            error_flag=error_flag,
        )
    return _retrieve


def build_b0_app(llm):
    """B0: Zero-shot — no retrieval, no Reflexion."""
    os.chdir(REPO_DIR)
    vote_parser = PydanticOutputParser(pydantic_object=AnswerFormat)
    voting_prompt = PromptTemplate.from_template(basic_voting_template)
    voting_prompt = voting_prompt.partial(format=vote_parser.get_format_instructions())
    voting_chain = voting_prompt | llm

    def voting_phase(state: BasicGraphState) -> BasicGraphState:
        resolution = state["question"]
        country = state["country"]
        response = voting_chain.invoke({"country": country, "resolution": resolution})
        content = response.content if hasattr(response, "content") else str(response)
        parsed_result, _ = vote_normalizer_v2(content, debug_flag=False)
        return BasicGraphState(vote_and_rationale=parsed_result)

    workflow = StateGraph(BasicGraphState)
    workflow.add_node("voting_phase", voting_phase)
    workflow.add_edge("voting_phase", END)
    workflow.set_entry_point("voting_phase")
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    os.chdir(original_cwd)
    return app


def build_rag_app(llm, condition):
    """B1 / B2 / B3: RAG + Reflexion with different retrievers."""
    os.chdir(REPO_DIR)
    importlib.reload(rag_module)
    from Framework.RAG_RFLX_GRAPH import RAGRflxGraph

    if condition == "B1":
        # Original AAAI retriever — no monkey-patch
        pass
    elif condition == "B2":
        RAGRflxGraph.retrieve_document = make_csr_retrieve(dense_retriever)
    elif condition == "B3":
        RAGRflxGraph.retrieve_document = make_csr_retrieve(csr_retriever)

    workflow_builder = RAGRflxGraph(llm, debug_flag=False, Verbal=False)
    workflow = workflow_builder.building_workflow()
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)
    os.chdir(original_cwd)
    return app


# ─────────────────────────────────────────────────────────────
# Load or init results dict (crash-safe checkpoint)
# ─────────────────────────────────────────────────────────────

def results_path(model_key, condition):
    return os.path.join(RESULTS_DIR, f"{model_key}_{condition}.json")

def load_results(model_key, condition):
    path = results_path(model_key, condition)
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return {}

def save_results(results, model_key, condition):
    path = results_path(model_key, condition)
    with open(path, "w") as f:
        json.dump(results, f, indent=2,
                  default=lambda o: o.__dict__ if hasattr(o, "__dict__") else str(o))


# ─────────────────────────────────────────────────────────────
# Core runner: one (model, condition, nation) sweep
# ─────────────────────────────────────────────────────────────

def run_one_sweep(model_key, condition, persona, n_runs=3):
    """
    Runs all 66 test resolutions for one (model, condition, persona).
    Saves after every resolution. Skips already-completed entries.
    """
    print(f"\n{'='*65}")
    print(f"  {MODEL_REGISTRY[model_key]} | {condition} | {persona}")
    print(f"{'='*65}")

    results = load_results(model_key, condition)
    if persona not in results:
        results[persona] = {}

    llm = build_llm(model_key)

    # Build the app once per sweep
    if condition == "B0":
        app = build_b0_app(llm)
    else:
        app = build_rag_app(llm, condition)

    timestamp = datetime.now().strftime("%m%d%H%M")

    for idx, res in enumerate(AAAI_NONADOPTED_DATASET):
        res_no = res["res_no"]

        # Skip if all runs done
        existing = results[persona].get(res_no, {})
        done_runs = [r for r in existing.get("runs", [])
                     if r.get("vote") not in (None, "dumb_data")]
        if len(done_runs) >= n_runs:
            print(f"  [{idx+1:02d}/66] {res_no} — skipped (already done)")
            continue

        true_vote = get_vote_result(res["vote"], persona)
        run_votes = []

        for run_i in range(n_runs):
            thread_id = f"{model_key}_{condition}_{persona}_{res_no}_r{run_i}_{timestamp}"
            config = RunnableConfig(
                recursion_limit=25,
                configurable={"thread_id": thread_id}
            )

            try:
                if condition == "B0":
                    # B0 uses BasicGraphState
                    input_dict = {
                        "question": res["context"],
                        "question_date": res["date"],
                        "question_summary": res["summary"],
                        "question_action_item": res.get("action_item", ""),
                        "question_keywords": res.get("keyword", ""),
                        "country": persona,
                    }
                    for event in app.stream(input_dict, config=config):
                        pass
                    final = app.get_state(config).values
                    vote = final["vote_and_rationale"].vote
                    rationale = final["vote_and_rationale"].rationale
                else:
                    # B1 / B2 / B3 use CustomGraphState
                    input_dict = input_dict_generator(res, persona, k=1)
                    for event in app.stream(input_dict, config=config):
                        pass
                    final = app.get_state(config).values
                    vote = final["answer"].vote
                    rationale = final["answer"].rationale

            except Exception as e:
                print(f"    ✗ Run {run_i+1} error: {type(e).__name__}: {str(e)[:80]}")
                vote, rationale = "error", str(e)[:200]

            run_votes.append({"run": run_i, "vote": vote, "rationale": rationale})
            time.sleep(1.5)  # rate-limit buffer

        # Majority vote across runs
        valid_votes = [r["vote"] for r in run_votes
                       if r["vote"] not in ("error", "dumb_data")]
        if valid_votes:
            majority_vote = max(set(valid_votes), key=valid_votes.count)
        else:
            majority_vote = "error"

        results[persona][res_no] = {
            "true_vote": true_vote,
            "majority_vote": majority_vote,
            "runs": run_votes,
            "region": res.get("geopolitical_region", "Unknown"),
        }

        # Save after every resolution
        save_results(results, model_key, condition)
        print(f"  [{idx+1:02d}/66] {res_no} | true={true_vote} | "
              f"pred={majority_vote} | "
              f"votes={[r['vote'] for r in run_votes]}")
        time.sleep(1)

    print(f"\n  ✓ Done: {persona} under {condition} with {MODEL_REGISTRY[model_key]}")
    return results


# ─────────────────────────────────────────────────────────────
# MASTER RUNNER — configure what to run here
# ─────────────────────────────────────────────────────────────

# Adjust this list to run a subset — good for testing one model first
EXPERIMENT_PLAN = [
    # (model_key,     condition, persona)
    # ── GPT-4o-mini ──────────────────────────────────────────
    ("gpt4o_mini", "B0", "Russian Federation"),
    ("gpt4o_mini", "B0", "China"),
    ("gpt4o_mini", "B0", "France"),
    ("gpt4o_mini", "B0", "United Kingdom"),
    ("gpt4o_mini", "B0", "United States"),
    ("gpt4o_mini", "B1", "Russian Federation"),
    ("gpt4o_mini", "B1", "China"),
    ("gpt4o_mini", "B1", "France"),
    ("gpt4o_mini", "B1", "United Kingdom"),
    ("gpt4o_mini", "B1", "United States"),
    ("gpt4o_mini", "B2", "Russian Federation"),
    ("gpt4o_mini", "B2", "China"),
    ("gpt4o_mini", "B2", "France"),
    ("gpt4o_mini", "B2", "United Kingdom"),
    ("gpt4o_mini", "B2", "United States"),
    ("gpt4o_mini", "B3", "Russian Federation"),
    ("gpt4o_mini", "B3", "China"),
    ("gpt4o_mini", "B3", "France"),
    ("gpt4o_mini", "B3", "United Kingdom"),
    ("gpt4o_mini", "B3", "United States"),
    # ── Llama-3.3-70B ────────────────────────────────────────
    ("llama70b",   "B0", "Russian Federation"),
    ("llama70b",   "B0", "China"),
    ("llama70b",   "B0", "France"),
    ("llama70b",   "B0", "United Kingdom"),
    ("llama70b",   "B0", "United States"),
    ("llama70b",   "B1", "Russian Federation"),
    ("llama70b",   "B1", "China"),
    ("llama70b",   "B1", "France"),
    ("llama70b",   "B1", "United Kingdom"),
    ("llama70b",   "B1", "United States"),
    ("llama70b",   "B2", "Russian Federation"),
    ("llama70b",   "B2", "China"),
    ("llama70b",   "B2", "France"),
    ("llama70b",   "B2", "United Kingdom"),
    ("llama70b",   "B2", "United States"),
    ("llama70b",   "B3", "Russian Federation"),
    ("llama70b",   "B3", "China"),
    ("llama70b",   "B3", "France"),
    ("llama70b",   "B3", "United Kingdom"),
    ("llama70b",   "B3", "United States"),
    # ── Mistral-Small-24B ────────────────────────────────────
    ("mistral24b", "B0", "Russian Federation"),
    ("mistral24b", "B0", "China"),
    ("mistral24b", "B0", "France"),
    ("mistral24b", "B0", "United Kingdom"),
    ("mistral24b", "B0", "United States"),
    ("mistral24b", "B1", "Russian Federation"),
    ("mistral24b", "B1", "China"),
    ("mistral24b", "B1", "France"),
    ("mistral24b", "B1", "United Kingdom"),
    ("mistral24b", "B1", "United States"),
    ("mistral24b", "B2", "Russian Federation"),
    ("mistral24b", "B2", "China"),
    ("mistral24b", "B2", "France"),
    ("mistral24b", "B2", "United Kingdom"),
    ("mistral24b", "B2", "United States"),
    ("mistral24b", "B3", "Russian Federation"),
    ("mistral24b", "B3", "China"),
    ("mistral24b", "B3", "France"),
    ("mistral24b", "B3", "United Kingdom"),
    ("mistral24b", "B3", "United States"),
]


# ── SMOKE TEST: run one resolution first to verify cost + timing ──
# Uncomment to test one resolution before the full sweep:
#
# print("SMOKE TEST: one resolution, one model, one condition")
# run_one_sweep("gpt4o_mini", "B3", "Russian Federation", n_runs=1)
# print("Smoke test done. Check result and cost before running full sweep.")


# ── FULL SWEEP ────────────────────────────────────────────────
total = len(EXPERIMENT_PLAN)
print(f"\nStarting full sweep: {total} (model, condition, nation) combinations")
print(f"Each = 66 resolutions × 3 runs = 198 LLM calls")
print(f"Total LLM calls: {total * 66 * 3:,}")
print(f"Estimated cost: ~$12 total\n")

for i, (model_key, condition, persona) in enumerate(EXPERIMENT_PLAN):
    print(f"\n[{i+1}/{total}] Starting: {model_key} | {condition} | {persona}")
    try:
        run_one_sweep(model_key, condition, persona, n_runs=3)
    except Exception as e:
        print(f"  ✗ SWEEP FAILED: {type(e).__name__}: {e}")
        print(f"  Continuing to next sweep...")
        time.sleep(5)

print("\n" + "="*65)
print("✓ ALL SWEEPS COMPLETE")
print(f"Results saved to: {RESULTS_DIR}")
print("="*65)

In [ ]:
# ============================================================
# CELL 31 — Compute Weighted F1 and BCS from saved results
# ============================================================
# Run after Cell 30 finishes (or partially finishes).
# Can be re-run at any time to check interim progress.

import json, os
import numpy as np
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import f1_score
from collections import defaultdict, Counter

RESULTS_DIR = os.path.join(WORKDIR, "llm_results")

NATION_CODE = {
    "China":              "CHN",
    "France":             "FRA",
    "United Kingdom":     "GBR",
    "Russian Federation": "RUS",
    "United States":      "USA",
}

CONDITIONS   = ["B0", "B1", "B2", "B3"]
MODEL_KEYS   = ["gpt4o_mini", "llama70b", "mistral24b"]
MODEL_LABELS = {
    "gpt4o_mini":  "GPT-4o-mini",
    "llama70b":    "Llama-3.3-70B",
    "mistral24b":  "Mistral-Small-24B",
}
VOTE_CLASSES = ["favour", "against", "abstention"]


def normalize_vote(v):
    if not v: return "error"
    v = v.lower().strip()
    if any(x in v for x in ["favour", "favor", "yes", "for"]):   return "favour"
    if any(x in v for x in ["against", "no", "veto"]):           return "against"
    if "abst" in v:                                               return "abstention"
    return "error"


def compute_weighted_f1(true_list, pred_list):
    """Weighted F1 per AAAI methodology."""
    true_norm = [normalize_vote(v) for v in true_list]
    pred_norm = [normalize_vote(v) for v in pred_list]
    valid = [(t, p) for t, p in zip(true_norm, pred_norm)
             if t != "error" and p != "error"]
    if not valid: return 0.0
    t_list, p_list = zip(*valid)
    present = list(set(t_list))
    try:
        return f1_score(t_list, p_list, labels=present,
                        average="weighted", zero_division=0)
    except Exception:
        return 0.0


def compute_bcs(pred_votes, true_votes, n_bootstrap=10000):
    """
    BCS = 1 - JSD(P_LLM || P_historical)
    Returns (bcs_mean, ci_lower, ci_upper).
    """
    pred_norm = [normalize_vote(v) for v in pred_votes if normalize_vote(v) != "error"]
    true_norm = [normalize_vote(v) for v in true_votes if normalize_vote(v) != "error"]
    if not pred_norm or not true_norm:
        return None, None, None

    def to_dist(votes):
        c = Counter(votes)
        total = sum(c.values())
        return np.array([c.get(cls, 0) / total for cls in VOTE_CLASSES])

    # Smooth to avoid zero-probability issues
    eps = 1e-9
    p_llm  = to_dist(pred_norm) + eps
    p_hist = to_dist(true_norm) + eps
    p_llm  /= p_llm.sum()
    p_hist /= p_hist.sum()

    jsd = jensenshannon(p_llm, p_hist, base=2) ** 2  # JSD is sqrt, square back
    bcs_point = float(1.0 - jsd)

    # Bootstrap CI
    n = len(pred_norm)
    bcs_samples = []
    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        sample_pred = [pred_norm[i] for i in idx]
        sample_true = [true_norm[i] for i in idx]
        p_b = to_dist(sample_pred) + eps; p_b /= p_b.sum()
        p_h = to_dist(sample_true) + eps; p_h /= p_h.sum()
        jsd_b = jensenshannon(p_b, p_h, base=2) ** 2
        bcs_samples.append(1.0 - jsd_b)

    ci_lo = float(np.percentile(bcs_samples, 2.5))
    ci_hi = float(np.percentile(bcs_samples, 97.5))
    return round(bcs_point, 4), round(ci_lo, 4), round(ci_hi, 4)


# ─────────────────────────────────────────────────────────────
# Load all results and compute metrics
# ─────────────────────────────────────────────────────────────

# metrics[model][condition][nation] = {"f1": float, "bcs": float, "ci_lo": float, "ci_hi": float}
metrics = defaultdict(lambda: defaultdict(dict))
# region_metrics[model][condition][nation][region] = {"f1", "bcs", ...}
region_metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for model_key in MODEL_KEYS:
    for condition in CONDITIONS:
        path = os.path.join(RESULTS_DIR, f"{model_key}_{condition}.json")
        if not os.path.exists(path):
            print(f"  Missing: {model_key}_{condition}.json — skipping")
            continue
        with open(path) as f:
            results = json.load(f)

        for persona, res_dict in results.items():
            if not res_dict:
                continue
            nation_code = NATION_CODE.get(persona, persona)

            true_votes, pred_votes = [], []
            # Group by region for per-region analysis
            region_data = defaultdict(lambda: {"true": [], "pred": []})

            for res_no, entry in res_dict.items():
                tv = entry.get("true_vote", "")
                pv = entry.get("majority_vote", "error")
                region = entry.get("region", "Unknown")

                if normalize_vote(tv) != "error":
                    true_votes.append(tv)
                    pred_votes.append(pv)
                    region_data[region]["true"].append(tv)
                    region_data[region]["pred"].append(pv)

            if not true_votes:
                continue

            f1  = compute_weighted_f1(true_votes, pred_votes)
            bcs_mean, ci_lo, ci_hi = compute_bcs(pred_votes, true_votes)

            metrics[model_key][condition][nation_code] = {
                "f1":    round(f1, 4),
                "bcs":   bcs_mean,
                "ci_lo": ci_lo,
                "ci_hi": ci_hi,
                "n":     len(true_votes),
            }

            for region, data in region_data.items():
                r_f1 = compute_weighted_f1(data["true"], data["pred"])
                r_bcs, r_lo, r_hi = compute_bcs(data["pred"], data["true"],
                                                   n_bootstrap=2000)
                region_metrics[model_key][condition][nation_code][region] = {
                    "f1":    round(r_f1, 4),
                    "bcs":   r_bcs,
                    "ci_lo": r_lo,
                    "ci_hi": r_hi,
                    "n":     len(data["true"]),
                }

# ─────────────────────────────────────────────────────────────
# Print: Weighted F1 table (mirrors AAAI Table 3)
# ─────────────────────────────────────────────────────────────
NATIONS_ORDER = ["CHN", "FRA", "GBR", "RUS", "USA"]

print("\n" + "="*80)
print("TABLE: Weighted F1 (mirrors Choi et al. 2026 Table 3)")
print("="*80)
header = f"{'Model':<20} {'Cond':<5} " + "  ".join(f"{n:>6}" for n in NATIONS_ORDER) + "  Mean"
print(header)
print("-"*80)

for model_key in MODEL_KEYS:
    for condition in CONDITIONS:
        row = f"{MODEL_LABELS[model_key]:<20} {condition:<5} "
        vals = []
        for nation in NATIONS_ORDER:
            entry = metrics[model_key][condition].get(nation)
            if entry:
                val = entry["f1"]
                row += f"  {val:>5.3f}"
                vals.append(val)
            else:
                row += "  -----"
        if vals:
            row += f"  {np.mean(vals):>5.3f}"
        print(row)
    print()

# ─────────────────────────────────────────────────────────────
# Print: BCS table
# ─────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TABLE: Behavioral Consistency Score (BCS) with 95% CI")
print("="*80)
header = f"{'Model':<20} {'Cond':<5} " + "  ".join(f"{n:>8}" for n in NATIONS_ORDER)
print(header)
print("-"*80)

for model_key in MODEL_KEYS:
    for condition in CONDITIONS:
        row = f"{MODEL_LABELS[model_key]:<20} {condition:<5} "
        for nation in NATIONS_ORDER:
            entry = metrics[model_key][condition].get(nation)
            if entry and entry["bcs"] is not None:
                bcs  = entry["bcs"]
                ci_lo = entry["ci_lo"]
                ci_hi = entry["ci_hi"]
                row += f"  {bcs:.3f}±{(ci_hi-ci_lo)/2:.3f}"
            else:
                row += "  " + "-"*8
        print(row)
    print()

# ─────────────────────────────────────────────────────────────
# Print: Per-region BCS (Middle East vs rest)
# ─────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TABLE: BCS by Region (selected regions) — best condition per model")
print("="*80)

key_regions = ["Middle East", "Eastern Europe", "Africa", "Sudan",
               "North East Asia", "Global"]

for model_key in MODEL_KEYS:
    print(f"\n{MODEL_LABELS[model_key]}:")
    print(f"  {'Region':<22} {'B0':>6} {'B1':>6} {'B2':>6} {'B3':>6}")
    print(f"  {'-'*50}")
    for region in key_regions:
        row = f"  {region:<22}"
        for condition in CONDITIONS:
            # Average BCS across all nations for this region
            vals = []
            for nation in NATIONS_ORDER:
                entry = region_metrics[model_key][condition].get(nation, {}).get(region)
                if entry and entry["bcs"] is not None:
                    vals.append(entry["bcs"])
            if vals:
                row += f"  {np.mean(vals):>5.3f}"
            else:
                row += "  -----"
        print(row)

# ─────────────────────────────────────────────────────────────
# Save all metrics to JSON
# ─────────────────────────────────────────────────────────────
metrics_path = os.path.join(WORKDIR, "paper_results", "llm_metrics.json")
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump({
        "metrics": {
            mk: {c: dict(cm) for c, cm in mv.items()}
            for mk, mv in metrics.items()
        },
        "region_metrics": {
            mk: {c: {n: dict(rm) for n, rm in nm.items()}
                 for c, nm in cm.items()}
            for mk, cm in region_metrics.items()
        },
        "conditions": CONDITIONS,
        "models": MODEL_KEYS,
        "created_at": datetime.now().isoformat(),
    }, f, indent=2)

print(f"\n✓ Metrics saved to {metrics_path}")
print("✓ Cell 31 complete")

In [ ]:
# ============================================================
# CELL 32 — Generate paper-ready formatted tables for §7
# ============================================================
# Run after Cell 31. Outputs the exact text to paste into the
# EMNLP Word template for §7.2 (F1) and §7.3 (BCS).

import json, os
import numpy as np
from collections import defaultdict

metrics_path = os.path.join(WORKDIR, "paper_results", "llm_metrics.json")
with open(metrics_path) as f:
    saved = json.load(f)

metrics        = saved["metrics"]
region_metrics = saved["region_metrics"]
NATIONS_ORDER  = ["CHN", "FRA", "GBR", "RUS", "USA"]
MODEL_LABELS   = {
    "gpt4o_mini":  "GPT-4o-mini",
    "llama70b":    "Llama-3.3-70B",
    "mistral24b":  "Mistral-Small-24B",
}
CONDITIONS = ["B0", "B1", "B2", "B3"]
MODEL_KEYS = ["gpt4o_mini", "llama70b", "mistral24b"]


def get_f1(model, cond, nation):
    try: return metrics[model][cond][nation]["f1"]
    except: return None

def get_bcs(model, cond, nation):
    try:
        e = metrics[model][cond][nation]
        return e["bcs"], e["ci_lo"], e["ci_hi"]
    except: return None, None, None


print("=" * 80)
print("TABLE 5 (§7.2): Weighted F1 — paste into paper")
print("=" * 80)
print()
print("| Model | Cond | CHN | FRA | GBR | RUS | USA | Mean |")
print("|---|---|---|---|---|---|---|---|")
for mk in MODEL_KEYS:
    for cond in CONDITIONS:
        vals = [get_f1(mk, cond, n) for n in NATIONS_ORDER]
        valid = [v for v in vals if v is not None]
        row = f"| {MODEL_LABELS[mk]} | {cond} |"
        for v in vals:
            row += f" {v:.3f} |" if v is not None else " — |"
        row += f" {np.mean(valid):.3f} |" if valid else " — |"
        print(row)
    print("|   |   |   |   |   |   |   |   |")

print()
print("=" * 80)
print("TABLE 6 (§7.3): BCS with 95% CI — paste into paper")
print("=" * 80)
print()
print("| Model | Cond | CHN | FRA | GBR | RUS | USA |")
print("|---|---|---|---|---|---|---|")
for mk in MODEL_KEYS:
    for cond in CONDITIONS:
        row = f"| {MODEL_LABELS[mk]} | {cond} |"
        for nation in NATIONS_ORDER:
            bcs, lo, hi = get_bcs(mk, cond, nation)
            if bcs is not None:
                margin = (hi - lo) / 2
                row += f" {bcs:.3f}±{margin:.3f} |"
            else:
                row += " — |"
        print(row)
    print("|   |   |   |   |   |   |   |")

print()
print("=" * 80)
print("BCS IMPROVEMENT over B0 (key debiasing finding)")
print("=" * 80)
print()
print("| Model | Nation | B0→B1 Δ | B0→B2 Δ | B0→B3 Δ | Best cond |")
print("|---|---|---|---|---|---|")
for mk in MODEL_KEYS:
    for nation in NATIONS_ORDER:
        b0_bcs, _, _ = get_bcs(mk, "B0", nation)
        deltas = {}
        for cond in ["B1", "B2", "B3"]:
            bcs, _, _ = get_bcs(mk, cond, nation)
            if bcs is not None and b0_bcs is not None:
                deltas[cond] = bcs - b0_bcs
        if not deltas: continue
        best_cond = max(deltas, key=deltas.get)
        row = f"| {MODEL_LABELS[mk]} | {nation} |"
        for cond in ["B1", "B2", "B3"]:
            d = deltas.get(cond)
            row += f" {d:+.3f} |" if d is not None else " — |"
        row += f" {best_cond} |"
        print(row)
    print("|   |   |   |   |   |   |")

print()
print("=" * 80)
print("F1 vs BCS DIVERGENCE — where the two metrics disagree")
print("=" * 80)
print()
print("Cases where F1 improves but BCS does not (or vice versa):")
print()
for mk in MODEL_KEYS:
    for nation in NATIONS_ORDER:
        b0_f1  = get_f1(mk, "B0", nation)
        b0_bcs, _, _ = get_bcs(mk, "B0", nation)
        for cond in ["B1", "B2", "B3"]:
            f1  = get_f1(mk, cond, nation)
            bcs, _, _ = get_bcs(mk, cond, nation)
            if None in (b0_f1, b0_bcs, f1, bcs): continue
            f1_delta  = f1  - b0_f1
            bcs_delta = bcs - b0_bcs
            # Flag divergence: one improves, other degrades by >0.02
            if f1_delta > 0.02 and bcs_delta < -0.02:
                print(f"  {MODEL_LABELS[mk]} | {nation} | {cond}: "
                      f"F1 {f1_delta:+.3f} ↑  BCS {bcs_delta:+.3f} ↓  "
                      f"(F1 improves, BCS degrades)")
            elif bcs_delta > 0.02 and f1_delta < -0.02:
                print(f"  {MODEL_LABELS[mk]} | {nation} | {cond}: "
                      f"F1 {f1_delta:+.3f} ↓  BCS {bcs_delta:+.3f} ↑  "
                      f"(BCS improves, F1 degrades)")

print("\n✓ Cell 32 complete — tables ready to paste into §7.2 and §7.3")